# Contribution A — multi-round distribution-aware active learning with official PROB

Runs the frozen Contribution-A pilot protocol end to end on a Google Colab NVIDIA
GPU runtime: four acquisition variants (`random`, `rarity_no_coherence`, `full`
with coherence power p=0.5, `full` with p=1.0), three feedback-driven rounds
each, on the M-OWODB Task-1 → Task-2 transition, with official PROB as the
detector behind `daowod_prob_bridge.py`.

Twelve real PROB fine-tuning runs and twelve official OWOD evaluations are
executed. Nothing is simulated.

**Pinned inputs**

| | |
|---|---|
| DAOWOD | `https://github.com/gubiczam/distribution-aware-owod.git` @ `8199d2b76da2032d04732a2c173d9583741ad9ce` |
| PROB | `https://github.com/gubiczam/PROB.git`, branch `feat/daowod-bridge-v2` @ `980cf3a796f064dd4c56f573ba10cc755143e116` |
| Backbone | DINO ResNet-50 self-supervised weights, size- and SHA-256-verified |
| Detector op | `models/ops` `MultiScaleDeformableAttention`, compiled and executed on the GPU |

**Runtime inputs you must provide on Google Drive**

1. The M-OWODB source data — either a directory holding `Annotations/` and
   `JPEGImages/`, or a `.tar` / `.tar.gz` / `.tar.zst` archive containing them.
2. A complete PROB **Task-1** training checkpoint (a `.pth` with `model`,
   `epoch` and ideally `args`), e.g. `t1.pth` or `checkout0040.pth`.

Both are discovered automatically under `DRIVE_SEARCH_ROOTS`; set
`DATASET_SOURCE` / `TASK1_CHECKPOINT` in the configuration cell to pin them
explicitly. Discovery prefers an already-extracted dataset directory over an
archive, but only a *complete* one: every candidate directory is measured against
the TOWOD split IDs the protocol consumes, and a partial folder is reported and
ignored instead of being chosen over the archive that holds the real dataset.
Discovery stops only when the choice is genuinely ambiguous — several complete
directories, or several archives and no complete directory. The notebook never invents data: if either input is absent it stops
with one error that lists every location it searched.

**Execution model**

Run the cells top to bottom exactly once in a fresh GPU runtime
(*Runtime → Run all*). Every cell is idempotent, so re-running one after a
transient failure is safe. Stages A–T below run in order, and the twelve
training runs start only after every mandatory pre-flight check has passed:

`A` purpose · `B` configuration · `C` helpers · `D` interpreter and GPU ·
`E` Drive and storage · `F` pinned checkouts · `G` file and source markers ·
`G2` PROB clean-checkout validation · `H` dependencies and repository gates ·
`I` DINO weights · `J` CUDA-extension clean build · `K` CUDA-extension import
and forward/backward execution · `L` bridge CLI · `M` dataset and checkpoint
staging · `N` protocol construction · `O` pre-flight (model build, real
inference, real evaluation) · `P` twelve rounds · `Q` metrics · `R` tables and
plots · `S` Drive artifacts and manifest · `T` final report.

**The pinned PROB checkout is used exactly as cloned — nothing is patched**

The pinned commit `980cf3a7…` sits on the branch that carries the modern-PyTorch
fixes to `models/prob_deformable_detr.py`, and it also ships the decoder-feature
export the bridge needs:

```python
'pred_features': hs[-1]
```

Stage `G2` therefore only *validates* the fresh checkout, and never writes to it:
`git rev-parse HEAD` equals the full 40-character commit, `git status
--porcelain` is empty, `models/prob_deformable_detr.py` contains that export
exactly once and compiles, the modern in-place `torch.no_grad()` initialisation
is present, the deprecated `.data =` forms have not returned, `python
daowod_prob_bridge.py check` exits 0 and reports the export, and the working tree
is still clean afterwards. The notebook applies no diff, edits no PROB source
file and monkey-patches nothing, so every file in both checkouts stays
byte-identical to its pinned commit. All of this is recorded in the experiment
manifest saved to Drive.

**Guarantees built into this notebook**

* One interpreter throughout — every subprocess is launched as
  `[sys.executable, ...]`, and the kernel/subprocess identity is asserted.
* `import torch` always precedes `import MultiScaleDeformableAttention`, and
  the torch shared-library directory is prepended to `LD_LIBRARY_PATH` for
  children. A `pip` build that returns 0 is never reported as a build failure.
* No silent fallback: if the compiled deformable-attention op cannot build,
  import or execute, the run stops with diagnostics instead of quietly using
  PROB's ~10× slower pure-PyTorch kernel.
* Warnings are not suppressed globally. The `q_col` compiler warning and the
  `setup.py` deprecation warning are warnings, not failures.
* Pinned checkouts are verified against full 40-character hashes and must be
  clean; the notebook never modifies them.

This is a pilot protocol under a controlled long-tail pool, not a published
benchmark result.

In [ ]:
# ============================================================================
# B. CONFIGURATION
#
# This is the only cell intended to be edited. It uses builtins only, so it can
# be read and validated before anything is imported, installed or mounted.
# ============================================================================

# --- Repositories: pinned, cloned fresh, and verified after checkout ---------
DAOWOD_REPOSITORY_URL = "https://github.com/gubiczam/distribution-aware-owod.git"
DAOWOD_COMMIT = "8199d2b76da2032d04732a2c173d9583741ad9ce"

PROB_REPOSITORY_URL = "https://github.com/gubiczam/PROB.git"
PROB_BRANCH = "feat/daowod-bridge-v2"
# This commit carries both the modern-PyTorch model fixes and the
# 'pred_features': hs[-1] decoder-feature export the bridge requires, so the
# checkout is used exactly as cloned and is verified to stay clean.
PROB_COMMIT = "980cf3a796f064dd4c56f573ba10cc755143e116"

# --- Self-supervised backbone weights loaded by PROB/models/backbone.py ------
DINO_BACKBONE_URL = (
    "https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/"
    "dino_resnet50_pretrain.pth"
)
# Verified against the published artifact. If Meta ever republishes the file the
# notebook stops and prints both the expected and the observed values.
DINO_BACKBONE_SHA256 = (
    "156f8c4166a23dc2951ae811e39d76a06269c565932edf647c0187e65cd7aa7c"
)
DINO_BACKBONE_BYTES = 94345885

# --- Google Drive -----------------------------------------------------------
DRIVE_MOUNT = "/content/drive"
DRIVE_SEARCH_ROOTS = [
    "/content/drive/MyDrive/DAOWOD",
    "/content/drive/MyDrive",
]
DRIVE_SEARCH_MAX_DEPTH = 4
DRIVE_RESULT_PARENT = "/content/drive/MyDrive/DAOWOD/results"

# "auto" searches DRIVE_SEARCH_ROOTS; otherwise give an absolute path. In "auto"
# mode a *complete* extracted dataset directory is preferred over an archive, so
# having both is not ambiguous; a directory that does not cover the required
# TOWOD split IDs (a smoke-test folder, say) is reported and ignored. Several
# complete directories, or several archives and no complete directory, still stop
# the run. An explicit directory is content-checked too and fails early if partial.
DATASET_SOURCE = "auto"
TASK1_CHECKPOINT = "auto"

# --- Frozen pilot protocol --------------------------------------------------
SEED = 0
ROUNDS = 3
BUDGET_PER_ROUND = 10
SOURCE_TASK2_IMAGES = 300
REFERENCE_IMAGES = 30
EVAL_UNKNOWN_IMAGES = 100
EVAL_KNOWN_IMAGES = 100
IMBALANCE_RATIO = 20.0

ALPHA = 0.3          # uncertainty weight
BETA = 0.2           # novelty weight
GAMMA = 0.5          # rarity weight
RARITY_POWER = 1.0
TOP_K = 3            # proposals averaged per image

VARIANTS = {
    "random": {"strategy": "random", "coherence_power": 1.0},
    "rarity_no_coherence": {"strategy": "rarity_no_coherence", "coherence_power": 1.0},
    "full_p05": {"strategy": "full", "coherence_power": 0.5},
    "full_p1": {"strategy": "full", "coherence_power": 1.0},
}

# --- PROB invocation --------------------------------------------------------
DATASET = "TOWOD"
TASK2_SOURCE_SPLIT = "owod_t2_train"
TASK1_REFERENCE_SPLIT = "owod_t1_train"
OFFICIAL_EVAL_SPLIT = "owod_all_task_test"
PREVIOUS_CLASSES = 20
CURRENT_CLASSES = 20
NUM_CLASSES = 81
OBJECTNESS_TEMPERATURE = 1.3
TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-5
OBJECTNESS_LOSS_COEFFICIENT = 8e-4
FREEZE_PROB_MODEL = True
BATCH_SIZE = 1
NUM_WORKERS = 2
MAX_PROPOSALS_PER_IMAGE = 20
MINIMUM_UNKNOWN_SCORE = 0.0
DEVICE = "cuda"
PROB_TIMEOUT_SECONDS = 86400

# main_open_world.py only binds `test_stats` inside the in-training evaluation
# branch, so eval_every must stay 1 or the training loop raises NameError.
PROB_EVAL_EVERY = 1

# --- Execution control ------------------------------------------------------
RESUME_COMPLETED_ROUNDS = True
STOP_AFTER_SMOKE_TESTS = False
KEEP_EXTRA_EPOCH_CHECKPOINTS = False
MINIMUM_LOCAL_FREE_GB = 8.0
MINIMUM_DRIVE_FREE_GB = 3.0
# PROB ships a pure-PyTorch deformable-attention kernel that is roughly an order
# of magnitude slower than the compiled CUDA op, and models/ops/modules chooses
# it automatically whenever the extension is missing. That silent substitution
# would change nothing observable except the wall clock, and this campaign would
# not finish. The compiled op is therefore mandatory: build, import and CUDA
# forward/backward execution are all verified, and any failure stops the run.
REQUIRE_CUDA_ATTENTION_EXTENSION = True

# Deformable-attention build tuning. MAX_JOBS bounds ninja's parallelism so a
# 12 GB Colab runtime cannot be pushed into an OOM kill while nvcc runs.
EXTENSION_BUILD_MAX_JOBS = 4
# Number of pre-flight evaluation images for the real end-to-end evaluator run.
PREFLIGHT_EVAL_IMAGES = 4
PREFLIGHT_PREDICT_IMAGES = 2

# --- Derived paths (not user-editable) --------------------------------------
EXPERIMENT_NAME = f"contribution_a_multiround_seed{SEED}"
CONTENT_ROOT = "/content"
DAOWOD_PATH = f"{CONTENT_ROOT}/distribution-aware-owod"
PROB_PATH = f"{CONTENT_ROOT}/PROB"
DATA_ROOT = f"{CONTENT_ROOT}/data/OWOD"
LOCAL_RESULT_ROOT = f"{CONTENT_ROOT}/{EXPERIMENT_NAME}"
LOCAL_CHECKPOINT_DIR = f"{CONTENT_ROOT}/daowod_checkpoints"
COMPAT_DIR = f"{CONTENT_ROOT}/daowod_compat"
DRIVE_RESULT_DIR = f"{DRIVE_RESULT_PARENT}/{EXPERIMENT_NAME}"
PILOT_EVAL_SPLIT = f"daowod_multiround_balanced_seed{SEED}_test"
PILOT_SOURCE_SPLIT = f"daowod_multiround_t2_source_{SEED}_train"
PREFLIGHT_EVAL_SPLIT = f"daowod_gate_seed{SEED}_test"

# --- Configuration validation (builtins only) -------------------------------
HEX_DIGITS = set("0123456789abcdef")
for _name, _commit in (("DAOWOD_COMMIT", DAOWOD_COMMIT), ("PROB_COMMIT", PROB_COMMIT)):
    if len(_commit) != 40 or not set(_commit) <= HEX_DIGITS:
        raise RuntimeError(
            f"{_name}={_commit!r} is not a full lowercase 40-character git hash. "
            "Abbreviated hashes are rejected because the checkout is verified "
            "with an exact string comparison against `git rev-parse HEAD`."
        )
if len(DINO_BACKBONE_SHA256) != 64 or not set(DINO_BACKBONE_SHA256) <= HEX_DIGITS:
    raise RuntimeError("DINO_BACKBONE_SHA256 must be a 64-character hex digest.")

# datasets/coco.py make_coco_transforms(image_set) picks its transform pipeline
# by substring, testing 'train', then 'ft', then 'val', then 'test', and raising
# ValueError when none match. An evaluation split whose name happens to contain
# 'train', 'ft' or 'val' would silently get augmented training transforms.
for _split in (PILOT_EVAL_SPLIT, PREFLIGHT_EVAL_SPLIT):
    if "test" not in _split:
        raise RuntimeError(f"Evaluation split {_split!r} must contain 'test'.")
    for _earlier in ("train", "ft", "val"):
        if _earlier in _split:
            raise RuntimeError(
                f"Evaluation split {_split!r} contains {_earlier!r}, which "
                "make_coco_transforms matches before 'test'; it would be evaluated "
                "with the wrong transforms."
            )
if "train" not in PILOT_SOURCE_SPLIT:
    raise RuntimeError("The Task-2 source split name must contain 'train'.")
if set(VARIANTS) != {"random", "rarity_no_coherence", "full_p05", "full_p1"}:
    raise RuntimeError("The four protocol variants were changed.")
if next(iter(VARIANTS)) != "random":
    raise RuntimeError("The single-round gate expects 'random' to be the first "
                       "variant so that round 1 is the random baseline.")
if DEVICE != "cuda":
    raise RuntimeError(
        f"DEVICE is {DEVICE!r}. This notebook targets a Colab NVIDIA GPU runtime "
        "and PROB's training loop requires CUDA (datasets/data_prefetcher.py "
        "constructs a torch.cuda.Stream unconditionally)."
    )
if PROB_EVAL_EVERY != 1:
    raise RuntimeError(
        f"PROB_EVAL_EVERY is {PROB_EVAL_EVERY}. In main_open_world.py the name "
        "`test_stats` is only bound inside the in-training evaluation branch, so "
        "any value that skips that branch raises NameError mid-training."
    )
if REQUIRE_CUDA_ATTENTION_EXTENSION is not True:
    raise RuntimeError(
        "REQUIRE_CUDA_ATTENTION_EXTENSION must stay True. This notebook has no "
        "pure-PyTorch fallback path: the compiled MultiScaleDeformableAttention "
        "op is a hard requirement, and models/ops/modules/ms_deform_attn.py would "
        "otherwise substitute the slow kernel without any visible error."
    )
if PREFLIGHT_PREDICT_IMAGES < 2 or PREFLIGHT_EVAL_IMAGES < 2:
    raise RuntimeError("The pre-flight needs at least two images per path.")
if REFERENCE_IMAGES < PREFLIGHT_PREDICT_IMAGES:
    raise RuntimeError("REFERENCE_IMAGES must cover the pre-flight prediction.")
if min(EVAL_UNKNOWN_IMAGES, EVAL_KNOWN_IMAGES) * 2 < PREFLIGHT_EVAL_IMAGES:
    raise RuntimeError("The evaluation split is too small for the pre-flight.")
if ROUNDS * BUDGET_PER_ROUND >= SOURCE_TASK2_IMAGES:
    raise RuntimeError("The cumulative budget must stay below the source pool.")
if not (0.0 <= MINIMUM_UNKNOWN_SCORE <= 1.0):
    raise RuntimeError("MINIMUM_UNKNOWN_SCORE must lie in [0, 1].")
if PREVIOUS_CLASSES + CURRENT_CLASSES >= NUM_CLASSES:
    raise RuntimeError("The introduced-class count must stay below NUM_CLASSES.")
if EXTENSION_BUILD_MAX_JOBS < 1:
    raise RuntimeError("EXTENSION_BUILD_MAX_JOBS must be positive.")

# Every working path must live under /content on Colab: Drive is FUSE-mounted
# and too slow for checkpoints and image reads, and other locations do not
# survive a factory reset in a predictable way.
COLAB_LOCAL_PATHS = {
    "CONTENT_ROOT": CONTENT_ROOT,
    "DAOWOD_PATH": DAOWOD_PATH,
    "PROB_PATH": PROB_PATH,
    "DATA_ROOT": DATA_ROOT,
    "LOCAL_RESULT_ROOT": LOCAL_RESULT_ROOT,
    "LOCAL_CHECKPOINT_DIR": LOCAL_CHECKPOINT_DIR,
    "COMPAT_DIR": COMPAT_DIR,
}
for _name, _value in COLAB_LOCAL_PATHS.items():
    if not (_value == "/content" or _value.startswith("/content/")):
        raise RuntimeError(f"{_name}={_value!r} must be under /content.")
    if _value.startswith(DRIVE_MOUNT):
        raise RuntimeError(f"{_name}={_value!r} must not live on Drive.")
COLAB_DRIVE_PATHS = {
    "DRIVE_MOUNT": DRIVE_MOUNT,
    "DRIVE_RESULT_PARENT": DRIVE_RESULT_PARENT,
    "DRIVE_RESULT_DIR": DRIVE_RESULT_DIR,
    **{f"DRIVE_SEARCH_ROOTS[{_index}]": _root
       for _index, _root in enumerate(DRIVE_SEARCH_ROOTS)},
}
for _name, _value in COLAB_DRIVE_PATHS.items():
    if not (_value == DRIVE_MOUNT or _value.startswith(f"{DRIVE_MOUNT}/")):
        raise RuntimeError(f"{_name}={_value!r} must be under {DRIVE_MOUNT}.")

print(f"experiment     : {EXPERIMENT_NAME}")
print(f"DAOWOD commit  : {DAOWOD_COMMIT}")
print(f"PROB branch    : {PROB_BRANCH}")
print(f"PROB commit    : {PROB_COMMIT}")
print(f"variants       : {list(VARIANTS)}")
print(f"rounds         : {ROUNDS} x budget {BUDGET_PER_ROUND} "
      f"= {ROUNDS * BUDGET_PER_ROUND} images per variant")
print(f"detector runs  : {len(VARIANTS) * ROUNDS} training runs "
      f"+ {len(VARIANTS) * ROUNDS} official evaluations")
print("configuration validated")

In [ ]:
# ============================================================================
# C. IMPORTS AND HELPER UTILITIES
#
# Every helper used anywhere below is defined here exactly once. Nothing in this
# cell touches the GPU, Drive or the network, so it is safe to re-run at any
# point. No warning filter is installed: compiler warnings such as the unused
# `q_col` warning and the setup.py deprecation warning are warnings, not
# failures, and hiding them would hide real ones too.
# ============================================================================
import csv
import gc
import hashlib
import importlib
import importlib.util
import inspect
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

SECRET_HINTS = ("SECRET", "TOKEN", "KEY", "PASSWORD", "PASSWD", "CREDENTIAL")
RULE = "=" * 78


def banner(title):
    print(f"\n{RULE}\n{title}\n{RULE}")


def redacted_environment(names=None):
    """Render the environment, hiding the values of secret-looking names."""
    lines = []
    for name in sorted(os.environ if names is None else names):
        value = os.environ.get(name, "<unset>")
        if any(hint in name.upper() for hint in SECRET_HINTS):
            value = "<redacted>"
        lines.append(f"{name}={value}")
    return "\n".join(lines)


# The variables that decide which interpreter, libraries and modules a child
# process resolves. Printed on every subprocess failure.
DIAGNOSTIC_ENVIRONMENT_NAMES = (
    "PATH",
    "LD_LIBRARY_PATH",
    "PYTHONPATH",
    "PYTHONHOME",
    "VIRTUAL_ENV",
    "CUDA_HOME",
    "CUDA_PATH",
    "TORCH_CUDA_ARCH_LIST",
    "MAX_JOBS",
    "WANDB_MODE",
)


def runtime_diagnostics():
    """Interpreter, torch, GPU and disk facts, as far as they are known yet."""
    rows = [
        ("sys.executable", sys.executable),
        ("python version", platform.python_version()),
        ("platform", platform.platform()),
        ("cwd", os.getcwd()),
    ]
    if "torch" in globals():
        rows += [
            ("torch", torch.__version__),
            ("torch file", torch.__file__),
            ("torch.version.cuda", torch.version.cuda),
            ("cuda available", torch.cuda.is_available()),
            ("device count", torch.cuda.device_count()),
        ]
        if torch.cuda.is_available():
            rows.append(("gpu", torch.cuda.get_device_name(0)))
            rows.append(("gpu capability",
                         ".".join(str(v) for v in torch.cuda.get_device_capability(0))))
    for label, path in (("free /content GB", CONTENT_ROOT),
                        ("free Drive GB", DRIVE_RESULT_PARENT)):
        try:
            rows.append((label, round(shutil.disk_usage(path).free / 1024 ** 3, 2)))
        except OSError as error:
            rows.append((label, f"unavailable ({error})"))
    return rows


def report_failure_context(command, cwd, result):
    """Print everything needed to debug a failed subprocess. Hides nothing."""
    banner("SUBPROCESS FAILED")
    print("COMMAND      :", " ".join(str(part) for part in command))
    print("CWD          :", cwd if cwd else os.getcwd())
    print("RETURN CODE  :", getattr(result, "returncode", "n/a"))
    banner("FAILED STDOUT")
    print(getattr(result, "stdout", ""))
    banner("FAILED STDERR")
    print(getattr(result, "stderr", ""))
    banner("RUNTIME")
    for key, value in runtime_diagnostics():
        print(f"{key:<20} : {value}")
    banner("RELEVANT ENVIRONMENT")
    print(redacted_environment(DIAGNOSTIC_ENVIRONMENT_NAMES))


def run(command, *, cwd=None, timeout, check=True, quiet=False):
    """Run a subprocess, echo command/cwd/returncode/stdout/stderr, never swallow.

    Children inherit ``os.environ``, which cell D has already extended with the
    torch and CUDA shared-library directories, so every subprocess launched here
    and every subprocess launched later by ProbAdapter sees the same environment.
    """
    command = [str(part) for part in command]
    banner("COMMAND")
    print(" ".join(command))
    print("CWD:", cwd if cwd else os.getcwd())
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        timeout=timeout,
    )
    print("RETURN CODE:", result.returncode)
    if not quiet or result.returncode != 0:
        banner("STDOUT")
        print(result.stdout)
        banner("STDERR")
        print(result.stderr)
    if check and result.returncode != 0:
        report_failure_context(command, cwd, result)
        raise subprocess.CalledProcessError(
            result.returncode, command, output=result.stdout, stderr=result.stderr
        )
    return result


def torch_snippet(*lines):
    """Build a subprocess snippet whose very first import is torch.

    The compiled MultiScaleDeformableAttention extension links against libc10.so
    from the torch wheel and does not carry an RPATH to it. Importing the
    extension in a process that has not already imported torch therefore raises

        ImportError: libc10.so: cannot open shared object file

    even when the build and the install both succeeded. Importing torch first
    loads libc10.so into the process, after which the extension resolves. Every
    snippet in this notebook is built through this function, so no extension
    import can ever precede `import torch`.
    """
    return "\n".join(("import torch  # must precede the CUDA extension", *lines))


def python_probe(code, *, cwd, timeout, arguments=(), check=True):
    """Run a snippet in a fresh copy of this kernel's interpreter."""
    return run([sys.executable, "-c", code, *arguments],
               cwd=cwd, timeout=timeout, check=check)


def json_probe(code, *, cwd, timeout, arguments=()):
    """Run a snippet that prints one JSON object as its last line."""
    result = python_probe(code, cwd=cwd, timeout=timeout, arguments=arguments)
    lines = [line for line in result.stdout.splitlines() if line.strip()]
    if not lines:
        report_failure_context([sys.executable, "-c", code], cwd, result)
        raise RuntimeError("The probe printed nothing on stdout.")
    try:
        return json.loads(lines[-1])
    except json.JSONDecodeError as error:
        report_failure_context([sys.executable, "-c", code], cwd, result)
        raise RuntimeError(f"The probe's last stdout line is not JSON: {lines[-1]!r}") from error


def read_ids(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing image-ID file: {path}")
    values = [
        line.split()[0]
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    return list(dict.fromkeys(values))


def write_ids(path, image_ids):
    """Write an image-ID list atomically."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = "\n".join(str(value) for value in image_ids)
    temporary = path.with_name(f".{path.name}.tmp")
    temporary.write_text(text + ("\n" if image_ids else ""), encoding="utf-8")
    temporary.replace(path)


def unique(values):
    return list(dict.fromkeys(str(value) for value in values))


def sha256_of_ids(values):
    digest = hashlib.sha256()
    for value in values:
        digest.update(str(value).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


def sha256_of_object(value):
    payload = json.dumps(value, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def sha256_of_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, data):
    """Write JSON atomically: readers never observe a half-written file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    temporary.write_text(
        json.dumps(data, indent=2, sort_keys=True, default=str) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def free_gb(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return round(shutil.disk_usage(path).free / 1024 ** 3, 2)


def require_free_space(path, minimum_gb, description):
    available = free_gb(path)
    if available < minimum_gb:
        raise RuntimeError(
            f"{description}: only {available} GB free at {path}, "
            f"{minimum_gb} GB required."
        )
    return available


def require_writable_directory(path, description):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    if not path.is_dir():
        raise RuntimeError(f"{description} is not a directory: {path}")
    probe = path / ".daowod_write_probe"
    probe.write_text("probe", encoding="utf-8")
    if probe.read_text(encoding="utf-8") != "probe":
        raise RuntimeError(f"{description} did not read back a test write: {path}")
    probe.unlink()
    return path


def directory_size_gb(path):
    total = sum(item.stat().st_size for item in Path(path).rglob("*") if item.is_file())
    return round(total / 1024 ** 3, 3)


def remove_paths(paths, *, description):
    """Delete a known, explicitly named set of generated artifacts."""
    removed = []
    for path in paths:
        path = Path(path)
        if path.is_symlink() or path.is_file():
            path.unlink()
            removed.append(str(path))
        elif path.is_dir():
            shutil.rmtree(path)
            removed.append(f"{path}/")
    print(f"{description}: removed {len(removed)} item(s)")
    for entry in removed:
        print(f"  {entry}")
    return removed


def table(title, rows):
    banner(title)
    width = max((len(str(key)) for key, _ in rows), default=0)
    for key, value in rows:
        print(f"{str(key):<{width}} : {value}")


def seeded_order_preserving_sample(image_ids, count, *, seed, salt):
    """Deterministic subset that keeps the original file order."""
    ids = unique(image_ids)
    if len(ids) != len(image_ids):
        raise ValueError(f"{salt}: source IDs contain duplicates.")
    if count > len(ids):
        raise ValueError(f"{salt}: requested {count} images from only {len(ids)}.")
    chosen = set(random.Random(f"{seed}:{salt}").sample(ids, count))
    return [image_id for image_id in ids if image_id in chosen]


def cleanup_runtime():
    gc.collect()
    if "torch" in globals() and torch.cuda.is_available():
        torch.cuda.empty_cache()


def elapsed_text(seconds):
    seconds = int(round(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:d}h{minutes:02d}m{seconds:02d}s"


# --- stage bookkeeping ------------------------------------------------------
PREFLIGHT_STAGES = [
    "runtime",
    "drive",
    "repositories",
    "checkout files",
    "prob source",
    "dependencies",
    "daowod gates",
    "dino weights",
    "extension build",
    "extension execution",
    "bridge",
    "drive assets",
    "dataset",
    "protocol",
    "images",
    "preflight",
]
ROUND_STAGES = [
    f"{variant} round {index}"
    for variant in VARIANTS for index in range(1, ROUNDS + 1)
]
STAGES = [*PREFLIGHT_STAGES, *ROUND_STAGES, "metrics", "report", "drive persistence"]
STATUS = dict.fromkeys(STAGES, "PENDING")
NOTEBOOK_STARTED = time.time()



def mark(stage, value="OK"):
    if stage not in STATUS:
        raise KeyError(f"Unknown stage: {stage}")
    STATUS[stage] = value
    print(f"[stage] {stage}: {value}")


def require_stages(*stages):
    """Refuse to run a cell whose prerequisites did not complete."""
    unmet = [stage for stage in stages if STATUS[stage] not in {"OK", "RESTORED"}]
    if unmet:
        raise RuntimeError(
            f"These stages must complete first: {unmet}. Run the notebook cells "
            "in order from the top."
        )


print(f"helpers defined: {len(STAGES)} stages tracked "
      f"({len(PREFLIGHT_STAGES)} pre-flight, {len(ROUND_STAGES)} rounds)")

In [ ]:
# ============================================================================
# D. RUNTIME, INTERPRETER AND GPU VERIFICATION
#
# One interpreter is used for everything: `sys.executable`. Every subprocess in
# this notebook is launched as [sys.executable, ...] or
# [sys.executable, "-m", "pip", ...] — never as `python`, `python3`,
# `/usr/bin/python3` or a bare `pip`. That identity is asserted here, not
# assumed, and the torch/CUDA shared-library directories are added to the
# environment every child process inherits.
# ============================================================================
require_stages()

# --- interpreter version ----------------------------------------------------
if sys.version_info[:2] < (3, 11):
    raise RuntimeError(
        "distribution-aware-owod declares requires-python '>=3.11,<3.13' and its "
        "source uses 3.11 syntax, but this runtime is Python "
        f"{platform.python_version()}. Use a Colab runtime with Python >= 3.11."
    )

# The pinned DAOWOD commit caps requires-python at <3.13. The package is pure
# Python with three well-behaved dependencies, so on a newer Colab interpreter
# the cap is relaxed for pip only, and the repository's own ruff/pytest/compileall
# gates in cell H then prove the code actually runs on this interpreter.
DAOWOD_PIP_FLAGS = []
if sys.version_info[:2] >= (3, 13):
    DAOWOD_PIP_FLAGS = ["--ignore-requires-python"]
    print(
        f"NOTE: this runtime is Python {platform.python_version()}, above the "
        "'<3.13' cap declared by the pinned DAOWOD pyproject.toml. The cap is "
        "relaxed for the editable install only (--ignore-requires-python); the "
        "repository's own lint, test and compile gates run immediately after the "
        "install and will fail loudly if the code is genuinely incompatible."
    )

# PROB's training loop, evaluator and dataloaders all assume CUDA; wandb is
# disabled so no subprocess can block on an interactive login.
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"

import torch
import torch.utils.cpp_extension
import torchvision

# --- GPU ---------------------------------------------------------------------
if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is required. Select Runtime > Change runtime type > "
        "T4 GPU (or better), then Runtime > Run all."
    )
if torch.cuda.device_count() < 1:
    raise RuntimeError("torch.cuda reports no devices.")
if torch.version.cuda is None:
    raise RuntimeError(
        "This PyTorch build has no CUDA support; PROB cannot compile its "
        "MultiScaleDeformableAttention extension against it."
    )

# util/misc.py branches on float(torchvision.__version__.split('.')[1]).
torchvision_minor = float(torchvision.__version__.split(".")[1])

probe = torch.randn(512, 512, device="cuda") @ torch.randn(512, 512, device="cuda")
torch.cuda.synchronize()
probe_value = float(probe.sum().item())
if not math.isfinite(probe_value):
    raise RuntimeError("A CUDA matrix multiplication produced a non-finite result.")
del probe
torch.cuda.empty_cache()

gpu_capability = torch.cuda.get_device_capability(0)
gpu_architecture = f"{gpu_capability[0]}.{gpu_capability[1]}"

# --- shared libraries for every child process -------------------------------
# The compiled extension links against libc10.so / libtorch_cpu.so from the
# torch wheel. Those directories are derived from torch itself, never hardcoded,
# and are prepended while the existing value is preserved.
torch_package_dir = Path(torch.__file__).resolve().parent
torch_library_dir = torch_package_dir / "lib"
if not torch_library_dir.is_dir():
    raise RuntimeError(f"The torch shared-library directory is missing: {torch_library_dir}")
if not any(torch_library_dir.glob("libc10.so*")):
    raise RuntimeError(f"libc10.so is not present in {torch_library_dir}.")

site_packages_dir = torch_package_dir.parent
cuda_wheel_library_dirs = [
    str(path) for path in sorted(site_packages_dir.glob("nvidia/*/lib")) if path.is_dir()
]

CUDA_HOME = torch.utils.cpp_extension.CUDA_HOME
cuda_home_library_dirs = []
if CUDA_HOME:
    os.environ.setdefault("CUDA_HOME", str(CUDA_HOME))
    for name in ("lib64", "lib"):
        candidate = Path(CUDA_HOME, name)
        if candidate.is_dir():
            cuda_home_library_dirs.append(str(candidate))

LIBRARY_PATH_ADDITIONS = [
    str(torch_library_dir),
    *cuda_wheel_library_dirs,
    *cuda_home_library_dirs,
]
previous_library_path = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = os.pathsep.join(dict.fromkeys([
    *LIBRARY_PATH_ADDITIONS,
    *[part for part in previous_library_path.split(os.pathsep) if part],
]))

# --- nvcc --------------------------------------------------------------------
nvcc_path = shutil.which("nvcc")
if nvcc_path is None and CUDA_HOME:
    candidate = Path(CUDA_HOME, "bin", "nvcc")
    if candidate.is_file():
        nvcc_path = str(candidate)
        os.environ["PATH"] = os.pathsep.join([str(candidate.parent), os.environ.get("PATH", "")])
if nvcc_path is None:
    raise RuntimeError(
        "nvcc was not found on PATH and torch.utils.cpp_extension.CUDA_HOME "
        f"({CUDA_HOME!r}) does not contain bin/nvcc. The CUDA toolkit is required "
        "to compile PROB's MultiScaleDeformableAttention extension. Use a Colab "
        "GPU runtime, which ships the toolkit at /usr/local/cuda."
    )
nvcc_version_output = run([nvcc_path, "--version"], timeout=300).stdout.strip()
nvcc_release = next(
    (line.strip() for line in nvcc_version_output.splitlines() if "release" in line),
    nvcc_version_output.splitlines()[-1] if nvcc_version_output else "unknown",
)

# --- kernel / subprocess interpreter identity -------------------------------
INTERPRETER_PROBE = torch_snippet(
    "import json, sys, sysconfig",
    "print(json.dumps({",
    "    'executable': sys.executable,",
    "    'version': sys.version.split()[0],",
    "    'prefix': sys.prefix,",
    "    'platlib': sysconfig.get_paths()['platlib'],",
    "    'torch': torch.__version__,",
    "    'torch_file': torch.__file__,",
    "    'torch_cuda': torch.version.cuda,",
    "    'cuda_available': torch.cuda.is_available(),",
    "    'device_count': torch.cuda.device_count(),",
    "    'ld_library_path': __import__('os').environ.get('LD_LIBRARY_PATH', ''),",
    "}))",
)
child = json_probe(INTERPRETER_PROBE, cwd=CONTENT_ROOT, timeout=1200)

if Path(child["executable"]).resolve() != Path(sys.executable).resolve():
    raise RuntimeError(
        "Interpreter mismatch: this kernel is "
        f"{Path(sys.executable).resolve()} but a subprocess launched with "
        f"sys.executable reported {Path(child['executable']).resolve()}. Every "
        "build and validation step below would then run against a different "
        "Python and a different torch."
    )
if child["version"] != platform.python_version():
    raise RuntimeError(
        f"Interpreter version mismatch: kernel {platform.python_version()}, "
        f"subprocess {child['version']}."
    )
if child["prefix"] != sys.prefix:
    raise RuntimeError(
        f"Interpreter prefix mismatch: kernel {sys.prefix}, subprocess {child['prefix']}."
    )
if Path(child["torch_file"]).resolve() != Path(torch.__file__).resolve():
    raise RuntimeError(
        f"A subprocess imports torch from {child['torch_file']} but this kernel "
        f"imported {torch.__file__}."
    )
if child["torch"] != torch.__version__ or child["torch_cuda"] != torch.version.cuda:
    raise RuntimeError(
        f"torch differs between kernel ({torch.__version__}/{torch.version.cuda}) "
        f"and subprocess ({child['torch']}/{child['torch_cuda']})."
    )
if not child["cuda_available"] or child["device_count"] < 1:
    raise RuntimeError(f"A fresh interpreter cannot see the GPU: {child}")
if str(torch_library_dir) not in child["ld_library_path"].split(os.pathsep):
    raise RuntimeError(
        f"{torch_library_dir} did not reach the subprocess LD_LIBRARY_PATH "
        f"({child['ld_library_path']!r}); the CUDA extension would fail to load "
        "its libc10.so dependency."
    )

# --- pip must belong to this interpreter ------------------------------------
pip_specification = importlib.util.find_spec("pip")
if pip_specification is None or pip_specification.origin is None:
    raise RuntimeError(
        f"{sys.executable} has no importable pip module, so "
        "[sys.executable, '-m', 'pip', ...] cannot install anything."
    )
pip_version_line = run([sys.executable, "-m", "pip", "--version"], timeout=600).stdout.strip()
pip_python_tag = f"(python {sys.version_info.major}.{sys.version_info.minor})"
if pip_python_tag not in pip_version_line:
    raise RuntimeError(
        f"`{sys.executable} -m pip --version` reported {pip_version_line!r}, which "
        f"does not identify itself as {pip_python_tag}. pip belongs to another "
        "interpreter and would install into the wrong site-packages."
    )
pip_module_dir = str(Path(pip_specification.origin).parent)

# Informational only: these names are never used to launch anything.
path_interpreters = {
    name: shutil.which(name) or "<not on PATH>"
    for name in ("python", "python3", "pip", "pip3")
}

run(["nvidia-smi"], timeout=300)

# Recorded now and re-checked after the pip installs: installing the DAOWOD and
# PROB dependencies must not replace the working torch/numpy build.
BASELINE_VERSIONS = {
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
}

require_writable_directory(CONTENT_ROOT, "Colab local disk (/content)")
for _name, _value in COLAB_LOCAL_PATHS.items():
    require_writable_directory(_value, _name)

banner("DETECTOR STACK")
print("PROB is a Deformable-DETR derivative. Neither repository imports")
print("detectron2, so there is no Detectron2 component to verify; the")
print("equivalent custom operator is MultiScaleDeformableAttention")
print("(PROB/models/ops), compiled in cell J and executed in cell K.")
print("detectron2 present in this runtime:",
      importlib.util.find_spec("detectron2") is not None)

RUNTIME_INFO = {
    "python_version": platform.python_version(),
    "python_implementation": platform.python_implementation(),
    "sys_executable": sys.executable,
    "sys_prefix": sys.prefix,
    "platform": platform.platform(),
    "pip": pip_version_line,
    "pip_module_dir": pip_module_dir,
    "path_interpreters": path_interpreters,
    "torch": torch.__version__,
    "torch_file": str(torch.__file__),
    "torch_cuda": torch.version.cuda,
    "torch_library_dir": str(torch_library_dir),
    "cudnn": torch.backends.cudnn.version(),
    "torchvision": torchvision.__version__,
    "torchvision_minor": torchvision_minor,
    "cuda_home": str(CUDA_HOME),
    "nvcc": nvcc_path,
    "nvcc_release": nvcc_release,
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "gpu_capability": gpu_architecture,
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 2),
    "ld_library_path_additions": LIBRARY_PATH_ADDITIONS,
    "daowod_pip_flags": DAOWOD_PIP_FLAGS,
}

table("RUNTIME", [
    ("sys.executable", sys.executable),
    ("Python", platform.python_version()),
    ("sys.prefix", sys.prefix),
    ("subprocess interpreter", child["executable"]),
    ("subprocess platlib", child["platlib"]),
    ("pip", pip_version_line),
    ("pip module", pip_module_dir),
    ("PATH python/python3", f"{path_interpreters['python']} | {path_interpreters['python3']}"),
    ("PATH pip/pip3", f"{path_interpreters['pip']} | {path_interpreters['pip3']}"),
    ("Platform", platform.platform()),
    ("PyTorch", torch.__version__),
    ("PyTorch file", torch.__file__),
    ("PyTorch CUDA", torch.version.cuda),
    ("cuDNN", torch.backends.cudnn.version()),
    ("torchvision", torchvision.__version__),
    ("torchvision minor", torchvision_minor),
    ("CUDA_HOME", CUDA_HOME),
    ("nvcc", nvcc_path),
    ("nvcc release", nvcc_release),
    ("GPU", torch.cuda.get_device_name(0)),
    ("GPU count", torch.cuda.device_count()),
    ("GPU capability", gpu_architecture),
    ("GPU memory GB", RUNTIME_INFO["gpu_memory_gb"]),
    ("CUDA probe finite", True),
    ("torch library dir", torch_library_dir),
    ("LD_LIBRARY_PATH additions", " ".join(LIBRARY_PATH_ADDITIONS)),
    ("/content writable", True),
    ("free /content GB", free_gb(CONTENT_ROOT)),
])
mark("runtime")

In [ ]:
# ============================================================================
# E. GOOGLE DRIVE MOUNT AND STORAGE CHECKS
#
# Drive holds the two inputs (dataset, Task-1 checkpoint) and receives the
# durable results. All training and image I/O happens under /content; Drive is a
# FUSE mount and is far too slow for checkpoint writes and per-image reads.
# Re-running this cell is safe: force_remount is False.
# ============================================================================
require_stages("runtime")

from google.colab import drive

drive.mount(DRIVE_MOUNT, force_remount=False)
if not Path(DRIVE_MOUNT, "MyDrive").is_dir():
    raise RuntimeError(
        f"Google Drive is not mounted at {DRIVE_MOUNT}/MyDrive. Colab asks for "
        "Drive access once per runtime; approve the prompt and re-run this cell."
    )

require_writable_directory(DRIVE_RESULT_PARENT, "DRIVE_RESULT_PARENT")
require_writable_directory(LOCAL_RESULT_ROOT, "LOCAL_RESULT_ROOT")

local_free_gb = require_free_space(
    CONTENT_ROOT, MINIMUM_LOCAL_FREE_GB,
    "Local Colab disk is too small for the dataset, checkpoints and rounds")
drive_free_gb = require_free_space(
    DRIVE_RESULT_PARENT, MINIMUM_DRIVE_FREE_GB,
    "Google Drive has too little space for the durable results")

existing_search_roots = [root for root in DRIVE_SEARCH_ROOTS if Path(root).is_dir()]
if not existing_search_roots:
    raise FileNotFoundError(
        "None of the configured Drive search roots exist: "
        f"{DRIVE_SEARCH_ROOTS}. Create the folder that holds the M-OWODB data "
        "and the PROB Task-1 checkpoint, or set DATASET_SOURCE and "
        "TASK1_CHECKPOINT to absolute paths in the configuration cell."
    )

table("DRIVE AND STORAGE", [
    ("Drive mount", DRIVE_MOUNT),
    ("Drive result parent", DRIVE_RESULT_PARENT),
    ("Drive result dir", DRIVE_RESULT_DIR),
    ("Drive writable", True),
    ("search roots configured", DRIVE_SEARCH_ROOTS),
    ("search roots present", existing_search_roots),
    ("search depth", DRIVE_SEARCH_MAX_DEPTH),
    ("free /content GB", f"{local_free_gb} (minimum {MINIMUM_LOCAL_FREE_GB})"),
    ("free Drive GB", f"{drive_free_gb} (minimum {MINIMUM_DRIVE_FREE_GB})"),
    ("local result root", LOCAL_RESULT_ROOT),
])
mark("drive")

In [ ]:
# ============================================================================
# F. PINNED REPOSITORY CHECKOUTS AND EXACT COMMIT VERIFICATION
#
# Both checkouts are deleted and cloned fresh, then detached onto the pinned
# commits. `git rev-parse HEAD` is compared with the full 40-character hash by
# exact string equality (never against an abbreviation), and each working tree
# must be clean immediately after checkout. Neither checkout is modified by this
# notebook: no patches, and nothing generated is committed.
# ============================================================================
require_stages("drive")

remove_paths([DAOWOD_PATH, PROB_PATH], description="previous checkouts")

run(["git", "clone", DAOWOD_REPOSITORY_URL, DAOWOD_PATH],
    cwd=CONTENT_ROOT, timeout=1800)
run(["git", "-C", DAOWOD_PATH, "checkout", "--detach", DAOWOD_COMMIT], timeout=600)

run(["git", "clone", "--branch", PROB_BRANCH, "--single-branch",
     PROB_REPOSITORY_URL, PROB_PATH], cwd=CONTENT_ROOT, timeout=1800)
run(["git", "-C", PROB_PATH, "checkout", "--detach", PROB_COMMIT], timeout=600)

repo_commits = {}
repo_details = {}
for name, path, expected, branch in (
    ("DAOWOD", DAOWOD_PATH, DAOWOD_COMMIT, None),
    ("PROB", PROB_PATH, PROB_COMMIT, PROB_BRANCH),
):
    head = run(["git", "-C", path, "rev-parse", "HEAD"], timeout=120).stdout.strip()
    if len(head) != 40:
        raise RuntimeError(
            f"{name}: `git rev-parse HEAD` returned {head!r}, which is not a full "
            "40-character hash; the equality check below would be meaningless."
        )
    if head != expected:
        raise RuntimeError(
            f"{name} checkout mismatch.\n  expected {expected}\n  actual   {head}\n"
            "The pinned commit is not what was checked out."
        )
    dirty = run(["git", "-C", path, "status", "--porcelain"], timeout=120).stdout.strip()
    if dirty:
        raise RuntimeError(
            f"{name} working tree is not clean immediately after checkout:\n{dirty}"
        )
    subject = run(["git", "-C", path, "log", "-1", "--pretty=%s"], timeout=120).stdout.strip()
    committed = run(["git", "-C", path, "log", "-1", "--pretty=%cI"], timeout=120).stdout.strip()
    if branch is not None:
        contains = run(["git", "-C", path, "branch", "--all", "--contains", head],
                       timeout=120).stdout
        if branch not in contains:
            raise RuntimeError(
                f"{name}: commit {head} is not reachable from {branch!r}. "
                f"`git branch --all --contains` reported:\n{contains}"
            )
    repo_commits[name] = head
    repo_details[name] = {
        "path": str(path),
        "url": DAOWOD_REPOSITORY_URL if name == "DAOWOD" else PROB_REPOSITORY_URL,
        "branch": branch,
        "commit": head,
        "subject": subject,
        "committed": committed,
        "clean_after_checkout": True,
    }

table("REPOSITORIES", [
    ("DAOWOD path", DAOWOD_PATH),
    ("DAOWOD commit", repo_commits["DAOWOD"]),
    ("DAOWOD subject", repo_details["DAOWOD"]["subject"]),
    ("DAOWOD committed", repo_details["DAOWOD"]["committed"]),
    ("PROB path", PROB_PATH),
    ("PROB branch", PROB_BRANCH),
    ("PROB commit", repo_commits["PROB"]),
    ("PROB subject", repo_details["PROB"]["subject"]),
    ("PROB committed", repo_details["PROB"]["committed"]),
    ("both clean after checkout", True),
])
mark("repositories")

In [ ]:
# ============================================================================
# G. REQUIRED-FILE AND SOURCE-MARKER VERIFICATION
#
# Every file this experiment cannot run without is checked for existence and for
# the exact source marker that the notebook depends on, before anything is
# installed or compiled. A renamed function or a dropped model output is caught
# here rather than three hours into the campaign.
# ============================================================================
require_stages("repositories")

REQUIRED_CHECKOUT_FILES = {
    # DAOWOD: the acquisition method and the detector adapter.
    f"{DAOWOD_PATH}/pyproject.toml": None,
    f"{DAOWOD_PATH}/src/daowod/experiment.py": "def run_active_round(",
    f"{DAOWOD_PATH}/src/daowod/prob_adapter.py": "class ProposalBatch:",
    f"{DAOWOD_PATH}/src/daowod/acquisition.py": "def score_proposals(",
    # PROB: the bridge, the official entry point, the decoder-feature export and
    # the deformable-attention operator.
    f"{PROB_PATH}/daowod_prob_bridge.py": "def predict(args:",
    f"{PROB_PATH}/main_open_world.py": "def get_args_parser(",
    # The decoder-feature export marker ("'pred_features': hs[-1]") shipped by the
    # pinned commit is validated in the next cell, together with the rest of the
    # clean-checkout assertions. Nothing here modifies the checkout.
    f"{PROB_PATH}/models/prob_deformable_detr.py": "class DeformableDETR(nn.Module):",
    f"{PROB_PATH}/models/ops/setup.py": "MultiScaleDeformableAttention",
    f"{PROB_PATH}/models/ops/functions/ms_deform_attn_func.py":
        "def ms_deform_attn_core_pytorch",
    # TOWOD splits shipped by PROB.
    f"{PROB_PATH}/data/OWOD/ImageSets/{DATASET}/{OFFICIAL_EVAL_SPLIT}.txt": None,
    f"{PROB_PATH}/data/OWOD/ImageSets/{DATASET}/{TASK2_SOURCE_SPLIT}.txt": None,
    f"{PROB_PATH}/data/OWOD/ImageSets/{DATASET}/{TASK1_REFERENCE_SPLIT}.txt": None,
}

# Additional markers that keep the CUDA path honest: the operator sources must
# still carry the modern PyTorch/ATen forms, and the module must still be able to
# route through the compiled function. `value.type()` and THC headers were
# removed from these files and must not come back.
REQUIRED_SOURCE_MARKERS = {
    f"{PROB_PATH}/models/ops/src/cuda/ms_deform_attn_cuda.cu": [
        "value.scalar_type()",
        "TORCH_CHECK(value.is_cuda()",
        "data_ptr<scalar_t>()",
    ],
    f"{PROB_PATH}/models/ops/src/ms_deform_attn.h": [
        "value.is_cuda()",
        "TORCH_CHECK(false",
    ],
    f"{PROB_PATH}/models/ops/setup.py": [
        '"-std=c++17"',
        "CUDAExtension",
    ],
    f"{PROB_PATH}/models/ops/functions/ms_deform_attn_func.py": [
        "import MultiScaleDeformableAttention as MSDA",
        "MSDA.ms_deform_attn_forward(",
        "MSDA.ms_deform_attn_backward(",
    ],
    f"{PROB_PATH}/models/ops/modules/ms_deform_attn.py": [
        "MSDeformAttnFunction.apply(",
        "MSDA_AVAILABLE",
    ],
    f"{PROB_PATH}/util/box_ops.py": [
        'torch.meshgrid(y, x, indexing="ij")',
    ],
}

FORBIDDEN_SOURCE_MARKERS = {
    f"{PROB_PATH}/models/ops/src/cuda/ms_deform_attn_cuda.cu": [
        "value.type()",
        "THCudaCheck",
        "AT_CHECK",
        "#include <THC/THC.h>",
    ],
    f"{PROB_PATH}/models/ops/src/ms_deform_attn.h": [
        "AT_CHECK",
    ],
}

verified_files = 0
for _path, _needle in REQUIRED_CHECKOUT_FILES.items():
    _file = Path(_path)
    if not _file.is_file():
        raise FileNotFoundError(f"The checkout is missing a required file: {_file}")
    if _needle is not None and _needle not in _file.read_text(encoding="utf-8"):
        raise RuntimeError(f"{_file} does not contain the expected marker {_needle!r}.")
    verified_files += 1

verified_markers = 0
for _path, _needles in REQUIRED_SOURCE_MARKERS.items():
    _file = Path(_path)
    if not _file.is_file():
        raise FileNotFoundError(f"The checkout is missing a required file: {_file}")
    _source = _file.read_text(encoding="utf-8")
    for _needle in _needles:
        if _needle not in _source:
            raise RuntimeError(
                f"{_file} does not contain the expected marker {_needle!r}. The "
                "pinned deformable-attention sources are not the modern ones this "
                "notebook builds against."
            )
        verified_markers += 1

for _path, _needles in FORBIDDEN_SOURCE_MARKERS.items():
    _source = Path(_path).read_text(encoding="utf-8")
    for _needle in _needles:
        if _needle in _source:
            raise RuntimeError(
                f"{_path} still contains the deprecated form {_needle!r}. The "
                "pinned commit is expected to use the modern ATen API "
                "(value.scalar_type(), TORCH_CHECK, data_ptr<scalar_t>())."
            )

# The bridge must compile under this interpreter before it is ever invoked.
run([sys.executable, "-m", "py_compile", str(Path(PROB_PATH, "daowod_prob_bridge.py"))],
    cwd=PROB_PATH, timeout=300)

split_line_counts = {
    split: len(read_ids(Path(PROB_PATH, "data", "OWOD", "ImageSets", DATASET, f"{split}.txt")))
    for split in (TASK2_SOURCE_SPLIT, TASK1_REFERENCE_SPLIT, OFFICIAL_EVAL_SPLIT)
}
for split, count in split_line_counts.items():
    if count < 1000:
        raise RuntimeError(
            f"The shipped split {split}.txt holds only {count} image IDs, which is "
            "far below the expected M-OWODB size; the checkout is incomplete."
        )

table("CHECKOUT VERIFICATION", [
    ("required files verified", verified_files),
    ("source markers verified", verified_markers),
    ("deprecated forms absent", sum(len(v) for v in FORBIDDEN_SOURCE_MARKERS.values())),
    ("bridge compiles", True),
    *[(f"split {split}", f"{count} image IDs")
      for split, count in split_line_counts.items()],
])
mark("checkout files")

In [ ]:
# ============================================================================
# G2. PROB DECODER-FEATURE EXPORT: CLEAN-CHECKOUT VALIDATION
#
# WHY THIS MATTERS
# ----------------
# daowod_prob_bridge.py requires PROB's forward pass to publish the final decoder
# hidden states:
#
#     'pred_features': hs[-1]
#
# check_patch() refuses to run *any* bridge subcommand without it, and `predict`
# reads outputs["pred_features"] to build the proposal embeddings the novelty,
# rarity and coherence terms are computed from. Without them there is no
# acquisition signal at all.
#
# WHAT THIS CELL DOES
# -------------------
# The pinned PROB commit ships that export itself, on the same branch that
# carries the modern-PyTorch initialisation of models/prob_deformable_detr.py, so
# there is nothing to modify. This notebook never writes to a PROB source file,
# applies no diff and monkey-patches nothing. This cell only validates the
# freshly cloned, untouched checkout:
#
#   1. `git rev-parse HEAD` equals the full 40-character PROB_COMMIT.
#   2. `git status --porcelain` is empty before anything reads the checkout.
#   3. models/prob_deformable_detr.py contains 'pred_features': hs[-1], exactly
#      once, and compiles under this interpreter.
#   4. The modern in-place `torch.no_grad()` initialisation is present.
#   5. The deprecated `.data =` initialisation forms have not returned.
#   6. `python daowod_prob_bridge.py check` exits 0 and reports the export.
#   7. `git status --porcelain` is still empty afterwards, so validating the
#      checkout left it pristine.
#
# Any failure stops the run: the fix is to re-pin PROB_COMMIT to a commit that
# satisfies these assertions, never to edit the checkout from here. Every result
# is recorded in PROB_SOURCE_VALIDATION and copied into the experiment manifest
# saved to Drive.
# ============================================================================
require_stages("checkout files")

PROB_MODEL_FILE = "models/prob_deformable_detr.py"
DECODER_FEATURE_MARKER = "'pred_features': hs[-1]"
# The exact wording daowod_prob_bridge.py prints when the export is present.
BRIDGE_CHECK_EXPECTED_LINE = "PROB decoder-feature patch: OK"
# The initialisation forms that make this branch the correct pin.
MODERN_INITIALISATION_MARKERS = (
    "with torch.no_grad():",
    "nn.init.constant_(",
    "self.class_embed.bias.fill_(bias_value)",
    "self.inv_obj_cov.copy_(",
)
# The deprecated forms those replaced. A bare ".data" search would match the
# legitimate `args.dataset_file`, so the exact assignment forms are listed.
DEPRECATED_INITIALISATION_PATTERNS = (
    ".data = ",
    ".data=",
    ".weight.data",
    ".bias.data",
    ".data.copy_(",
    ".data.fill_(",
    ".data.zero_(",
    ".data.uniform_(",
    ".data.normal_(",
)

# --- 1. the checkout is exactly the pinned commit ---------------------------
prob_head = run(["git", "-C", PROB_PATH, "rev-parse", "HEAD"], timeout=120).stdout.strip()
if len(prob_head) != 40:
    raise RuntimeError(
        f"`git rev-parse HEAD` in {PROB_PATH} returned {prob_head!r}, which is not "
        "a full 40-character hash; the equality check below would be meaningless."
    )
if prob_head != PROB_COMMIT:
    raise RuntimeError(
        f"The PROB checkout is at {prob_head}, not the pinned {PROB_COMMIT}."
    )

# --- 2. and it is clean before anything touches it --------------------------
worktree_before = run(["git", "-C", PROB_PATH, "status", "--porcelain"],
                      timeout=300).stdout.strip()
if worktree_before:
    raise RuntimeError(
        f"The pinned PROB checkout at {PROB_PATH} is not clean:\n{worktree_before}\n"
        "This notebook requires a pristine checkout — it never modifies PROB, so "
        "any difference from the pinned commit is unexplained."
    )

# --- 3. the decoder-feature export the bridge depends on --------------------
model_path = Path(PROB_PATH, PROB_MODEL_FILE)
if not model_path.is_file():
    raise FileNotFoundError(f"The PROB checkout is missing {model_path}.")
model_source = model_path.read_text(encoding="utf-8")
if DECODER_FEATURE_MARKER not in model_source:
    raise RuntimeError(
        f"{model_path} does not contain {DECODER_FEATURE_MARKER!r}.\n"
        f"The pinned commit {PROB_COMMIT} is expected to export the final decoder "
        "hidden states: without them `daowod_prob_bridge.py check` fails and "
        "`predict` has no embeddings for the novelty, rarity and coherence terms.\n"
        "Nothing is edited here. Re-pin PROB_COMMIT to a commit that exports "
        "pred_features."
    )
marker_occurrences = (model_source.count("'pred_features'")
                      + model_source.count('"pred_features"'))
if marker_occurrences != 1:
    raise RuntimeError(
        f"{model_path} mentions pred_features {marker_occurrences} times; the "
        "pinned source is expected to export it exactly once."
    )
model_file_sha256 = sha256_of_file(model_path)

# The model file must import cleanly under this interpreter. PROB's .gitignore
# excludes __pycache__/, so byte-compiling it cannot dirty the checkout — which
# assertion 7 below confirms rather than assumes.
run([sys.executable, "-m", "py_compile", str(model_path)], cwd=PROB_PATH, timeout=300)

# --- 4. the modern-PyTorch initialisation is present ------------------------
missing_modern = [marker for marker in MODERN_INITIALISATION_MARKERS
                  if marker not in model_source]
if missing_modern:
    raise RuntimeError(
        f"{model_path} does not contain {missing_modern}. The pinned branch "
        "initialises parameters in place under `with torch.no_grad():`, and this "
        "checkout is not that source."
    )

# --- 5. and the deprecated initialisation forms have not returned -----------
returned_deprecated = [pattern for pattern in DEPRECATED_INITIALISATION_PATTERNS
                       if pattern in model_source]
if returned_deprecated:
    raise RuntimeError(
        f"{model_path} contains the deprecated initialisation form(s) "
        f"{returned_deprecated}. The pinned branch replaced `.data` assignment "
        "with in-place no-grad updates and they must not come back."
    )

# --- 6. the bridge's own verification, straight from the clean checkout ------
bridge_check_command = [sys.executable,
                        str(Path(PROB_PATH, "daowod_prob_bridge.py")), "check"]
bridge_check = run(bridge_check_command, cwd=PROB_PATH, timeout=600, check=False)
if bridge_check.returncode != 0:
    report_failure_context(bridge_check_command, PROB_PATH, bridge_check)
    raise RuntimeError(
        "`python daowod_prob_bridge.py check` returned "
        f"{bridge_check.returncode} on the clean pinned checkout; exit code 0 is "
        "required before any PROB run is attempted."
    )
if BRIDGE_CHECK_EXPECTED_LINE not in bridge_check.stdout:
    raise RuntimeError(
        "`python daowod_prob_bridge.py check` exited 0 but did not report "
        f"{BRIDGE_CHECK_EXPECTED_LINE!r}:\n{bridge_check.stdout}"
    )

# --- 7. validating the checkout did not modify it ---------------------------
worktree_after = run(["git", "-C", PROB_PATH, "status", "--porcelain"],
                     timeout=300).stdout.strip()
if worktree_after:
    raise RuntimeError(
        f"Validating the pinned PROB checkout left it dirty:\n{worktree_after}\n"
        "Nothing in this notebook may modify PROB."
    )
head_after = run(["git", "-C", PROB_PATH, "rev-parse", "HEAD"], timeout=120).stdout.strip()
if head_after != PROB_COMMIT:
    raise RuntimeError(
        f"The PROB checkout moved to {head_after} during validation."
    )

PROB_SOURCE_VALIDATION = {
    "repository": PROB_REPOSITORY_URL,
    "branch": PROB_BRANCH,
    "expected_commit": PROB_COMMIT,
    "commit": prob_head,
    "commit_verified": True,
    "modified_by_notebook": False,
    "clean_checkout_before_validation": True,
    "clean_checkout_after_validation": True,
    "model_file": PROB_MODEL_FILE,
    "model_file_sha256": model_file_sha256,
    "model_file_compiles": True,
    "decoder_feature_marker": DECODER_FEATURE_MARKER,
    "decoder_feature_marker_present": True,
    "decoder_feature_marker_occurrences": marker_occurrences,
    "modern_initialisation_markers_present": list(MODERN_INITIALISATION_MARKERS),
    "deprecated_initialisation_patterns_absent": list(DEPRECATED_INITIALISATION_PATTERNS),
    "bridge_check_command": [str(part) for part in bridge_check_command],
    "bridge_check_returncode": bridge_check.returncode,
    "bridge_check_ok": True,
}
repo_details["PROB"]["clean_after_validation"] = True
repo_details["PROB"]["modified_by_notebook"] = False

table("PROB SOURCE VALIDATION", [
    ("PROB path", PROB_PATH),
    ("commit (expected == HEAD)", f"{PROB_COMMIT} == {prob_head}"),
    ("clean before validation", True),
    ("model file", model_path),
    ("model file sha256", model_file_sha256),
    ("decoder-feature marker", f"{DECODER_FEATURE_MARKER} (x{marker_occurrences})"),
    ("model file compiles", True),
    ("modern initialisation", f"{len(MODERN_INITIALISATION_MARKERS)} markers present"),
    ("deprecated `.data =` forms",
     f"{len(DEPRECATED_INITIALISATION_PATTERNS)} patterns absent"),
    ("bridge check return code", bridge_check.returncode),
    ("clean after validation", True),
    ("modified by this notebook", False),
])
print("the pinned PROB checkout is clean, unmodified and exports the decoder "
      "features the bridge requires")
mark("prob source")


In [ ]:
# ============================================================================
# H. SYSTEM AND PYTHON DEPENDENCIES, PLUS THE REPOSITORY'S OWN GATES
#
# Only what is missing is installed, and always with this kernel's interpreter.
# PROB/requirements.txt is deliberately NOT installed: it pins wandb 0.13.4,
# pandas 1.5.1 and scikit-image 0.19.2, and resolving those on a current Colab
# runtime would drag numpy and torch backwards and invalidate the CUDA extension
# built in cell J. Instead the modules the executed PROB code paths import are
# checked one by one, and the torch/torchvision/CUDA versions recorded in cell D
# are re-asserted afterwards.
#
# The DAOWOD `[dev]` extra is not requested either: it pins ipykernel>=7, and
# upgrading the kernel's own machinery mid-session is an avoidable hazard. The two
# tools its gates actually need — ruff and pytest — are installed directly.
# ============================================================================
require_stages("prob source")

# zstd decompresses .tar.zst dataset archives; ninja and the toolchain build the
# deformable-attention CUDA op.
run(["apt-get", "update", "-qq"], timeout=1200)
run(["apt-get", "install", "-y", "-qq", "zstd", "ninja-build", "build-essential"],
    timeout=2400)
for _tool in ("zstd", "ninja", "tar", "cc", "c++"):
    if shutil.which(_tool) is None:
        raise RuntimeError(f"{_tool} is still not on PATH after apt-get install.")

run([sys.executable, "-m", "pip", "install", *DAOWOD_PIP_FLAGS,
     "--editable", DAOWOD_PATH], timeout=3600)
run([sys.executable, "-m", "pip", "install", "ruff>=0.15,<1", "pytest>=9,<10"],
    timeout=3600)

# Third-party modules imported by the PROB code paths this experiment executes,
# plus the two the notebook itself needs for tables and plots.
PROB_MODULE_PACKAGES = {
    "PIL": "pillow",
    "einops": "einops",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "pycocotools": "pycocotools",
    "scipy": "scipy",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "torch": None,
    "torchvision": None,
    "tqdm": "tqdm",
    "wandb": "wandb",
    "yaml": "PyYAML",
}
missing_packages = sorted(
    package
    for module, package in PROB_MODULE_PACKAGES.items()
    if package is not None and importlib.util.find_spec(module) is None
)
if missing_packages:
    print(f"installing missing modules: {missing_packages}")
    run([sys.executable, "-m", "pip", "install", *missing_packages], timeout=3600)
else:
    print("every required module is already importable; nothing installed")
importlib.invalidate_caches()
still_missing = sorted(
    module for module in PROB_MODULE_PACKAGES
    if importlib.util.find_spec(module) is None
)
if still_missing:
    raise RuntimeError(f"PROB dependencies still unavailable: {still_missing}")

# Installing dependencies must not have replaced the CUDA-enabled torch build,
# and a fresh interpreter must still reach the GPU.
post_install = json_probe(
    torch_snippet(
        "import json, torchvision",
        "print(json.dumps({",
        "    'torch': torch.__version__,",
        "    'torchvision': torchvision.__version__,",
        "    'cuda': torch.version.cuda,",
        "    'cuda_available': torch.cuda.is_available(),",
        "    'device_count': torch.cuda.device_count(),",
        "}))",
    ),
    cwd=CONTENT_ROOT, timeout=1200,
)
if not post_install["cuda_available"] or post_install["device_count"] < 1:
    raise RuntimeError(
        "After installing dependencies a fresh interpreter can no longer see the "
        f"GPU: {post_install}. A pip resolution replaced the CUDA torch build."
    )
for _key, _expected in BASELINE_VERSIONS.items():
    if post_install[_key] != _expected:
        raise RuntimeError(
            f"pip changed {_key} from {_expected!r} to {post_install[_key]!r}; the "
            "PROB CUDA extension would then be built against a different torch. "
            "Re-run this notebook in a fresh runtime."
        )
print(f"post-install runtime unchanged: {post_install}")
mark("dependencies")

# --- the daowod package must come from the pinned clone ---------------------
daowod_src = str(Path(DAOWOD_PATH, "src"))
if daowod_src not in sys.path:
    sys.path.insert(0, daowod_src)
importlib.invalidate_caches()

import numpy as np

import daowod
from daowod import ProbAdapter, ProposalBatch, run_active_round
from daowod.acquisition import AcquisitionWeights, score_proposals, select_images
from daowod.config import AcquisitionConfig
from daowod.dataset import build_long_tail_pool

expected_package = Path(DAOWOD_PATH, "src", "daowod").resolve()
imported_from = Path(daowod.__file__).resolve()
if imported_from.parent != expected_package:
    raise RuntimeError(
        f"daowod was imported from {imported_from}, not from the pinned clone "
        f"at {expected_package}."
    )

pip_show = run([sys.executable, "-m", "pip", "show", "--files",
                "distribution-aware-owod"], timeout=600).stdout
editable_location = next(
    (line.split(":", 1)[1].strip() for line in pip_show.splitlines()
     if line.startswith("Editable project location:")),
    None,
)
if editable_location is None:
    raise RuntimeError(
        "distribution-aware-owod is installed but not as an editable install; "
        f"`pip show` reported:\n{pip_show}"
    )
if Path(editable_location).resolve() != Path(DAOWOD_PATH).resolve():
    raise RuntimeError(
        f"The editable install points at {editable_location}, not at the pinned "
        f"clone {DAOWOD_PATH}."
    )
subprocess_import = python_probe(
    "import daowod, daowod.experiment, daowod.prob_adapter; print(daowod.__file__)",
    cwd=CONTENT_ROOT, timeout=600,
).stdout.strip().splitlines()[-1]
if Path(subprocess_import).resolve() != imported_from:
    raise RuntimeError(
        f"A fresh interpreter imports daowod from {subprocess_import}, but this "
        f"session imported {imported_from}."
    )

required_round_parameters = {
    "adapter", "checkpoint", "candidate_ids", "reference_ids", "labelled_ids",
    "output_dir", "strategy", "budget", "acquisition_config", "seed", "round_index",
}
round_signature = inspect.signature(run_active_round)
missing_parameters = required_round_parameters - set(round_signature.parameters)
if missing_parameters:
    raise RuntimeError(f"run_active_round lacks parameters: {sorted(missing_parameters)}")
for variant_name, variant_config in VARIANTS.items():
    if variant_config["strategy"] not in {"random", "rarity_no_coherence", "full"}:
        raise RuntimeError(
            f"{variant_name}: run_active_round accepts only 'random', "
            "'rarity_no_coherence' and 'full'."
        )

table("DAOWOD PACKAGE", [
    ("daowod.__file__", imported_from),
    ("editable project location", editable_location),
    ("fresh-interpreter import", subprocess_import),
    ("run_active_round", str(round_signature)),
    ("ProposalBatch fields", ", ".join(ProposalBatch.__dataclass_fields__)),
    ("numpy", np.__version__),
    ("pip flags used", DAOWOD_PIP_FLAGS or "none"),
])

# --- the repository's own quality gates, on this interpreter ----------------
run([sys.executable, "-m", "ruff", "check", "."], cwd=DAOWOD_PATH, timeout=900)
run([sys.executable, "-m", "pytest", "-q"], cwd=DAOWOD_PATH, timeout=3600)
run([sys.executable, "-m", "compileall", "-q", "src", "tests"],
    cwd=DAOWOD_PATH, timeout=900)
mark("daowod gates")

In [ ]:
# ============================================================================
# I. DINO RESNET-50 BACKBONE WEIGHTS
#
# PROB/models/backbone.py loads models/dino_resnet50_pretrain.pth from the
# repository directory. The download is verified by exact byte count and SHA-256
# so a truncated or proxy-mangled file cannot silently become a randomly
# initialised backbone. A file that already matches is not downloaded again.
# ============================================================================
require_stages("daowod gates")

backbone_path = Path(PROB_PATH, "models", "dino_resnet50_pretrain.pth")


def backbone_is_valid(path):
    if not path.is_file():
        return False, "absent"
    size = path.stat().st_size
    if size != DINO_BACKBONE_BYTES:
        return False, f"{size} bytes, expected {DINO_BACKBONE_BYTES}"
    digest = sha256_of_file(path)
    if digest != DINO_BACKBONE_SHA256:
        return False, f"sha256 {digest}, expected {DINO_BACKBONE_SHA256}"
    return True, f"{size} bytes, sha256 {digest}"


valid, reason = backbone_is_valid(backbone_path)
if valid:
    print(f"{backbone_path} already present and verified: {reason}")
    backbone_downloaded = False
else:
    print(f"downloading the DINO backbone ({reason}) from {DINO_BACKBONE_URL}")
    if backbone_path.exists():
        backbone_path.unlink()
    download_snippet = "\n".join([
        "import shutil, sys, urllib.request",
        "source, destination = sys.argv[1], sys.argv[2]",
        "temporary = destination + '.partial'",
        "with urllib.request.urlopen(source, timeout=600) as response, "
        "open(temporary, 'wb') as handle:",
        "    shutil.copyfileobj(response, handle, 1024 * 1024)",
        "shutil.move(temporary, destination)",
        "print('downloaded', destination)",
    ])
    python_probe(download_snippet, cwd=PROB_PATH, timeout=3600,
                 arguments=[DINO_BACKBONE_URL, str(backbone_path)])
    backbone_downloaded = True
    valid, reason = backbone_is_valid(backbone_path)
    if not valid:
        raise RuntimeError(
            f"The downloaded DINO backbone does not match the pinned artifact "
            f"({reason}).\n  url      : {DINO_BACKBONE_URL}\n"
            f"  expected : {DINO_BACKBONE_BYTES} bytes, sha256 {DINO_BACKBONE_SHA256}\n"
            "If upstream legitimately republished the file, update "
            "DINO_BACKBONE_BYTES and DINO_BACKBONE_SHA256 in the configuration "
            "cell after verifying the new artifact yourself."
        )

# Readability is proven by actually loading it, not by stat alone.
backbone_state = torch.load(backbone_path, map_location="cpu", weights_only=False)
if not isinstance(backbone_state, dict) or not backbone_state:
    raise RuntimeError(f"{backbone_path} is not a state dict.")
backbone_tensor_count = sum(1 for value in backbone_state.values() if torch.is_tensor(value))
backbone_parameter_count = sum(
    int(value.numel()) for value in backbone_state.values() if torch.is_tensor(value)
)
if backbone_tensor_count == 0:
    raise RuntimeError(f"{backbone_path} contains no tensors.")
if not any(key.startswith("conv1") or "layer4" in key for key in backbone_state):
    raise RuntimeError(
        f"{backbone_path} does not look like a ResNet-50 state dict; keys begin "
        f"with {sorted(backbone_state)[:5]}."
    )
del backbone_state
cleanup_runtime()

table("DINO BACKBONE", [
    ("path", backbone_path),
    ("url", DINO_BACKBONE_URL),
    ("downloaded now", backbone_downloaded),
    ("bytes", f"{backbone_path.stat().st_size:,} (expected {DINO_BACKBONE_BYTES:,})"),
    ("sha256", DINO_BACKBONE_SHA256),
    ("sha256 verified", True),
    ("tensors", backbone_tensor_count),
    ("parameters", f"{backbone_parameter_count:,}"),
])
mark("dino weights")

In [ ]:
# ============================================================================
# J. MULTISCALEDEFORMABLEATTENTION: CLEAN BUILD
#
# Three things are kept strictly separate, in this order:
#
#   J  pip build/install result   <- this cell, and ONLY this cell, can declare
#                                    a build failure
#   K  extension import           <- next cell
#   K  CUDA forward/backward      <- next cell
#
# A pip return code of 0 means the extension compiled and installed. Nothing that
# happens afterwards is allowed to relabel that as a compilation failure.
#
# models/ops/make.sh is not used: it runs `python setup.py build install`, which
# resolves `python` from PATH instead of this kernel's interpreter, and
# `setup.py install` was removed from modern setuptools. Installing the very same
# setup.py through pip with build isolation disabled compiles the identical
# CUDAExtension against the torch verified in cell D. The setup.py deprecation
# warning and the unused-`q_col` nvcc warning are warnings, not errors.
# ============================================================================
require_stages("dino weights")

ops_dir = Path(PROB_PATH, "models", "ops")
if not (ops_dir / "setup.py").is_file():
    raise FileNotFoundError(f"{ops_dir}/setup.py is missing.")

# --- 1. remove any previously installed copy --------------------------------
run([sys.executable, "-m", "pip", "uninstall", "-y", "MultiScaleDeformableAttention"],
    timeout=1800, check=False, quiet=True)

# --- 2. remove only known generated artifacts -------------------------------
stale_build_artifacts = [
    ops_dir / "build",
    ops_dir / "dist",
    *sorted(ops_dir.glob("*.egg-info")),
    *sorted(ops_dir.glob("*.egg-link")),
    *sorted(ops_dir.rglob("*.so")),
    *sorted(ops_dir.rglob("__pycache__")),
    # A previous install may have left the module or its metadata behind even
    # after pip uninstall (for example from a legacy `setup.py install` egg).
    *sorted(site_packages_dir.glob("MultiScaleDeformableAttention*")),
    # setuptools/ninja scratch space for torch C++/CUDA extensions.
    Path.home() / ".cache" / "torch_extensions",
]
remove_paths(stale_build_artifacts, description="stale extension artifacts")

importlib.invalidate_caches()
if importlib.util.find_spec("MultiScaleDeformableAttention") is not None:
    raise RuntimeError(
        "MultiScaleDeformableAttention is still resolvable after the clean-up, so "
        "the build below would not be validated against a fresh artifact. Restart "
        "the runtime and run the notebook again."
    )

# --- 3. build environment ---------------------------------------------------
# Compile for exactly this GPU (plus PTX for forward compatibility) instead of
# every architecture torch knows about: it is faster and it guarantees the binary
# matches the device the smoke test in cell K runs on.
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{gpu_architecture}+PTX"
os.environ["MAX_JOBS"] = str(EXTENSION_BUILD_MAX_JOBS)
if CUDA_HOME:
    os.environ["CUDA_HOME"] = str(CUDA_HOME)

table("BUILD ENVIRONMENT", [
    ("interpreter", sys.executable),
    ("source directory", ops_dir),
    ("torch", torch.__version__),
    ("torch CUDA", torch.version.cuda),
    ("CUDA_HOME", os.environ.get("CUDA_HOME")),
    ("nvcc", nvcc_path),
    ("TORCH_CUDA_ARCH_LIST", os.environ["TORCH_CUDA_ARCH_LIST"]),
    ("MAX_JOBS", os.environ["MAX_JOBS"]),
    ("ninja", shutil.which("ninja")),
    ("LD_LIBRARY_PATH", os.environ.get("LD_LIBRARY_PATH", "")),
])

# --- 4. build ---------------------------------------------------------------
# --no-cache-dir defeats pip's wheel cache, so this is a real compile rather than
# the replay of a wheel built during an earlier attempt.
ops_install_command = [
    sys.executable, "-m", "pip", "install",
    "--no-build-isolation", "--no-deps", "--force-reinstall", "--no-cache-dir",
    str(ops_dir),
]
build_started = time.time()
compile_result = run(ops_install_command, cwd=ops_dir, timeout=7200, check=False)
build_seconds = round(time.time() - build_started, 1)

# --- 5. the only place a build failure can be declared ----------------------
if compile_result.returncode != 0:
    report_failure_context(ops_install_command, ops_dir, compile_result)
    raise RuntimeError(
        "BUILD FAILURE: compiling PROB's MultiScaleDeformableAttention extension "
        f"returned {compile_result.returncode} (a non-zero pip exit code). The "
        "compiler output is printed above.\n"
        f"  interpreter          : {sys.executable}\n"
        f"  torch / CUDA         : {torch.__version__} / {torch.version.cuda}\n"
        f"  nvcc                 : {nvcc_path} ({nvcc_release})\n"
        f"  CUDA_HOME            : {os.environ.get('CUDA_HOME')}\n"
        f"  TORCH_CUDA_ARCH_LIST : {os.environ['TORCH_CUDA_ARCH_LIST']}\n"
        f"  MAX_JOBS             : {os.environ['MAX_JOBS']}\n"
        f"  free /content GB     : {free_gb(CONTENT_ROOT)}\n"
        "This notebook has no pure-PyTorch fallback: PROB's reference kernel is "
        "roughly an order of magnitude slower and this twelve-run campaign would "
        "not finish with it."
    )

print()
print("pip returned 0: the extension compiled and installed successfully.")
print("Import validation and CUDA execution are checked in the next cell. Neither")
print("of them can turn this successful build into a build failure; if they fail,")
print("they report an import or execution problem with its own diagnostics.")

if "Successfully installed" not in compile_result.stdout:
    print("\nNOTE: pip returned 0 but did not print 'Successfully installed'. The "
          "install is still treated as successful, on the return code.")

# --- 6. what the build produced --------------------------------------------
installed_files = run([sys.executable, "-m", "pip", "show", "--files",
                       "MultiScaleDeformableAttention"], timeout=600, check=False)
extension_binaries = sorted(site_packages_dir.glob("MultiScaleDeformableAttention*.so"))
build_directory_exists = (ops_dir / "build").is_dir()

worktree_after_build = run(["git", "-C", PROB_PATH, "status", "--porcelain"],
                           timeout=300).stdout
tracked_after_build = sorted(
    line[3:] for line in worktree_after_build.splitlines()
    if line.strip() and not line.startswith("??")
)
if tracked_after_build:
    raise RuntimeError(
        f"The build modified tracked files in the PROB checkout: {tracked_after_build}. "
        "The pinned checkout must stay clean: no tracked file may differ from the "
        "pinned commit."
    )

EXTENSION_BUILD = {
    "command": [str(part) for part in ops_install_command],
    "cwd": str(ops_dir),
    "returncode": compile_result.returncode,
    "seconds": build_seconds,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "nvcc": nvcc_path,
    "nvcc_release": nvcc_release,
    "arch_list": os.environ["TORCH_CUDA_ARCH_LIST"],
    "max_jobs": os.environ["MAX_JOBS"],
    "binaries": [str(path) for path in extension_binaries],
    "installed_reported": "Successfully installed" in compile_result.stdout,
}

table("EXTENSION BUILD", [
    ("pip return code", compile_result.returncode),
    ("verdict", "BUILD SUCCEEDED"),
    ("build seconds", build_seconds),
    ("build/ directory created", build_directory_exists),
    ("installed .so", [str(path.name) for path in extension_binaries] or "none found yet"),
    ("installed bytes", [path.stat().st_size for path in extension_binaries] or "n/a"),
    ("pip show returncode", installed_files.returncode),
    ("tracked files modified", tracked_after_build or "none"),
    ("free /content GB", free_gb(CONTENT_ROOT)),
])
mark("extension build")

In [ ]:
# ============================================================================
# K. MULTISCALEDEFORMABLEATTENTION: IMPORT AND REAL CUDA FORWARD/BACKWARD
#
# The build in cell J already succeeded. This cell answers three *different*
# questions, and a failure here is reported as what it is — an import or an
# execution problem — never as a compilation failure.
#
#   1. Does `import MultiScaleDeformableAttention` succeed in a fresh process?
#   2. Does the compiled kernel produce the same numbers as PROB's own reference
#      implementation, forwards, on the GPU?
#   3. Do real gradients flow through it (autograd backward plus a numerical
#      gradcheck), and is every gradient finite?
#
# THE libc10.so FALSE NEGATIVE
# ---------------------------
# The extension links against libc10.so from the torch wheel and carries no
# RPATH to it, so importing it in a process that has not already imported torch
# raises `ImportError: libc10.so: cannot open shared object file` even when the
# build and install both succeeded. Every snippet below is constructed with
# torch_snippet(), whose first line is always `import torch`, and cell D already
# prepended the torch library directory to the LD_LIBRARY_PATH that children
# inherit. That is why the import here is a real signal and not an artefact.
#
# Note that PROB itself never hits that trap: models/ops/functions/
# ms_deform_attn_func.py imports torch at module scope before it imports the
# extension. It does, however, degrade *silently* — `MSDA_AVAILABLE = False` makes
# models/ops/modules/ms_deform_attn.py substitute the ~10x slower pure-PyTorch
# kernel with no error at all. So MSDA_AVAILABLE is asserted here as well.
# ============================================================================
require_stages("extension build")

# --- 1. import in a fresh process, from a directory unrelated to the repo ----
IMPORT_PROBE = torch_snippet(
    "import json",
    "import MultiScaleDeformableAttention as msda",
    "print(json.dumps({",
    "    'file': msda.__file__,",
    "    'has_forward': hasattr(msda, 'ms_deform_attn_forward'),",
    "    'has_backward': hasattr(msda, 'ms_deform_attn_backward'),",
    "    'torch': torch.__version__,",
    "    'torch_cuda': torch.version.cuda,",
    "    'gpu': torch.cuda.get_device_name(0),",
    "}))",
)
import_result = python_probe(IMPORT_PROBE, cwd=CONTENT_ROOT, timeout=1200, check=False)

if import_result.returncode != 0:
    banner("EXTENSION IMPORT FAILED (THE BUILD IN CELL J DID NOT FAIL)")
    print("The pip build and install returned 0. What failed is importing the")
    print("installed module, which is a different problem with different causes.")
    banner("IMPORT STDERR")
    print(import_result.stderr)
    banner("IMPORT STDOUT")
    print(import_result.stdout)
    banner("SHARED LIBRARY ENVIRONMENT")
    print("LD_LIBRARY_PATH =")
    for _entry in os.environ.get("LD_LIBRARY_PATH", "").split(os.pathsep):
        print(f"  {_entry}")
    print(f"torch library dir : {torch_library_dir}")
    print(f"libc10 present    : {sorted(p.name for p in torch_library_dir.glob('libc10*'))}")
    banner("INSTALLED ARTIFACTS")
    for _path in sorted(site_packages_dir.glob("MultiScaleDeformableAttention*")):
        print(f"  {_path}  ({_path.stat().st_size:,} bytes)")
    run([sys.executable, "-m", "pip", "show", "--files",
         "MultiScaleDeformableAttention"], timeout=600, check=False)
    for _binary in sorted(site_packages_dir.glob("MultiScaleDeformableAttention*.so")):
        if shutil.which("ldd"):
            run(["ldd", str(_binary)], timeout=600, check=False)
    report_failure_context([sys.executable, "-c", "<import probe>"], CONTENT_ROOT,
                           import_result)
    raise RuntimeError(
        "IMPORT FAILURE: MultiScaleDeformableAttention was built and installed "
        f"successfully (pip returncode {EXTENSION_BUILD['returncode']}) but cannot "
        "be imported. The exact ImportError, the LD_LIBRARY_PATH, the installed "
        "artifacts and `ldd` output are above. This is not a compilation failure."
    )

extension_module = json.loads(
    [line for line in import_result.stdout.splitlines() if line.strip()][-1]
)
for _entry_point in ("has_forward", "has_backward"):
    if not extension_module[_entry_point]:
        raise RuntimeError(
            f"The imported extension lacks {_entry_point.replace('has_', 'ms_deform_attn_')}: "
            f"{extension_module}. The binary does not expose the pybind entry "
            "points PROB calls."
        )
print(f"\nextension imported: {extension_module}")

# --- 2 and 3. numerical agreement, autograd backward, and gradcheck ---------
# Sizes follow models/ops/test.py, the upstream reference test for this operator.
OPERATOR_PROBE = torch_snippet(
    "import json",
    "from torch.autograd import gradcheck",
    "import MultiScaleDeformableAttention as msda",
    "from models.ops.functions.ms_deform_attn_func import (",
    "    MSDA,",
    "    MSDA_AVAILABLE,",
    "    MSDeformAttnFunction,",
    "    ms_deform_attn_core_pytorch,",
    ")",
    "",
    "assert MSDA_AVAILABLE, (",
    "    'models/ops/functions/ms_deform_attn_func.py set MSDA_AVAILABLE=False, so "
    "PROB would silently use the slow pure-PyTorch kernel.')",
    "assert MSDA is not None, 'MSDA is None inside the PROB package.'",
    "assert MSDA.__file__ == msda.__file__, (MSDA.__file__, msda.__file__)",
    "",
    "device = torch.device('cuda')",
    "torch.manual_seed(3)",
    "N, M, D_ = 2, 2, 4",
    "Lq, L, P = 2, 2, 2",
    "shapes = torch.as_tensor([(6, 4), (3, 2)], dtype=torch.long, device=device)",
    "starts = torch.cat((shapes.new_zeros((1,)), shapes.prod(1).cumsum(0)[:-1]))",
    "S = int(shapes.prod(1).sum())",
    "# The reference kernel is given plain Python sizes so it never depends on",
    "# implicit device-scalar conversion.",
    "reference_shapes = [(int(h), int(w)) for h, w in shapes.tolist()]",
    "im2col_step = 64  # the value MSDeformAttn uses; the kernel clamps it to batch",
    "results = {'extension': msda.__file__, 'msda_available': bool(MSDA_AVAILABLE)}",
    "",
    "def make_inputs(dtype):",
    "    value = (torch.rand(N, S, M, D_, device=device, dtype=dtype) * 0.01)",
    "    locations = torch.rand(N, Lq, M, L, P, 2, device=device, dtype=dtype)",
    "    weights = torch.rand(N, Lq, M, L, P, device=device, dtype=dtype) + 1e-5",
    "    weights = weights / weights.sum(-1, keepdim=True).sum(-2, keepdim=True)",
    "    return value, locations, weights",
    "",
    "for name, dtype, rtol, atol in (('float64', torch.float64, 1e-9, 1e-9),",
    "                               ('float32', torch.float32, 1e-2, 1e-3)):",
    "    value, locations, weights = make_inputs(dtype)",
    "    for tensor in (value, locations, weights):",
    "        assert tensor.is_cuda, 'inputs must be CUDA tensors'",
    "        assert tensor.is_contiguous(), 'inputs must be contiguous'",
    "    value.requires_grad_(True)",
    "    locations.requires_grad_(True)",
    "    weights.requires_grad_(True)",
    "    compiled = MSDeformAttnFunction.apply(",
    "        value, shapes, starts, locations, weights, im2col_step)",
    "    reference = ms_deform_attn_core_pytorch(",
    "        value, reference_shapes, locations, weights)",
    "    assert compiled.shape == (N, Lq, M * D_), compiled.shape",
    "    assert compiled.shape == reference.shape, (compiled.shape, reference.shape)",
    "    assert torch.isfinite(compiled).all(), 'compiled forward produced non-finite values'",
    "    difference = float((compiled - reference).abs().max())",
    "    assert torch.allclose(compiled, reference, rtol=rtol, atol=atol), (",
    "        f'{name}: compiled kernel disagrees with the reference by {difference}')",
    "    compiled.square().mean().backward()",
    "    gradients = {}",
    "    for label, tensor in (('value', value), ('locations', locations),",
    "                          ('weights', weights)):",
    "        assert tensor.grad is not None, f'{name}: no gradient for {label}'",
    "        assert torch.isfinite(tensor.grad).all(), f'{name}: non-finite {label} gradient'",
    "        gradients[label] = float(tensor.grad.abs().max())",
    "    assert max(gradients.values()) > 0.0, f'{name}: every gradient is exactly zero'",
    "    results[name] = {",
    "        'output_shape': list(compiled.shape),",
    "        'max_abs_difference': difference,",
    "        'output_absmax': float(compiled.abs().max()),",
    "        'gradient_absmax': gradients,",
    "    }",
    "",
    "# Definitive backward validation: a numerical gradcheck in float64. The",
    "# kernel accumulates grad_value with atomicAdd, so the analytical jacobian is",
    "# not bitwise reproducible and nondet_tol must be non-zero; every other",
    "# gradcheck test is left at its default strictness.",
    "value, locations, weights = make_inputs(torch.float64)",
    "value.requires_grad_(True)",
    "locations.requires_grad_(True)",
    "weights.requires_grad_(True)",
    "results['gradcheck'] = bool(gradcheck(",
    "    MSDeformAttnFunction.apply,",
    "    (value, shapes, starts, locations, weights, im2col_step),",
    "    eps=1e-6, atol=1e-4, nondet_tol=1e-5))",
    "assert results['gradcheck'], 'gradcheck failed for the compiled CUDA kernel'",
    "",
    "torch.cuda.synchronize()",
    "print(json.dumps(results))",
)
operator = json_probe(OPERATOR_PROBE, cwd=PROB_PATH, timeout=1800)

# --- the nn.Module PROB actually instantiates -------------------------------
MODULE_PROBE = torch_snippet(
    "import json",
    "from models.ops.functions.ms_deform_attn_func import MSDA_AVAILABLE",
    "from models.ops.modules.ms_deform_attn import MSDeformAttn",
    "",
    "assert MSDA_AVAILABLE, 'MSDeformAttn would route through the slow kernel.'",
    "device = torch.device('cuda')",
    "torch.manual_seed(0)",
    "module = MSDeformAttn(d_model=32, n_levels=2, n_heads=8, n_points=4).to(device)",
    "shapes = torch.as_tensor([(6, 4), (3, 2)], dtype=torch.long, device=device)",
    "starts = torch.cat((shapes.new_zeros((1,)), shapes.prod(1).cumsum(0)[:-1]))",
    "length = int(shapes.prod(1).sum())",
    "query = torch.randn(2, 5, 32, device=device, requires_grad=True)",
    "reference_points = torch.rand(2, 5, 2, 2, device=device)",
    "flat = torch.randn(2, length, 32, device=device, requires_grad=True)",
    "output = module(query, reference_points, flat, shapes, starts)",
    "assert output.shape == (2, 5, 32), output.shape",
    "assert output.is_cuda and torch.isfinite(output).all()",
    "output.square().mean().backward()",
    "for label, tensor in (('query', query), ('flat', flat)):",
    "    assert tensor.grad is not None, f'no gradient for {label}'",
    "    assert torch.isfinite(tensor.grad).all(), f'non-finite {label} gradient'",
    "parameter_gradients = [p.grad for p in module.parameters() if p.grad is not None]",
    "assert parameter_gradients, 'no module parameter received a gradient'",
    "assert all(torch.isfinite(g).all() for g in parameter_gradients)",
    "torch.cuda.synchronize()",
    "print(json.dumps({",
    "    'output_shape': list(output.shape),",
    "    'output_absmax': float(output.abs().max()),",
    "    'query_gradient_absmax': float(query.grad.abs().max()),",
    "    'flat_gradient_absmax': float(flat.grad.abs().max()),",
    "    'parameters_with_gradients': len(parameter_gradients),",
    "    'im2col_step': module.im2col_step,",
    "}))",
)
attention_module = json_probe(MODULE_PROBE, cwd=PROB_PATH, timeout=1800)

attention_backend = "compiled CUDA extension"
EXTENSION_VALIDATION = {
    "module": extension_module,
    "operator": operator,
    "attention_module": attention_module,
    "backend": attention_backend,
}

table("EXTENSION EXECUTION", [
    ("backend", attention_backend),
    ("extension file", extension_module["file"]),
    ("ms_deform_attn_forward", extension_module["has_forward"]),
    ("ms_deform_attn_backward", extension_module["has_backward"]),
    ("MSDA_AVAILABLE inside PROB", operator["msda_available"]),
    ("float64 vs reference", f"max abs diff {operator['float64']['max_abs_difference']:.3e}"),
    ("float32 vs reference", f"max abs diff {operator['float32']['max_abs_difference']:.3e}"),
    ("float64 gradients", operator["float64"]["gradient_absmax"]),
    ("float32 gradients", operator["float32"]["gradient_absmax"]),
    ("numerical gradcheck", operator["gradcheck"]),
    ("MSDeformAttn output", attention_module["output_shape"]),
    ("MSDeformAttn grads finite", True),
    ("module params with grads", attention_module["parameters_with_gradients"]),
])
mark("extension execution")

In [ ]:
# ============================================================================
# L. PROB BRIDGE CLI AND SUBPROCESS ENVIRONMENT
#
# The bridge is the only thing that talks to official PROB. Here its CLI is
# exercised, the flags this notebook will pass are confirmed to exist, the PROB
# runtime is imported, and the environment a bridge subprocess actually observes
# is inspected — including that the compiled attention extension is reachable
# from inside it, since ProbAdapter launches the bridge with this process's
# environment.
# ============================================================================
require_stages("extension execution")

bridge_path = Path(PROB_PATH, "daowod_prob_bridge.py")
if not bridge_path.is_file():
    raise FileNotFoundError(
        f"{bridge_path} is missing. The DAOWOD bridge lives on the "
        f"'{PROB_BRANCH}' branch of {PROB_REPOSITORY_URL}."
    )

# --- np.bool compatibility --------------------------------------------------
# datasets/open_world_eval.py calls .astype(np.bool), which NumPy 1.24 removed
# and NumPy 2.0 reinstated. The bridge restores the alias inside `evaluate`, but
# PROB's training loop also evaluates, so it is restored for every subprocess via
# sitecustomize whenever the running NumPy lacks it.
if hasattr(np, "bool"):
    print(f"numpy {np.__version__} provides np.bool; no compatibility shim needed.")
else:
    compat_dir = Path(COMPAT_DIR)
    compat_dir.mkdir(parents=True, exist_ok=True)
    (compat_dir / "sitecustomize.py").write_text(
        "# Restores the np.bool alias removed in NumPy 1.24 and reinstated in 2.0.\n"
        "import numpy\n"
        "if not hasattr(numpy, 'bool'):\n"
        "    numpy.bool = numpy.bool_\n",
        encoding="utf-8",
    )
    existing = os.environ.get("PYTHONPATH", "")
    os.environ["PYTHONPATH"] = os.pathsep.join(dict.fromkeys(
        [str(compat_dir), *[part for part in existing.split(os.pathsep) if part]]
    ))
    print(f"np.bool shim installed; PYTHONPATH = {os.environ['PYTHONPATH']}")

# --- what a bridge subprocess actually sees ---------------------------------
ENVIRONMENT_PROBE = torch_snippet(
    "import json, os",
    "import numpy",
    "import MultiScaleDeformableAttention as msda",
    "from models.ops.functions.ms_deform_attn_func import MSDA_AVAILABLE",
    "print(json.dumps({",
    "    'executable': __import__('sys').executable,",
    "    'pythonpath': os.environ.get('PYTHONPATH', ''),",
    "    'ld_library_path': os.environ.get('LD_LIBRARY_PATH', ''),",
    "    'numpy': numpy.__version__,",
    "    'np_bool': hasattr(numpy, 'bool'),",
    "    'wandb_mode': os.environ.get('WANDB_MODE', ''),",
    "    'extension': msda.__file__,",
    "    'msda_available': bool(MSDA_AVAILABLE),",
    "    'cuda_available': torch.cuda.is_available(),",
    "}))",
)
child_environment = json_probe(ENVIRONMENT_PROBE, cwd=PROB_PATH, timeout=1200)

if Path(child_environment["executable"]).resolve() != Path(sys.executable).resolve():
    raise RuntimeError(
        f"A bridge-like subprocess runs {child_environment['executable']}, not "
        f"{sys.executable}."
    )
if child_environment["pythonpath"] != os.environ.get("PYTHONPATH", ""):
    raise RuntimeError(
        "PYTHONPATH does not reach subprocesses: this session has "
        f"{os.environ.get('PYTHONPATH', '')!r} but the child saw "
        f"{child_environment['pythonpath']!r}."
    )
if not child_environment["np_bool"]:
    raise RuntimeError(
        "datasets/open_world_eval.py calls .astype(np.bool), which the bridge "
        "subprocess cannot resolve. The compatibility shim did not take effect "
        f"(PYTHONPATH={child_environment['pythonpath']!r})."
    )
if child_environment["wandb_mode"] != "disabled":
    raise RuntimeError(
        "WANDB_MODE=disabled did not reach subprocesses; a PROB run could block "
        "on an interactive wandb login."
    )
if not child_environment["msda_available"]:
    raise RuntimeError(
        "Inside a PROB subprocess MSDA_AVAILABLE is False, so PROB would fall back "
        "to the slow pure-PyTorch attention kernel without reporting anything."
    )
if not child_environment["cuda_available"]:
    raise RuntimeError("A bridge-like subprocess cannot see the GPU.")
print(f"subprocess environment verified: {child_environment}")

# --- bridge CLI -------------------------------------------------------------
# `check` compiles models/prob_deformable_detr.py and asserts the
# 'pred_features': hs[-1] export the bridge depends on. Cell G2 already ran it
# on the clean checkout; it is re-run here in the fully installed environment,
# and a non-zero exit code stops the run.
check_result = run([sys.executable, str(bridge_path), "check"],
                   cwd=PROB_PATH, timeout=600, check=False)
if check_result.returncode != 0:
    report_failure_context([sys.executable, str(bridge_path), "check"],
                           PROB_PATH, check_result)
    raise RuntimeError(
        f"`daowod_prob_bridge.py check` returned {check_result.returncode}; exit "
        "code 0 is required."
    )
check_output = check_result.stdout
for _expected in (BRIDGE_CHECK_EXPECTED_LINE,
                  "Available commands: train, evaluate, predict"):
    if _expected not in check_output:
        raise RuntimeError(
            f"`daowod_prob_bridge.py check` did not print {_expected!r}:\n{check_output}"
        )

# Neither the bridge nor anything else above may have touched the checkout.
bridge_stage_worktree = run(["git", "-C", PROB_PATH, "status", "--porcelain"],
                            timeout=300).stdout.strip()
bridge_stage_tracked = sorted(
    line[3:] for line in bridge_stage_worktree.splitlines()
    if line.strip() and not line.startswith("??")
)
if bridge_stage_tracked:
    raise RuntimeError(
        f"The PROB checkout is no longer clean: {bridge_stage_tracked} differ from "
        f"the pinned commit {PROB_COMMIT}."
    )

# Every flag this notebook passes must exist in the corresponding subcommand.
BRIDGE_REQUIRED_FLAGS = {
    "train": [
        "--labelled-ids", "--previous-checkpoint", "--output-checkpoint",
        "--output-dir", "--test-set", "--epochs", "--learning-rate",
        "--objectness-loss-coefficient", "--eval-every", "--freeze-prob-model",
        "--data-root", "--dataset", "--prev-introduced-classes",
        "--current-introduced-classes", "--num-classes",
        "--objectness-temperature", "--batch-size", "--num-workers",
        "--device", "--seed",
    ],
    "predict": [
        "--image-ids", "--checkpoint", "--output", "--max-proposals-per-image",
        "--minimum-unknown-score", "--data-root", "--dataset",
        "--prev-introduced-classes", "--current-introduced-classes",
        "--num-classes", "--objectness-temperature", "--batch-size",
        "--num-workers", "--device", "--seed",
    ],
    "evaluate": [
        "--checkpoint", "--output", "--output-dir", "--test-set", "--data-root",
        "--dataset", "--prev-introduced-classes", "--current-introduced-classes",
        "--num-classes", "--objectness-temperature", "--batch-size",
        "--num-workers", "--device", "--seed",
    ],
}
bridge_help = {}
for subcommand, flags in BRIDGE_REQUIRED_FLAGS.items():
    help_text = run([sys.executable, str(bridge_path), subcommand, "--help"],
                    cwd=PROB_PATH, timeout=600).stdout
    missing_flags = [flag for flag in flags if flag not in help_text]
    if missing_flags:
        raise RuntimeError(
            f"`daowod_prob_bridge.py {subcommand} --help` does not offer "
            f"{missing_flags}. The bridge CLI changed and the commands this "
            "notebook builds would be rejected."
        )
    bridge_help[subcommand] = len(flags)

# --- the PROB runtime must import -------------------------------------------
prob_imports = json_probe(
    torch_snippet(
        "import json",
        "import engine, main_open_world, util.misc",
        "from datasets.coco import make_coco_transforms",
        "from datasets.open_world_eval import OWEvaluator",
        "from datasets.torchvision_datasets.open_world import (",
        "    OWDetection, VOC_COCO_CLASS_NAMES)",
        "from models import build_model",
        "print(json.dumps({",
        "    'main_open_world': main_open_world.__file__,",
        "    'engine': engine.__file__,",
        "    'build_model': build_model.__module__,",
        "    'evaluator': OWEvaluator.__module__,",
        "    'dataset': OWDetection.__module__,",
        f"    'class_count': len(VOC_COCO_CLASS_NAMES[{DATASET!r}]),",
        "    'test_transforms': str(make_coco_transforms('probe_test')),",
        "}))",
    ),
    cwd=PROB_PATH, timeout=1800,
)
if prob_imports["class_count"] != NUM_CLASSES:
    raise RuntimeError(
        f"PROB defines {prob_imports['class_count']} {DATASET} classes but "
        f"NUM_CLASSES={NUM_CLASSES}."
    )
if not Path(prob_imports["main_open_world"]).resolve().is_relative_to(Path(PROB_PATH).resolve()):
    raise RuntimeError(
        f"PROB was imported from {prob_imports['main_open_world']}, outside the "
        f"pinned clone {PROB_PATH}."
    )

table("BRIDGE", [
    ("bridge", bridge_path),
    ("check", f"decoder-feature export OK (exit {check_result.returncode}), 3 commands"),
    ("PROB checkout", "clean, unmodified"),
    *[(f"{name} flags verified", count) for name, count in bridge_help.items()],
    ("main_open_world", prob_imports["main_open_world"]),
    ("engine", prob_imports["engine"]),
    ("build_model module", prob_imports["build_model"]),
    ("evaluator module", prob_imports["evaluator"]),
    ("dataset module", prob_imports["dataset"]),
    (f"{DATASET} classes", prob_imports["class_count"]),
    ("numpy", f"{np.__version__} (np.bool: {child_environment['np_bool']})"),
    ("subprocess extension", child_environment["extension"]),
    ("subprocess MSDA_AVAILABLE", child_environment["msda_available"]),
    ("PYTHONPATH", os.environ.get("PYTHONPATH", "") or "<unset>"),
])
mark("bridge")

In [ ]:
# ============================================================================
# M1. DATASET AND TASK-1 CHECKPOINT DISCOVERY, PLUS CHECKPOINT STAGING
#
# Nothing is assumed to be at a fixed path: candidates under DRIVE_SEARCH_ROOTS
# are discovered, listed, and validated before use. No dataset file is ever
# invented — if an input is missing, the cell stops with one error that names
# every location searched and every pattern accepted.
#
# With DATASET_SOURCE="auto" an extracted dataset directory wins over an archive
# — but only a *complete* one. Annotations/ and JPEGImages/ merely existing is not
# enough: every directory candidate is measured against the TOWOD split IDs the
# protocol consumes, and a partial folder (a one-image smoke test, say) is
# reported and ignored rather than preferred over the archive that holds the real
# dataset. The run stops only on a genuine ambiguity: several complete
# directories, or no complete directory and several archives.
#
# The checkpoint is copied to /content because every detector invocation reads
# it; the copy is skipped when a byte-identical local copy already exists.
# ============================================================================
require_stages("bridge")

ARCHIVE_SUFFIXES = (".tar.zst", ".tar.zstd", ".tar.gz", ".tgz", ".tar")


def walk_drive(root, max_depth):
    root = Path(root)
    if not root.is_dir():
        return
    stack = [(root, 0)]
    while stack:
        directory, depth = stack.pop()
        try:
            entries = sorted(directory.iterdir())
        except OSError as error:
            print(f"  (skipping unreadable {directory}: {error})")
            continue
        for entry in entries:
            yield entry, depth
            if entry.is_dir() and depth < max_depth:
                stack.append((entry, depth + 1))


def is_archive(path):
    name = path.name.lower()
    return path.is_file() and any(name.endswith(suffix) for suffix in ARCHIVE_SUFFIXES)


def is_dataset_directory(path):
    return (
        path.is_dir()
        and (path / "Annotations").is_dir()
        and (path / "JPEGImages").is_dir()
    )


def classify_dataset_source(path):
    path = Path(path)
    if is_dataset_directory(path):
        return "directory"
    if is_archive(path):
        return "archive"
    raise FileNotFoundError(
        f"{path} is neither a directory containing Annotations/ and JPEGImages/ "
        f"nor an archive ending in {ARCHIVE_SUFFIXES}."
    )


# --- content-aware completeness -------------------------------------------
# Annotations/ and JPEGImages/ existing says nothing about what is inside them: a
# one-image smoke-test folder has exactly the same shape as the full M-OWODB
# release. A directory is therefore measured against the split IDs the protocol
# actually consumes before it is allowed to win over an archive.
#
# Cell M2 stages annotations as <id>.xml and images as <id>.jpg — those are the
# only extensions the staging helpers copy or extract, so they are the only ones
# that can count towards coverage. Files under other image extensions are counted
# separately so a .png-only or .jpeg-only directory is diagnosed precisely
# instead of being reported as empty.
ANNOTATION_SUFFIX = ".xml"
SUPPORTED_IMAGE_SUFFIXES = (".jpg",)
UNSUPPORTED_IMAGE_SUFFIXES = (".jpeg", ".png")

# The official evaluation split is the mandatory one: cell M2 stages the
# annotation of *every* image in it, and cell N samples the balanced pilot test
# set from it. The two protocol source splits are checked as well, because cell N
# samples SOURCE_TASK2_IMAGES and REFERENCE_IMAGES from their full ID lists and
# cell N2 then stages whichever IDs the seeded draw returns — any missing file
# would only surface hours later, mid-campaign.
#
# No subset is intentionally staged at this level, so the threshold is total
# coverage: a directory qualifies only at 100% annotations and 100% images.
PRIMARY_COMPLETENESS_SPLIT = OFFICIAL_EVAL_SPLIT
DIRECTORY_COMPLETENESS_SPLITS = (OFFICIAL_EVAL_SPLIT, TASK2_SOURCE_SPLIT,
                                 TASK1_REFERENCE_SPLIT)
REQUIRED_COVERAGE = 1.0


def resolve_split_file(split, directory=None):
    """Locate a TOWOD split list.

    The pinned PROB checkout is authoritative: it ships data/OWOD/ImageSets and
    cell M2 rebuilds the local ImageSets tree from it, so coverage is measured
    against exactly the IDs the protocol will later read. DAOWOD ships no
    ImageSets of its own; a dataset directory that carries its own copy is
    accepted as a fallback for candidates discovered outside the checkout.
    """
    searched = [Path(PROB_PATH, "data", "OWOD", "ImageSets", DATASET, f"{split}.txt")]
    if directory is not None:
        searched.append(Path(directory, "ImageSets", DATASET, f"{split}.txt"))
        searched.append(Path(directory, "ImageSets", f"{split}.txt"))
    for candidate in searched:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"The {split!r} split list was not found. Searched:\n"
        + "\n".join(f"  {path}" for path in searched)
    )


# read_ids() de-duplicates while preserving order, so these are the unique IDs.
REQUIRED_SPLIT_IDS = {
    split: read_ids(resolve_split_file(split))
    for split in DIRECTORY_COMPLETENESS_SPLITS
}


def directory_stems(directory, suffixes):
    """Map each suffix to the set of file stems carrying it, from one listing.

    One `iterdir()` per directory rather than a stat per required ID: over the
    Drive FUSE mount the difference is a single listing against tens of thousands
    of round trips.
    """
    stems = {suffix: set() for suffix in suffixes}
    for entry in directory.iterdir():
        lowered = entry.name.lower()
        for suffix in suffixes:
            if lowered.endswith(suffix):
                stems[suffix].add(entry.name[: -len(suffix)])
                break
    return stems


def evaluate_dataset_directory(path):
    """Measure a structurally valid directory against the required split IDs."""
    path = Path(path)
    report = {
        "path": str(path),
        "readable": True,
        "annotation_files": 0,
        "image_files": 0,
        "unsupported_image_files": {},
        "required_ids": len(REQUIRED_SPLIT_IDS[PRIMARY_COMPLETENESS_SPLIT]),
        "primary_split": PRIMARY_COMPLETENESS_SPLIT,
        "required_coverage": REQUIRED_COVERAGE,
        "annotation_coverage": 0.0,
        "image_coverage": 0.0,
        "splits": {},
        "reasons": [],
    }
    try:
        annotation_stems = directory_stems(
            path / "Annotations", (ANNOTATION_SUFFIX,))[ANNOTATION_SUFFIX]
        image_stems = directory_stems(
            path / "JPEGImages",
            (*SUPPORTED_IMAGE_SUFFIXES, *UNSUPPORTED_IMAGE_SUFFIXES))
    except OSError as error:
        report["readable"] = False
        report["status"] = "incomplete"
        report["complete"] = False
        report["reasons"].append(f"the directory could not be listed ({error})")
        return report

    supported_images = set()
    for suffix in SUPPORTED_IMAGE_SUFFIXES:
        supported_images |= image_stems[suffix]
    report["annotation_files"] = len(annotation_stems)
    report["image_files"] = len(supported_images)
    report["unsupported_image_files"] = {
        suffix: len(image_stems[suffix])
        for suffix in UNSUPPORTED_IMAGE_SUFFIXES if image_stems[suffix]
    }

    for split, ids in REQUIRED_SPLIT_IDS.items():
        required = set(ids)
        present_annotations = required & annotation_stems
        present_images = required & supported_images
        report["splits"][split] = {
            "required": len(required),
            "annotations_present": len(present_annotations),
            "images_present": len(present_images),
            "annotations_missing": len(required) - len(present_annotations),
            "images_missing": len(required) - len(present_images),
            "annotation_coverage": len(present_annotations) / len(required),
            "image_coverage": len(present_images) / len(required),
            "missing_annotation_examples": sorted(required - present_annotations)[:5],
            "missing_image_examples": sorted(required - present_images)[:5],
        }

    report["annotation_coverage"] = min(
        split["annotation_coverage"] for split in report["splits"].values())
    report["image_coverage"] = min(
        split["image_coverage"] for split in report["splits"].values())

    for split, measurements in report["splits"].items():
        if measurements["annotation_coverage"] < REQUIRED_COVERAGE:
            report["reasons"].append(
                f"{split}: {measurements['annotations_present']}/"
                f"{measurements['required']} annotations "
                f"({measurements['annotation_coverage']:.2%}), missing e.g. "
                f"{measurements['missing_annotation_examples']}"
            )
        if measurements["image_coverage"] < REQUIRED_COVERAGE:
            report["reasons"].append(
                f"{split}: {measurements['images_present']}/"
                f"{measurements['required']} images "
                f"({measurements['image_coverage']:.2%}), missing e.g. "
                f"{measurements['missing_image_examples']}"
            )
    if report["unsupported_image_files"] and not report["image_files"]:
        report["reasons"].append(
            "JPEGImages/ holds only "
            f"{report['unsupported_image_files']}; the staging helpers in cell M2 "
            f"copy and extract {SUPPORTED_IMAGE_SUFFIXES} only"
        )
    report["complete"] = not report["reasons"]
    report["status"] = "complete" if report["complete"] else "incomplete"
    return report


def print_directory_report(report):
    """Requirement of the candidate listing: counts, coverage and verdict."""
    print(f"  directory  {report['path']}")
    print(f"      annotation files  : {report['annotation_files']}")
    print(f"      image files       : {report['image_files']} "
          f"({'/'.join(SUPPORTED_IMAGE_SUFFIXES)})")
    if report["unsupported_image_files"]:
        print(f"      other image files : {report['unsupported_image_files']} "
              "(not usable downstream)")
    print(f"      required IDs      : {report['required_ids']} "
          f"({report['primary_split']})")
    print(f"      annotation cover  : {report['annotation_coverage']:.2%}")
    print(f"      image coverage    : {report['image_coverage']:.2%}")
    print(f"      status            : {report['status'].upper()}")
    for split, measurements in report["splits"].items():
        print(f"        {split:<24} "
              f"annotations {measurements['annotations_present']}/{measurements['required']} "
              f"({measurements['annotation_coverage']:.2%})  "
              f"images {measurements['images_present']}/{measurements['required']} "
              f"({measurements['image_coverage']:.2%})")
    for reason in report["reasons"]:
        print(f"        incomplete: {reason}")


banner("SEARCHING DRIVE")
dataset_candidates, checkpoint_candidates = [], []
searched_roots = []
for search_root in DRIVE_SEARCH_ROOTS:
    if dataset_candidates and checkpoint_candidates:
        print(f"root: {search_root} (skipped, candidates already found)")
        continue
    print(f"root: {search_root} (depth <= {DRIVE_SEARCH_MAX_DEPTH})")
    searched_roots.append(search_root)
    for entry, _ in walk_drive(search_root, DRIVE_SEARCH_MAX_DEPTH):
        if is_archive(entry) or is_dataset_directory(entry):
            dataset_candidates.append(entry)
        elif entry.is_file() and entry.suffix == ".pth":
            checkpoint_candidates.append(entry)

dataset_candidates = sorted(set(dataset_candidates))
checkpoint_candidates = sorted(set(checkpoint_candidates))

candidate_directories = [c for c in dataset_candidates if is_dataset_directory(c)]
candidate_archives = [c for c in dataset_candidates if is_archive(c)]

banner("DATASET CANDIDATES")
print(f"completeness is measured against {list(DIRECTORY_COMPLETENESS_SPLITS)} "
      f"at {REQUIRED_COVERAGE:.0%} coverage")
directory_reports = [evaluate_dataset_directory(path) for path in candidate_directories]
for report in directory_reports:
    print_directory_report(report)
for archive in candidate_archives:
    print(f"  archive    {archive}  "
          f"({archive.stat().st_size / 1024 ** 3:.2f} GB)")
if not dataset_candidates:
    print("  (none)")

complete_reports = [report for report in directory_reports if report["complete"]]
incomplete_reports = [report for report in directory_reports if not report["complete"]]
complete_directories = [Path(report["path"]) for report in complete_reports]
print(f"  -> {len(complete_reports)} complete directory/directories, "
      f"{len(incomplete_reports)} incomplete, {len(candidate_archives)} archive(s)")

if DATASET_SOURCE == "auto":
    # A *complete* directory is preferred over an archive: it needs no
    # extraction. An incomplete one — a smoke-test folder with a handful of
    # files, say — is reported and then ignored, and never blocks the archive
    # that does hold the full dataset. Only a genuine ambiguity stops the run:
    # several complete directories, or no complete directory and several
    # archives.
    if not dataset_candidates:
        raise FileNotFoundError(
            "No M-OWODB dataset source was found.\n"
            f"  searched roots  : {searched_roots}\n"
            f"  maximum depth   : {DRIVE_SEARCH_MAX_DEPTH}\n"
            "  accepted forms  : a directory containing both Annotations/ and "
            "JPEGImages/, or a file ending in "
            f"{ARCHIVE_SUFFIXES} that contains them\n"
            "Upload the data to Drive, or set DATASET_SOURCE to its absolute path "
            "in the configuration cell. This notebook will not fabricate a dataset."
        )
    if incomplete_reports:
        print()
        for report in incomplete_reports:
            print(f"ignoring the incomplete dataset directory {report['path']} "
                  f"(annotations {report['annotation_coverage']:.2%}, images "
                  f"{report['image_coverage']:.2%} of the required split IDs)")
    if len(complete_reports) > 1:
        raise RuntimeError(
            "Several complete dataset directories were found, so the choice is "
            "ambiguous. Set DATASET_SOURCE to exactly one of: "
            f"{[report['path'] for report in complete_reports]}"
        )
    if complete_reports:
        dataset_source = complete_directories[0]
        dataset_selection = "auto: the one complete dataset directory"
        if candidate_archives:
            dataset_selection += (
                f", preferred over {len(candidate_archives)} archive(s)")
            print(f"selected the complete dataset directory {dataset_source} in "
                  f"preference to {[str(a) for a in candidate_archives]}; no "
                  "extraction is needed.")
    elif len(candidate_archives) > 1:
        raise RuntimeError(
            "No complete dataset directory was found and several archives are "
            "present, so the choice is ambiguous. Set DATASET_SOURCE to exactly "
            f"one of: {[str(archive) for archive in candidate_archives]}"
        )
    elif candidate_archives:
        dataset_source = candidate_archives[0]
        dataset_selection = "auto: the one archive"
        if incomplete_reports:
            dataset_selection += (
                f" ({len(incomplete_reports)} incomplete directory/directories "
                "ignored)")
            print(f"selected the archive {dataset_source}: no discovered directory "
                  "holds the full dataset.")
    else:
        # Directories exist but none of them is usable, and there is no archive.
        raise RuntimeError(
            "No usable M-OWODB dataset source was found. "
            f"{len(incomplete_reports)} directory/directories have the right shape "
            "but not the required content:\n"
            + "\n".join(
                f"  {report['path']}\n"
                f"    annotations {report['annotation_files']}, images "
                f"{report['image_files']}, required "
                f"{report['required_ids']} ({report['primary_split']})\n"
                + "\n".join(f"    - {reason}" for reason in report["reasons"])
                for report in incomplete_reports
            )
            + f"\nFull coverage of {list(DIRECTORY_COMPLETENESS_SPLITS)} is "
            "required. Upload the complete dataset (a directory or an archive), "
            "or set DATASET_SOURCE to it. This notebook will not run the protocol "
            "on partial data."
        )
else:
    dataset_source = Path(DATASET_SOURCE)
    if not dataset_source.exists():
        raise FileNotFoundError(f"DATASET_SOURCE does not exist: {dataset_source}")
    dataset_selection = "explicit DATASET_SOURCE"
dataset_mode = classify_dataset_source(dataset_source)

# An explicitly pinned directory is not exempt from the content check: it is
# validated here, before anything is staged, and a partial one fails immediately
# with the counts that prove it.
selected_directory_report = None
if dataset_mode == "directory":
    selected_directory_report = next(
        (report for report in directory_reports
         if report["path"] == str(dataset_source)), None)
    if selected_directory_report is None:
        banner("EXPLICIT DATASET DIRECTORY")
        selected_directory_report = evaluate_dataset_directory(dataset_source)
        directory_reports.append(selected_directory_report)
        print_directory_report(selected_directory_report)
    if not selected_directory_report["complete"]:
        raise RuntimeError(
            f"{dataset_source} is not a complete {DATASET} dataset directory:\n"
            f"  annotation files : {selected_directory_report['annotation_files']}\n"
            f"  image files      : {selected_directory_report['image_files']}\n"
            f"  required IDs     : {selected_directory_report['required_ids']} "
            f"({selected_directory_report['primary_split']})\n"
            f"  annotation cover : "
            f"{selected_directory_report['annotation_coverage']:.2%}\n"
            f"  image coverage   : "
            f"{selected_directory_report['image_coverage']:.2%}\n"
            + "\n".join(f"  - {reason}"
                        for reason in selected_directory_report["reasons"])
            + f"\nFull coverage of {list(DIRECTORY_COMPLETENESS_SPLITS)} is "
            "required. Point DATASET_SOURCE at the complete dataset directory or "
            "at the archive that contains it; this notebook will not run the "
            "protocol on partial data."
        )

banner("CHECKPOINT CANDIDATES")
for candidate in checkpoint_candidates:
    print(f"  {candidate}  ({candidate.stat().st_size / 1024 ** 2:.0f} MB)")
if not checkpoint_candidates:
    print("  (none)")

if TASK1_CHECKPOINT == "auto":
    preferred = [c for c in checkpoint_candidates if c.name == "t1.pth"]
    if not preferred:
        preferred = [c for c in checkpoint_candidates if "t1" in c.name.lower()]
    if not preferred:
        preferred = checkpoint_candidates
    if not preferred:
        raise FileNotFoundError(
            "No PROB Task-1 checkpoint was found.\n"
            f"  searched roots : {searched_roots}\n"
            f"  maximum depth  : {DRIVE_SEARCH_MAX_DEPTH}\n"
            "  accepted forms : any *.pth holding a PROB training checkpoint "
            "(a dict with 'model' and an integer 'epoch'), preferring a file named "
            "t1.pth or containing 't1'\n"
            "Upload the Task-1 checkpoint to Drive, or set TASK1_CHECKPOINT to its "
            "absolute path in the configuration cell."
        )
    if len(preferred) > 1:
        raise RuntimeError(
            "Several Task-1 checkpoint candidates were found; set TASK1_CHECKPOINT "
            f"explicitly. Candidates: {[str(path) for path in preferred]}"
        )
    task1_source = preferred[0]
else:
    task1_source = Path(TASK1_CHECKPOINT)
    if not task1_source.is_file():
        raise FileNotFoundError(f"TASK1_CHECKPOINT does not exist: {task1_source}")

# --- stage the checkpoint locally -------------------------------------------
Path(LOCAL_CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
task1_checkpoint = Path(LOCAL_CHECKPOINT_DIR) / "task1.pth"
source_bytes = task1_source.stat().st_size
require_free_space(CONTENT_ROOT, MINIMUM_LOCAL_FREE_GB, "Staging the Task-1 checkpoint")
if task1_checkpoint.exists() and task1_checkpoint.stat().st_size == source_bytes:
    print(f"{task1_checkpoint} already staged ({source_bytes:,} bytes); not copied again.")
else:
    print(f"copying {task1_source} -> {task1_checkpoint} ({source_bytes:,} bytes)")
    staging_checkpoint = task1_checkpoint.with_name(f".{task1_checkpoint.name}.partial")
    shutil.copy2(task1_source, staging_checkpoint)
    staging_checkpoint.replace(task1_checkpoint)
if task1_checkpoint.stat().st_size != source_bytes:
    raise RuntimeError(
        f"{task1_checkpoint} is {task1_checkpoint.stat().st_size} bytes but the "
        f"Drive source is {source_bytes}."
    )

# --- the checkpoint must be a complete PROB training checkpoint --------------
state = torch.load(task1_checkpoint, map_location="cpu", weights_only=False)
if not isinstance(state, dict) or "model" not in state:
    raise RuntimeError(
        f"{task1_source} is not a PROB training checkpoint: expected a dict with a "
        f"'model' entry, found {type(state).__name__} "
        f"{sorted(state) if isinstance(state, dict) else ''}."
    )
if not isinstance(state.get("epoch"), int):
    raise RuntimeError(
        f"{task1_source} has no integer 'epoch'. daowod_prob_bridge.py derives the "
        "fine-tuning epoch range from it, and this notebook will not invent a "
        "value. Use a complete PROB training checkpoint (e.g. checkpoint0040.pth)."
    )
checkpoint_args = state.get("args")
task1_epoch = int(state["epoch"])
task1_tensor_count = len(state["model"])
task1_parameter_count = sum(
    int(value.numel()) for value in state["model"].values() if torch.is_tensor(value)
)
task1_has_optimizer = "optimizer" in state
del state
cleanup_runtime()
task1_checkpoint_sha256 = sha256_of_file(task1_checkpoint)

Path(LOCAL_RESULT_ROOT).mkdir(parents=True, exist_ok=True)
Path(DRIVE_RESULT_DIR).mkdir(parents=True, exist_ok=True)

DATASET_DISCOVERY = {
    "searched_roots": searched_roots,
    "max_depth": DRIVE_SEARCH_MAX_DEPTH,
    "completeness_splits": list(DIRECTORY_COMPLETENESS_SPLITS),
    "primary_split": PRIMARY_COMPLETENESS_SPLIT,
    "required_coverage": REQUIRED_COVERAGE,
    "annotation_suffix": ANNOTATION_SUFFIX,
    "supported_image_suffixes": list(SUPPORTED_IMAGE_SUFFIXES),
    "directory_candidates": directory_reports,
    "complete_directories": [report["path"] for report in directory_reports
                             if report["complete"]],
    "incomplete_directories": [report["path"] for report in directory_reports
                               if not report["complete"]],
    "archive_candidates": [
        {"path": str(archive), "bytes": archive.stat().st_size}
        for archive in candidate_archives
    ],
    "selection": dataset_selection,
    "selected": str(dataset_source),
    "mode": dataset_mode,
    "selected_directory_complete": (
        None if selected_directory_report is None
        else selected_directory_report["complete"]),
}

table("DRIVE ASSETS", [
    ("searched roots", searched_roots),
    ("dataset source", dataset_source),
    ("dataset mode", dataset_mode),
    ("dataset selected by", dataset_selection),
    ("directory candidates",
     f"{len(DATASET_DISCOVERY['complete_directories'])} complete, "
     f"{len(DATASET_DISCOVERY['incomplete_directories'])} incomplete"),
    ("archive candidates", len(candidate_archives)),
    ("required split IDs",
     f"{len(REQUIRED_SPLIT_IDS[PRIMARY_COMPLETENESS_SPLIT])} "
     f"({PRIMARY_COMPLETENESS_SPLIT})"),
    ("selected annotation coverage",
     "n/a (archive)" if selected_directory_report is None
     else f"{selected_directory_report['annotation_coverage']:.2%}"),
    ("selected image coverage",
     "n/a (archive)" if selected_directory_report is None
     else f"{selected_directory_report['image_coverage']:.2%}"),
    ("task-1 checkpoint (Drive)", task1_source),
    ("task-1 checkpoint (local)", task1_checkpoint),
    ("task-1 bytes", f"{source_bytes:,}"),
    ("task-1 sha256", task1_checkpoint_sha256),
    ("task-1 epoch", task1_epoch),
    ("task-1 model tensors", task1_tensor_count),
    ("task-1 parameters", f"{task1_parameter_count:,}"),
    ("task-1 carries optimizer", task1_has_optimizer),
    ("task-1 carries args", checkpoint_args is not None),
    ("local result root", LOCAL_RESULT_ROOT),
    ("drive result dir", DRIVE_RESULT_DIR),
    ("free /content GB", free_gb(CONTENT_ROOT)),
    ("free Drive GB", free_gb(DRIVE_RESULT_PARENT)),
])
mark("drive assets")

In [ ]:
# ============================================================================
# M2. ASSEMBLE THE LOCAL DATASET ROOT
#
# Layout follows PROB/README.md: DATA_ROOT/{ImageSets,Annotations,JPEGImages}.
# Splits come from the pinned PROB checkout; annotations and images come from the
# verified Drive source and are *copied* to /content, never read from Drive at
# training time. PROB's OWDetection parses an annotation for every image it
# loads — on every predict, train and evaluate call — so leaving those files on a
# FUSE mount would dominate the wall clock of all twelve rounds.
#
#   archive source  : the whole Annotations tree is extracted in one sequential
#                     tar pass here; images are extracted by ID in cell N2
#   directory source: annotations and images are copied per ID, on demand, by the
#                     stage_annotations / stage_images helpers below
#
# Idempotency: ImageSets is always rebuilt from the checkout, which also clears
# any leftover `daowod_*` temporary split the bridge may have abandoned. Staged
# annotations and images are kept when a stamp file proves they came from this
# exact source, so a re-run copies nothing twice.
# ============================================================================
require_stages("drive assets")

data_root = Path(DATA_ROOT)
image_set_root = data_root / "ImageSets" / DATASET
annotations_dir = data_root / "Annotations"
jpeg_dir = data_root / "JPEGImages"
dataset_stamp_path = data_root / "dataset_source_stamp.json"

repository_image_sets = Path(PROB_PATH, "data", "OWOD", "ImageSets")
if not (repository_image_sets / DATASET).is_dir():
    raise FileNotFoundError(
        f"PROB does not ship the {DATASET} splits at {repository_image_sets / DATASET}."
    )

source_stat = dataset_source.stat()
dataset_stamp = {
    "source": str(dataset_source),
    "mode": dataset_mode,
    "size": source_stat.st_size,
    "mtime_ns": source_stat.st_mtime_ns,
    "dataset": DATASET,
    "layout": 2,
}
reuse_staged_data = (
    dataset_stamp_path.is_file()
    and read_json(dataset_stamp_path) == dataset_stamp
    and annotations_dir.is_dir()
    and not annotations_dir.is_symlink()
    and jpeg_dir.is_dir()
    and not jpeg_dir.is_symlink()
)

data_root.mkdir(parents=True, exist_ok=True)

# ImageSets is small and always rebuilt, so generated splits never accumulate.
if (data_root / "ImageSets").exists():
    shutil.rmtree(data_root / "ImageSets")
shutil.copytree(repository_image_sets, data_root / "ImageSets")
for split in (TASK2_SOURCE_SPLIT, TASK1_REFERENCE_SPLIT, OFFICIAL_EVAL_SPLIT):
    if not (image_set_root / f"{split}.txt").is_file():
        raise FileNotFoundError(f"Missing split file: {image_set_root / f'{split}.txt'}")


def archive_members(archive, limit=400):
    """List the first members of an archive to discover its internal layout."""
    name = archive.name.lower()
    if name.endswith((".tar.zst", ".tar.zstd")):
        script = 'zstd -dc "$1" | tar -tf - | head -n "$2"'
    elif name.endswith((".tar.gz", ".tgz")):
        script = 'tar -tzf "$1" | head -n "$2"'
    else:
        script = 'tar -tf "$1" | head -n "$2"'
    # `head` closes the pipe, so the producer may exit non-zero; the return code
    # and stderr are printed either way and empty output is a hard failure.
    result = run(["bash", "-lc", script, "bash", str(archive), str(limit)],
                 timeout=3600, check=False)
    members = [line for line in result.stdout.splitlines() if line.strip()]
    if not members:
        report_failure_context(["bash", "-lc", script, str(archive)], None, result)
        raise RuntimeError(f"Could not list the contents of {archive}.")
    return members


def archive_prefix(members):
    """Path components preceding Annotations/ JPEGImages/ ImageSets/ in the archive."""
    for member in members:
        parts = [part for part in member.split("/") if part]
        for index, part in enumerate(parts):
            if part in ("Annotations", "JPEGImages", "ImageSets"):
                return "/".join(parts[:index])
    raise RuntimeError(
        "No Annotations/, JPEGImages/ or ImageSets/ directory found in the first "
        f"{len(members)} archive members: {members[:10]}"
    )


def extract_from_archive(archive, member, strip_components, destination,
                         *, member_file=None, timeout):
    name = archive.name.lower()
    decompress = "zstd -dc" if name.endswith((".tar.zst", ".tar.zstd")) else None
    tar_flags = "-xz" if name.endswith((".tar.gz", ".tgz")) else "-x"
    selector = '-T "$5"' if member_file else '"$5"'
    if decompress:
        script = (f'{decompress} "$1" | tar -x --strip-components="$2" '
                  f'-C "$3" {selector}')
    else:
        script = (f'tar {tar_flags} -f "$1" --strip-components="$2" '
                  f'-C "$3" {selector}')
    run(["bash", "-lc", script, "bash", str(archive), str(strip_components),
         str(destination), "unused", str(member_file or member)], timeout=timeout)


def stage_from_source_directory(image_ids, *, subdirectory, suffix, description):
    """Copy the missing members of one kind from the Drive directory to /content."""
    target_directory = data_root / subdirectory
    target_directory.mkdir(parents=True, exist_ok=True)
    missing = [image_id for image_id in image_ids
               if not (target_directory / f"{image_id}{suffix}").is_file()]
    if not missing:
        print(f"{description}: all {len(image_ids)} files already staged locally")
        return 0
    source_directory = Path(dataset_source, subdirectory)
    if not source_directory.is_dir():
        raise FileNotFoundError(
            f"{source_directory} does not exist, so {description} cannot be staged."
        )
    print(f"{description}: copying {len(missing)} of {len(image_ids)} files from "
          f"{source_directory}")
    require_free_space(CONTENT_ROOT, MINIMUM_LOCAL_FREE_GB, f"Staging {description}")
    unavailable = []
    for position, image_id in enumerate(missing, start=1):
        source = source_directory / f"{image_id}{suffix}"
        if not source.is_file():
            unavailable.append(str(source))
            continue
        target = target_directory / f"{image_id}{suffix}"
        temporary = target.with_name(f".{target.name}.partial")
        shutil.copy2(source, temporary)
        temporary.replace(target)
        if position % 2000 == 0:
            print(f"  {position}/{len(missing)}")
    if unavailable:
        raise FileNotFoundError(
            f"{len(unavailable)} files are absent from the dataset source; "
            f"first 20:\n" + "\n".join(f"  {path}" for path in unavailable[:20])
        )
    print(f"{description}: staged {len(missing)} files")
    return len(missing)


def stage_annotations(image_ids, *, description):
    """Make the annotations of these images available locally."""
    if dataset_mode == "archive":
        # The whole Annotations tree was extracted below in one tar pass.
        missing = [image_id for image_id in image_ids
                   if not (annotations_dir / f"{image_id}.xml").is_file()]
        if missing:
            raise FileNotFoundError(
                f"{len(missing)} annotations are absent from the archive extraction "
                f"({description}); first 20: {missing[:20]}"
            )
        return 0
    return stage_from_source_directory(
        image_ids, subdirectory="Annotations", suffix=".xml", description=description)


def stage_images(image_ids, *, description):
    """Make the JPEGs of these images available locally."""
    if dataset_mode == "archive":
        missing = [image_id for image_id in image_ids
                   if not (jpeg_dir / f"{image_id}.jpg").is_file()]
        if not missing:
            print(f"{description}: all {len(image_ids)} images already extracted")
            return 0
        prefix = archive_layout["prefix"]
        member_list = Path(CONTENT_ROOT, "daowod_required_jpegs.txt")
        member_list.write_text(
            "\n".join(
                f"{prefix}/JPEGImages/{image_id}.jpg" if prefix
                else f"JPEGImages/{image_id}.jpg"
                for image_id in missing
            ) + "\n",
            encoding="utf-8",
        )
        print(f"{description}: extracting {len(missing)} of {len(image_ids)} images")
        require_free_space(CONTENT_ROOT, MINIMUM_LOCAL_FREE_GB, f"Staging {description}")
        extract_from_archive(dataset_source, None, archive_layout["strip_components"],
                             data_root, member_file=member_list, timeout=7200)
        return len(missing)
    return stage_from_source_directory(
        image_ids, subdirectory="JPEGImages", suffix=".jpg", description=description)


# --- prepare the local annotation and image directories ----------------------
archive_layout = None
if not reuse_staged_data:
    for stale in (annotations_dir, jpeg_dir, dataset_stamp_path):
        if stale.is_symlink() or stale.is_file():
            stale.unlink()
        elif stale.is_dir():
            shutil.rmtree(stale)
    annotations_dir.mkdir(parents=True)
    jpeg_dir.mkdir(parents=True)

if dataset_mode == "archive":
    members = archive_members(dataset_source)
    prefix = archive_prefix(members)
    strip = len([part for part in prefix.split("/") if part])
    archive_layout = {"prefix": prefix, "strip_components": strip}
    banner("ARCHIVE LAYOUT")
    print(f"archive          : {dataset_source}")
    print(f"detected prefix  : {prefix or '(none)'}")
    print(f"strip components : {strip}")
    print("first members    :")
    for member in members[:8]:
        print(f"  {member}")
    staged_annotations = sum(1 for _ in annotations_dir.glob("*.xml"))
    if reuse_staged_data and staged_annotations:
        print(f"\n{staged_annotations} annotations already extracted from this exact "
              "archive; not extracting again.")
    else:
        annotations_member = f"{prefix}/Annotations" if prefix else "Annotations"
        require_free_space(CONTENT_ROOT, MINIMUM_LOCAL_FREE_GB,
                          "Extracting the annotations")
        extract_from_archive(dataset_source, annotations_member, strip, data_root,
                             timeout=7200)
else:
    # Verify the source is readable and shaped as expected before copying from it.
    for subdirectory, suffix in (("Annotations", ".xml"), ("JPEGImages", ".jpg")):
        source_directory = Path(dataset_source, subdirectory)
        if not source_directory.is_dir():
            raise FileNotFoundError(f"{source_directory} does not exist.")
        if next(source_directory.glob(f"*{suffix}"), None) is None:
            raise RuntimeError(f"{source_directory} contains no *{suffix} files.")
    print(f"dataset source verified: {dataset_source} (Annotations/ and JPEGImages/ "
          "both readable)")
    print("annotations and images are copied to /content on demand by cells N and N2")

for directory in (annotations_dir, jpeg_dir):
    if directory.is_symlink():
        raise RuntimeError(
            f"{directory} is a symlink. All image and annotation I/O must happen on "
            "local disk, not through the Drive FUSE mount."
        )
    if not directory.is_dir():
        raise RuntimeError(f"{directory} was not created.")
write_json(dataset_stamp_path, dataset_stamp)

# The evaluation-split construction in cell N needs the annotation of every image
# in the official test split; PROB then re-reads them on every evaluation.
official_eval_ids_all = read_ids(image_set_root / f"{OFFICIAL_EVAL_SPLIT}.txt")
stage_annotations(official_eval_ids_all,
                  description=f"{OFFICIAL_EVAL_SPLIT} annotations")

# One annotation must parse into the VOC structure PROB's dataset reader expects.
sample_annotation = annotations_dir / f"{official_eval_ids_all[0]}.xml"
sample_root = ET.parse(sample_annotation).getroot()
sample_objects = sample_root.findall("./object/name")
if sample_root.find("./size/width") is None or not sample_objects:
    raise RuntimeError(
        f"{sample_annotation} is not a VOC-style annotation with size/ and object/ "
        "elements; PROB's OWDetection reader cannot use it."
    )

split_counts = {
    split: len(read_ids(image_set_root / f"{split}.txt"))
    for split in (TASK2_SOURCE_SPLIT, TASK1_REFERENCE_SPLIT, OFFICIAL_EVAL_SPLIT)
}
if split_counts[TASK2_SOURCE_SPLIT] < SOURCE_TASK2_IMAGES:
    raise RuntimeError(
        f"{TASK2_SOURCE_SPLIT} holds {split_counts[TASK2_SOURCE_SPLIT]} images, "
        f"fewer than SOURCE_TASK2_IMAGES={SOURCE_TASK2_IMAGES}."
    )
if split_counts[TASK1_REFERENCE_SPLIT] < REFERENCE_IMAGES:
    raise RuntimeError(
        f"{TASK1_REFERENCE_SPLIT} holds {split_counts[TASK1_REFERENCE_SPLIT]} "
        f"images, fewer than REFERENCE_IMAGES={REFERENCE_IMAGES}."
    )
if split_counts[OFFICIAL_EVAL_SPLIT] < EVAL_UNKNOWN_IMAGES + EVAL_KNOWN_IMAGES:
    raise RuntimeError(
        f"{OFFICIAL_EVAL_SPLIT} holds {split_counts[OFFICIAL_EVAL_SPLIT]} unique "
        f"images, fewer than the {EVAL_UNKNOWN_IMAGES + EVAL_KNOWN_IMAGES} requested."
    )

table("DATASET", [
    ("data root", data_root),
    ("ImageSets from", repository_image_sets),
    ("dataset source", dataset_source),
    ("dataset mode", dataset_mode),
    ("staged data reused", reuse_staged_data),
    ("annotations dir", annotations_dir),
    ("annotations staged", sum(1 for _ in annotations_dir.glob("*.xml"))),
    ("JPEGImages dir", jpeg_dir),
    ("images staged so far", sum(1 for _ in jpeg_dir.glob("*.jpg"))),
    ("symlinks into Drive", 0),
    ("sample annotation", f"{sample_annotation.name} ({len(sample_objects)} objects)"),
    *[(f"split {split}", f"{count} unique image IDs")
      for split, count in split_counts.items()],
    ("free /content GB", free_gb(CONTENT_ROOT)),
])
mark("dataset")

In [ ]:
# ============================================================================
# N. PROTOCOL CONSTRUCTION AND VALIDATION
#
# Three deterministic, seed-derived sets are built and then checked against each
# other:
#
#   candidate pool  : a controlled long-tail sample of Task-2 images, built by
#                     the pinned daowod.dataset.build_long_tail_pool
#   reference set   : Task-1 images used only as the novelty reference
#   evaluation set  : a fixed balanced split (unknown-bearing + known-only)
#
# Ground truth is used for exactly two things, both recorded in protocol.json:
# constructing the long-tail pool, and constructing/scoring the evaluation split.
# It never reaches the acquisition function.
# ============================================================================
require_stages("dataset")

if PROB_PATH not in sys.path:
    sys.path.insert(0, PROB_PATH)
from datasets.torchvision_datasets.open_world import (  # noqa: E402
    BASE_VOC_CLASS_NAMES,
    VOC_CLASS_NAMES_COCOFIED,
    VOC_COCO_CLASS_NAMES,
)

class_names = list(VOC_COCO_CLASS_NAMES[DATASET])
if len(class_names) != NUM_CLASSES:
    raise RuntimeError(
        f"{DATASET} defines {len(class_names)} classes but NUM_CLASSES={NUM_CLASSES}."
    )
class_to_index = {name: index for index, name in enumerate(class_names)}
index_to_class = dict(enumerate(class_names))
known_indices = range(0, PREVIOUS_CLASSES + CURRENT_CLASSES)
unknown_indices = range(PREVIOUS_CLASSES + CURRENT_CLASSES, NUM_CLASSES - 1)
task_class_names = class_names[PREVIOUS_CLASSES:PREVIOUS_CLASSES + CURRENT_CLASSES]
task_class_set = set(task_class_names)
UNKNOWN_CLASS_INDEX = NUM_CLASSES - 1


def annotation_classes(image_id):
    """VOC class names of one image, with PROB's COCO renaming applied."""
    path = annotations_dir / f"{image_id}.xml"
    if not path.exists():
        raise FileNotFoundError(f"Missing annotation: {path}")
    names = []
    for node in ET.parse(path).getroot().findall("./object/name"):
        if not node.text:
            continue
        name = node.text.strip()
        if name in VOC_CLASS_NAMES_COCOFIED:
            name = BASE_VOC_CLASS_NAMES[VOC_CLASS_NAMES_COCOFIED.index(name)]
        names.append(name)
    return names


protocol_dir = Path(LOCAL_RESULT_ROOT, "protocol")
protocol_dir.mkdir(parents=True, exist_ok=True)

# --- candidate pool ---------------------------------------------------------
task2_source_ids = seeded_order_preserving_sample(
    read_ids(image_set_root / f"{TASK2_SOURCE_SPLIT}.txt"),
    SOURCE_TASK2_IMAGES, seed=SEED, salt="task2-source",
)
# build_long_tail_pool reads the annotation of every source image.
stage_annotations(task2_source_ids, description="task-2 source annotations")
pilot_source_path = image_set_root / f"{PILOT_SOURCE_SPLIT}.txt"
write_ids(pilot_source_path, task2_source_ids)

long_tail_dir = protocol_dir / "long_tail"
if long_tail_dir.exists():
    shutil.rmtree(long_tail_dir)
pool = build_long_tail_pool(
    annotation_dir=annotations_dir,
    source_split=pilot_source_path,
    task_class_names=task_class_names,
    output_dir=long_tail_dir,
    imbalance_ratio=IMBALANCE_RATIO,
    seed=SEED,
)
candidate_ids = [str(image_id) for image_id in pool["selected_image_ids"]]
for artifact in (pool["pool_split_path"], pool["class_stats_path"], pool["manifest_path"]):
    if not Path(artifact).exists():
        raise FileNotFoundError(f"Missing long-tail artifact: {artifact}")
if len(candidate_ids) != len(set(candidate_ids)):
    raise RuntimeError("The long-tail pool contains duplicate image IDs.")
if len(candidate_ids) < ROUNDS * BUDGET_PER_ROUND:
    raise RuntimeError(
        f"The long-tail pool holds {len(candidate_ids)} images, below the "
        f"cumulative budget {ROUNDS * BUDGET_PER_ROUND}. Raise SOURCE_TASK2_IMAGES."
    )

# --- novelty reference set --------------------------------------------------
candidate_set = set(candidate_ids)
reference_pool = [
    image_id for image_id in read_ids(image_set_root / f"{TASK1_REFERENCE_SPLIT}.txt")
    if image_id not in candidate_set
]
base_reference_ids = seeded_order_preserving_sample(
    reference_pool, REFERENCE_IMAGES, seed=SEED, salt="reference",
)
if candidate_set & set(base_reference_ids):
    raise RuntimeError("Candidate and reference IDs overlap.")

# --- fixed evaluation split -------------------------------------------------
official_eval_ids = list(official_eval_ids_all)
unknown_candidates, known_only_candidates = [], []
unknown_object_count = 0
for image_id in official_eval_ids:
    indices = [class_to_index[name] for name in annotation_classes(image_id)
               if name in class_to_index]
    if not indices:
        continue
    if any(index in unknown_indices for index in indices):
        unknown_candidates.append(image_id)
        unknown_object_count += sum(1 for index in indices if index in unknown_indices)
    elif all(index in known_indices for index in indices):
        known_only_candidates.append(image_id)

if unknown_object_count < 1:
    raise RuntimeError("The evaluation source contains no unknown ground truth.")
unknown_eval_ids = seeded_order_preserving_sample(
    unknown_candidates, EVAL_UNKNOWN_IMAGES, seed=SEED, salt="eval-unknown")
known_eval_ids = seeded_order_preserving_sample(
    known_only_candidates, EVAL_KNOWN_IMAGES, seed=SEED, salt="eval-known")
if set(unknown_eval_ids) & set(known_eval_ids):
    raise RuntimeError("The unknown-bearing and known-only evaluation sets overlap.")
evaluation_selection = set(unknown_eval_ids) | set(known_eval_ids)
evaluation_ids = [image_id for image_id in official_eval_ids
                  if image_id in evaluation_selection]
evaluation_ids_sha256 = sha256_of_ids(evaluation_ids)

if len(evaluation_ids) != EVAL_UNKNOWN_IMAGES + EVAL_KNOWN_IMAGES:
    raise RuntimeError(f"Expected {EVAL_UNKNOWN_IMAGES + EVAL_KNOWN_IMAGES} "
                       f"evaluation images, built {len(evaluation_ids)}.")
if candidate_set & set(evaluation_ids):
    raise RuntimeError("Candidate images leak into the evaluation split.")
if set(base_reference_ids) & set(evaluation_ids):
    raise RuntimeError("Reference images leak into the evaluation split.")
write_ids(image_set_root / f"{PILOT_EVAL_SPLIT}.txt", evaluation_ids)

# --- candidate-pool class distribution --------------------------------------
candidate_class_counts = Counter()
for image_id in candidate_ids:
    for name in annotation_classes(image_id):
        if name in task_class_set:
            candidate_class_counts[name] += 1
observed_imbalance = (
    max(candidate_class_counts.values()) / max(min(candidate_class_counts.values()), 1)
    if candidate_class_counts else float("nan")
)

write_ids(protocol_dir / "candidate_ids.txt", candidate_ids)
write_ids(protocol_dir / "base_reference_ids.txt", base_reference_ids)
write_ids(protocol_dir / "evaluation_ids.txt", evaluation_ids)
write_json(protocol_dir / "candidate_class_counts.json",
           dict(sorted(candidate_class_counts.items())))
PROTOCOL_RECORD = {
    "seed": SEED,
    "rounds": ROUNDS,
    "budget_per_round": BUDGET_PER_ROUND,
    "dataset": DATASET,
    "task2_source_split": TASK2_SOURCE_SPLIT,
    "task2_source_images": len(task2_source_ids),
    "task1_reference_split": TASK1_REFERENCE_SPLIT,
    "official_eval_split": OFFICIAL_EVAL_SPLIT,
    "pilot_eval_split": PILOT_EVAL_SPLIT,
    "pilot_source_split": PILOT_SOURCE_SPLIT,
    "imbalance_ratio": IMBALANCE_RATIO,
    "task_class_names": task_class_names,
    "candidate_count": len(candidate_ids),
    "candidate_class_counts": dict(sorted(candidate_class_counts.items())),
    "candidate_observed_imbalance": observed_imbalance,
    "base_reference_count": len(base_reference_ids),
    "evaluation_count": len(evaluation_ids),
    "evaluation_unknown_images": len(unknown_eval_ids),
    "evaluation_known_only_images": len(known_eval_ids),
    "evaluation_unknown_objects_in_source": unknown_object_count,
    "candidate_ids_sha256": sha256_of_ids(candidate_ids),
    "base_reference_ids_sha256": sha256_of_ids(base_reference_ids),
    "evaluation_ids_sha256": evaluation_ids_sha256,
    "candidate_ground_truth_use": "long-tail pool construction only",
    "evaluation_ground_truth_use": "evaluation split construction and metrics only",
}
write_json(protocol_dir / "protocol.json", PROTOCOL_RECORD)

table("PROTOCOL", [
    ("task-2 source images", len(task2_source_ids)),
    ("task-2 classes", f"{len(task_class_names)}: "
                       f"{task_class_names[0]} .. {task_class_names[-1]}"),
    ("candidate pool", len(candidate_ids)),
    ("candidate class coverage", f"{len(candidate_class_counts)}/{len(task_class_names)}"),
    ("candidate observed imbalance", round(observed_imbalance, 2)),
    ("base references", len(base_reference_ids)),
    ("evaluation images", f"{len(evaluation_ids)} "
                          f"({len(unknown_eval_ids)} unknown-bearing + "
                          f"{len(known_eval_ids)} known-only)"),
    ("unknown GT objects in source", unknown_object_count),
    ("candidate ids sha256", sha256_of_ids(candidate_ids)),
    ("reference ids sha256", sha256_of_ids(base_reference_ids)),
    ("evaluation ids sha256", evaluation_ids_sha256),
    ("long-tail manifest", pool["manifest_path"]),
    ("pilot eval split", image_set_root / f"{PILOT_EVAL_SPLIT}.txt"),
])
print("\nlong-tail class statistics:")
print(Path(pool["class_stats_path"]).read_text(encoding="utf-8"))
mark("protocol")

In [ ]:
# ============================================================================
# N2. STAGE AND VERIFY EVERY IMAGE THE EXPERIMENT TOUCHES
#
# PROB's OWDetection reader parses the annotation of every image it loads, at
# inference time too, so both the .jpg and the .xml of every candidate, reference
# and evaluation image must exist on local disk before any round starts. Files
# already staged are not copied or extracted again.
# ============================================================================
require_stages("protocol")

required_image_ids = unique([*candidate_ids, *base_reference_ids, *evaluation_ids])

staged_annotations = stage_annotations(
    required_image_ids, description="candidate/reference/evaluation annotations")
staged_images = stage_images(
    required_image_ids, description="candidate/reference/evaluation images")

missing_files = []
for image_id in required_image_ids:
    if not (jpeg_dir / f"{image_id}.jpg").is_file():
        missing_files.append(str(jpeg_dir / f"{image_id}.jpg"))
    if not (annotations_dir / f"{image_id}.xml").is_file():
        missing_files.append(str(annotations_dir / f"{image_id}.xml"))
if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} required files are unavailable.\n"
        f"  dataset source : {dataset_source} ({dataset_mode})\n"
        f"  images dir     : {jpeg_dir}\n"
        f"  annotations dir: {annotations_dir}\n"
        "  first 20 missing:\n"
        + "\n".join(f"    {path}" for path in missing_files[:20])
    )

# Nothing the experiment reads may resolve back onto the Drive mount.
for probe_path in (jpeg_dir / f"{required_image_ids[0]}.jpg",
                   annotations_dir / f"{required_image_ids[0]}.xml"):
    resolved = probe_path.resolve()
    if str(resolved).startswith(f"{DRIVE_MOUNT}/"):
        raise RuntimeError(
            f"{probe_path} resolves to {resolved}, which is on the Drive FUSE "
            "mount. Training and image I/O must happen on local disk."
        )

# Images must actually decode, so a truncated copy or extraction is caught now.
from PIL import Image  # noqa: E402

decoded = []
for image_id in (required_image_ids[0], required_image_ids[len(required_image_ids) // 2],
                 required_image_ids[-1]):
    path = jpeg_dir / f"{image_id}.jpg"
    with Image.open(path) as image:
        image.load()
        decoded.append(f"{path.name} {image.size} {image.mode}")

image_bytes = sum((jpeg_dir / f"{image_id}.jpg").stat().st_size
                  for image_id in required_image_ids)
annotation_count = sum(1 for _ in annotations_dir.glob("*.xml"))
table("IMAGES", [
    ("candidate images", len(candidate_ids)),
    ("reference images", len(base_reference_ids)),
    ("evaluation images", len(evaluation_ids)),
    ("unique required images", len(required_image_ids)),
    ("annotations staged now", staged_annotations),
    ("images staged now", staged_images),
    ("annotations available", annotation_count),
    ("required image bytes GB", round(image_bytes / 1024 ** 3, 3)),
    ("decoded samples", decoded),
    ("resolves onto Drive", False),
    ("free /content GB", free_gb(CONTENT_ROOT)),
])
mark("images")

In [ ]:
# ============================================================================
# O. PRE-FLIGHT: THE REAL PATHS, ON REAL DATA, BEFORE THE TWELVE TRAINING RUNS
#
# Nothing here checks `--help` and calls it a day. This cell
#
#   * builds the detector adapter and asserts the exact commands it will run,
#   * registers the experiment identity on Drive (refusing to mix runs),
#   * constructs the PROB model from the parser defaults the bridge uses, loads
#     the Task-1 checkpoint into it and verifies the architecture agrees,
#   * runs a real PROB inference through the bridge and validates the exported
#     proposals, embeddings and posteriors,
#   * runs the real DAOWOD acquisition scoring over those proposals,
#   * runs the real official OWOD evaluator end to end on a small split.
#
# The twelve rounds start only after all of this passes.
# ============================================================================
require_stages("images")

# --- the commands the detector will run -------------------------------------
COMMON_BRIDGE_ARGUMENTS = [
    "--data-root", DATA_ROOT,
    "--dataset", DATASET,
    "--prev-introduced-classes", str(PREVIOUS_CLASSES),
    "--current-introduced-classes", str(CURRENT_CLASSES),
    "--num-classes", str(NUM_CLASSES),
    "--objectness-temperature", str(OBJECTNESS_TEMPERATURE),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--device", DEVICE,
    "--seed", str(SEED),
]
common_bridge_arguments = " ".join(COMMON_BRIDGE_ARGUMENTS)
freeze_flag = "--freeze-prob-model" if FREEZE_PROB_MODEL else "--no-freeze-prob-model"

adapter = ProbAdapter(
    repository_path=PROB_PATH,
    timeout_seconds=PROB_TIMEOUT_SECONDS,
    train_command=(
        f"{sys.executable} daowod_prob_bridge.py train "
        "--labelled-ids {labelled_ids} "
        "--previous-checkpoint {previous_checkpoint} "
        "--output-checkpoint {checkpoint} --output-dir {output_dir} "
        f"--test-set {PILOT_EVAL_SPLIT} --epochs {TRAIN_EPOCHS} "
        f"--learning-rate {LEARNING_RATE} "
        f"--objectness-loss-coefficient {OBJECTNESS_LOSS_COEFFICIENT} "
        f"--eval-every {PROB_EVAL_EVERY} {freeze_flag} {common_bridge_arguments}"
    ),
    predict_command=(
        f"{sys.executable} daowod_prob_bridge.py predict "
        "--image-ids {image_ids} --checkpoint {checkpoint} --output {proposals} "
        f"--max-proposals-per-image {MAX_PROPOSALS_PER_IMAGE} "
        f"--minimum-unknown-score {MINIMUM_UNKNOWN_SCORE} {common_bridge_arguments}"
    ),
    evaluate_command=(
        f"{sys.executable} daowod_prob_bridge.py evaluate "
        "--checkpoint {checkpoint} --output {metrics} --output-dir {output_dir} "
        f"--test-set {PILOT_EVAL_SPLIT} {common_bridge_arguments}"
    ),
)

DETECTOR_COMMANDS = {
    "train": adapter.train_command,
    "predict": adapter.predict_command,
    "evaluate": adapter.evaluate_command,
}
for _name, _command in DETECTOR_COMMANDS.items():
    if not _command.startswith(f"{sys.executable} "):
        raise RuntimeError(
            f"The {_name} command does not use this kernel's interpreter "
            f"({sys.executable}): {_command}"
        )
    if f"--device {DEVICE}" not in _command:
        raise RuntimeError(f"The {_name} command does not target {DEVICE}: {_command}")
    if f"--data-root {DATA_ROOT}" not in _command:
        raise RuntimeError(f"The {_name} command does not use {DATA_ROOT}: {_command}")
    if f"--seed {SEED}" not in _command:
        raise RuntimeError(f"The {_name} command does not carry the seed: {_command}")
if Path(adapter.repository_path) != Path(PROB_PATH).resolve():
    raise RuntimeError(
        f"The adapter runs in {adapter.repository_path}, not in the pinned PROB "
        f"clone {PROB_PATH}."
    )
banner("DETECTOR COMMANDS")
for _name, _command in DETECTOR_COMMANDS.items():
    print(f"\n{_name}:\n  {_command}")


def build_acquisition_config(variant_config):
    return AcquisitionConfig(
        strategies=(variant_config["strategy"],),
        uncertainty_mode="ambiguity",
        pseudo_label_source="cluster",
        cluster_count=20,
        neighbour_count=5,
        top_k=TOP_K,
        weights=AcquisitionWeights(
            uncertainty=ALPHA,
            novelty=BETA,
            rarity=GAMMA,
            coherence_power=float(variant_config["coherence_power"]),
            rarity_power=RARITY_POWER,
        ),
    )


def detector_call(description, function, *args, **kwargs):
    """Run a detector call, adding cwd/environment context to any failure."""
    try:
        return function(*args, **kwargs)
    except Exception:
        banner(f"DETECTOR CALL FAILED: {description}")
        print("PROB repository (subprocess cwd):", adapter.repository_path)
        print("interpreter                     :", sys.executable)
        banner("RUNTIME")
        for _key, _value in runtime_diagnostics():
            print(f"{_key:<20} : {_value}")
        banner("RELEVANT ENVIRONMENT")
        print(redacted_environment(DIAGNOSTIC_ENVIRONMENT_NAMES))
        raise


acquisition_configs = {name: build_acquisition_config(config)
                       for name, config in VARIANTS.items()}

# --- experiment identity ----------------------------------------------------
experiment_config = {
    "protocol": "contribution_a_multiround_v1",
    "seed": SEED,
    "rounds": ROUNDS,
    "budget_per_round": BUDGET_PER_ROUND,
    "variants": VARIANTS,
    "acquisition": {
        name: {
            "strategy": VARIANTS[name]["strategy"],
            "uncertainty": config.weights.uncertainty,
            "novelty": config.weights.novelty,
            "rarity": config.weights.rarity,
            "coherence_power": config.weights.coherence_power,
            "rarity_power": config.weights.rarity_power,
            "uncertainty_mode": config.uncertainty_mode,
            "pseudo_label_source": config.pseudo_label_source,
            "cluster_count": config.cluster_count,
            "neighbour_count": config.neighbour_count,
            "top_k": config.top_k,
        }
        for name, config in acquisition_configs.items()
    },
    "detector": {
        "dataset": DATASET,
        "epochs": TRAIN_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "objectness_loss_coefficient": OBJECTNESS_LOSS_COEFFICIENT,
        "objectness_temperature": OBJECTNESS_TEMPERATURE,
        "freeze_prob_model": FREEZE_PROB_MODEL,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "eval_every": PROB_EVAL_EVERY,
        "max_proposals_per_image": MAX_PROPOSALS_PER_IMAGE,
        "minimum_unknown_score": MINIMUM_UNKNOWN_SCORE,
        "task1_epoch": task1_epoch,
        "previous_classes": PREVIOUS_CLASSES,
        "current_classes": CURRENT_CLASSES,
        "num_classes": NUM_CLASSES,
    },
    "data": {
        "candidate_ids_sha256": sha256_of_ids(candidate_ids),
        "base_reference_ids_sha256": sha256_of_ids(base_reference_ids),
        "evaluation_ids_sha256": evaluation_ids_sha256,
        "candidate_count": len(candidate_ids),
        "base_reference_count": len(base_reference_ids),
        "evaluation_count": len(evaluation_ids),
        "imbalance_ratio": IMBALANCE_RATIO,
    },
    "commits": {"DAOWOD": repo_commits["DAOWOD"], "PROB": repo_commits["PROB"]},
    "prob_source": {
        "commit": PROB_SOURCE_VALIDATION["commit"],
        "clean_checkout": PROB_SOURCE_VALIDATION["clean_checkout_after_validation"],
        "modified_by_notebook": PROB_SOURCE_VALIDATION["modified_by_notebook"],
        "decoder_feature_marker": PROB_SOURCE_VALIDATION["decoder_feature_marker"],
        "bridge_check_returncode": PROB_SOURCE_VALIDATION["bridge_check_returncode"],
    },
}
experiment_config_sha256 = sha256_of_object(experiment_config)

drive_config_path = Path(DRIVE_RESULT_DIR, "experiment_config.json")
if drive_config_path.exists():
    stored = read_json(drive_config_path)
    if sha256_of_object(stored) != experiment_config_sha256:
        differences = {
            key: (stored.get(key), experiment_config[key])
            for key in experiment_config if stored.get(key) != experiment_config[key]
        }
        raise RuntimeError(
            f"{drive_config_path} describes a different experiment: {differences}. "
            "Change EXPERIMENT_NAME (or SEED) instead of mixing results."
        )
    print(f"Resuming the experiment recorded in {drive_config_path}.")
elif any(Path(DRIVE_RESULT_DIR).iterdir()):
    raise RuntimeError(
        f"{DRIVE_RESULT_DIR} already holds files but no experiment_config.json; "
        "refusing to mix results from another run."
    )
write_json(Path(LOCAL_RESULT_ROOT, "experiment_config.json"), experiment_config)
write_json(drive_config_path, experiment_config)

preflight_dir = Path(LOCAL_RESULT_ROOT, "preflight")
if preflight_dir.exists():
    shutil.rmtree(preflight_dir)
preflight_dir.mkdir(parents=True)

# --- O1. PROB model construction and checkpoint agreement -------------------
banner("PRE-FLIGHT 1/4: PROB MODEL CONSTRUCTION AND CHECKPOINT LOAD")
model_settings = {
    "dataset": DATASET,
    "data_root": DATA_ROOT,
    "previous_classes": PREVIOUS_CLASSES,
    "current_classes": CURRENT_CLASSES,
    "num_classes": NUM_CLASSES,
    "objectness_temperature": OBJECTNESS_TEMPERATURE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "seed": SEED,
}
MODEL_PROBE = torch_snippet(
    "import json, sys",
    "from main_open_world import get_args_parser",
    "from models import build_model",
    "",
    "settings = json.loads(sys.argv[1])",
    "checkpoint_path = sys.argv[2]",
    "args = get_args_parser().parse_args([])",
    "args.device = 'cuda'",
    "args.dataset = settings['dataset']",
    "args.data_root = settings['data_root']",
    "args.PREV_INTRODUCED_CLS = settings['previous_classes']",
    "args.CUR_INTRODUCED_CLS = settings['current_classes']",
    "args.num_classes = settings['num_classes']",
    "args.obj_temp = settings['objectness_temperature']",
    "args.batch_size = settings['batch_size']",
    "args.num_workers = settings['num_workers']",
    "args.model_type = 'prob'",
    "args.seed = settings['seed']",
    "args.wandb_project = ''",
    "args.wandb_name = ''",
    "",
    "model, criterion, postprocessors, _ = build_model(args, mode='prob')",
    "device = torch.device('cuda')",
    "model.to(device)",
    "criterion.to(device)",
    "model.eval()",
    "parameters = sum(p.numel() for p in model.parameters())",
    "",
    "state = torch.load(checkpoint_path, map_location='cpu', weights_only=False)",
    "loaded = model.load_state_dict(state['model'], strict=False)",
    "model_keys = list(model.state_dict())",
    "matched = len(model_keys) - len(loaded.missing_keys)",
    "fraction = matched / max(len(model_keys), 1)",
    "print('model tensors        :', len(model_keys))",
    "print('checkpoint tensors   :', len(state['model']))",
    "print('missing keys         :', len(loaded.missing_keys))",
    "for key in loaded.missing_keys[:20]:",
    "    print('  missing   ', key)",
    "print('unexpected keys      :', len(loaded.unexpected_keys))",
    "for key in loaded.unexpected_keys[:20]:",
    "    print('  unexpected', key)",
    "critical = sorted(key for key in loaded.missing_keys if any(",
    "    part in key for part in ('backbone', 'transformer', 'class_embed', 'bbox_embed')))",
    "assert not critical, (",
    "    'the checkpoint does not provide load-bearing weights: ' + str(critical[:20]))",
    "assert fraction >= 0.98, (",
    "    'only %.4f of the model tensors were loaded from the checkpoint' % fraction)",
    "",
    "saved = state.get('args')",
    "fields = ['backbone', 'hidden_dim', 'nheads', 'dim_feedforward', 'enc_layers',",
    "          'dec_layers', 'num_queries', 'num_feature_levels', 'enc_n_points',",
    "          'dec_n_points', 'two_stage', 'with_box_refine']",
    "mismatch = []",
    "if saved is None:",
    "    print('the checkpoint carries no args; the architecture is compared only "
    "through the state-dict key match above')",
    "else:",
    "    for field in fields:",
    "        want = getattr(args, field, None)",
    "        have = getattr(saved, field, None)",
    "        print('%-22s parser=%-12r checkpoint=%r' % (field, want, have))",
    "        if have is not None and have != want:",
    "            mismatch.append([field, want, have])",
    "    print('checkpoint obj_temp  :', getattr(saved, 'obj_temp', None))",
    "    print('checkpoint PREV/CUR  :', getattr(saved, 'PREV_INTRODUCED_CLS', None),",
    "          getattr(saved, 'CUR_INTRODUCED_CLS', None))",
    "assert not mismatch, 'architecture mismatch: ' + json.dumps(mismatch, default=str)",
    "",
    "torch.cuda.synchronize()",
    "print(json.dumps({",
    "    'parameters': int(parameters),",
    "    'model_tensors': len(model_keys),",
    "    'checkpoint_tensors': len(state['model']),",
    "    'missing_keys': len(loaded.missing_keys),",
    "    'unexpected_keys': len(loaded.unexpected_keys),",
    "    'loaded_fraction': round(fraction, 6),",
    "    'checkpoint_has_args': saved is not None,",
    "    'postprocessors': sorted(postprocessors),",
    "    'criterion': type(criterion).__name__,",
    "}))",
)
prob_model = json_probe(
    MODEL_PROBE, cwd=PROB_PATH, timeout=3600,
    arguments=[json.dumps(model_settings), str(task1_checkpoint)],
)
cleanup_runtime()

# --- O2. a real PROB inference through the bridge ---------------------------
banner("PRE-FLIGHT 2/4: REAL PROB INFERENCE AND PROPOSAL EXPORT")
preflight_predict_ids = base_reference_ids[:PREFLIGHT_PREDICT_IMAGES]
preflight_proposals_path = preflight_dir / "preflight_proposals.npz"
detector_call(
    "pre-flight predict",
    adapter.predict,
    preflight_predict_ids,
    checkpoint=task1_checkpoint,
    output_path=preflight_proposals_path,
)

preflight_metadata_path = preflight_proposals_path.with_suffix(".json")
if not preflight_metadata_path.is_file():
    raise FileNotFoundError(f"The bridge wrote no export metadata: {preflight_metadata_path}")
preflight_metadata = read_json(preflight_metadata_path)

proposals = ProposalBatch.load(preflight_proposals_path)
proposal_count = proposals.image_ids.shape[0]
exported_ids = {str(value) for value in proposals.image_ids.tolist()}
if not exported_ids:
    raise RuntimeError("The proposal export contains no image IDs.")
if not exported_ids <= set(preflight_predict_ids):
    raise RuntimeError(
        f"Unexpected exported image IDs: {sorted(exported_ids - set(preflight_predict_ids))}"
    )
if exported_ids != set(preflight_predict_ids):
    raise RuntimeError(
        "The export is missing images: "
        f"{sorted(set(preflight_predict_ids) - exported_ids)}"
    )
if proposal_count > len(preflight_predict_ids) * MAX_PROPOSALS_PER_IMAGE:
    raise RuntimeError(f"{proposal_count} proposals exceed the configured cap.")
for name, array in (("confidence", proposals.confidence),
                    ("embeddings", proposals.embeddings),
                    ("posterior", proposals.posterior),
                    ("boxes", proposals.boxes),
                    ("objectness", proposals.objectness)):
    if array is None:
        raise RuntimeError(f"The bridge did not export '{name}'.")
    if not np.all(np.isfinite(array)):
        raise RuntimeError(f"Exported '{name}' contains non-finite values.")
if proposals.predicted_labels is None:
    raise RuntimeError("The bridge did not export 'predicted_labels'.")
if not np.all((proposals.confidence >= 0.0) & (proposals.confidence <= 1.0)):
    raise RuntimeError("Exported unknown confidences fall outside [0, 1].")
posterior_sums = proposals.posterior.sum(axis=1)
if not np.allclose(posterior_sums, 1.0, atol=1e-6):
    raise RuntimeError(f"Posterior rows do not sum to 1: min={posterior_sums.min()}, "
                       f"max={posterior_sums.max()}")
if proposals.embeddings.shape[1] != int(preflight_metadata["feature_dimension"]):
    raise RuntimeError("Embedding width disagrees with the export metadata.")
if np.allclose(proposals.embeddings, 0.0):
    raise RuntimeError(
        "Every exported decoder embedding is zero, so pred_features carries no "
        "signal and novelty/rarity/coherence would be meaningless."
    )
valid_labels = set(range(PREVIOUS_CLASSES + CURRENT_CLASSES)) | {UNKNOWN_CLASS_INDEX}
unexpected_labels = sorted(set(proposals.predicted_labels.tolist()) - valid_labels)
if unexpected_labels:
    raise RuntimeError(
        f"predicted_labels contains indices outside the introduced classes and the "
        f"unknown index: {unexpected_labels}"
    )

# --- O3. the real DAOWOD acquisition path over those proposals --------------
banner("PRE-FLIGHT 3/4: REAL ACQUISITION SCORING")
preflight_scores = score_proposals(
    strategy="full",
    uncertainty_mode="ambiguity",
    pseudo_label_source="cluster",
    confidence=proposals.confidence,
    posterior=proposals.posterior,
    embeddings=proposals.embeddings,
    reference_embeddings=proposals.embeddings,
    predicted_labels=proposals.predicted_labels,
    cluster_count=min(20, proposal_count),
    neighbour_count=min(5, max(proposal_count - 1, 1)),
    seed=SEED,
    weights=acquisition_configs["full_p1"].weights,
)
preflight_selection = select_images(proposals.image_ids, preflight_scores.scores,
                                   budget=1, top_k=TOP_K)
for name, array in (("uncertainty", preflight_scores.uncertainty),
                    ("novelty", preflight_scores.novelty),
                    ("rarity", preflight_scores.rarity),
                    ("coherence", preflight_scores.coherence),
                    ("scores", preflight_scores.scores)):
    if array.shape != (proposal_count,) or not np.all(np.isfinite(array)):
        raise RuntimeError(f"Acquisition component '{name}' is malformed.")
if float(preflight_scores.scores.std()) == 0.0:
    raise RuntimeError(
        "Every acquisition score is identical, so the acquisition function cannot "
        "rank anything."
    )
if len(preflight_selection) != 1:
    raise RuntimeError(f"select_images returned {preflight_selection}.")

# --- O4. the real official OWOD evaluator, end to end -----------------------
banner("PRE-FLIGHT 4/4: REAL OFFICIAL OWOD EVALUATION")
preflight_unknown = unknown_eval_ids[:PREFLIGHT_EVAL_IMAGES // 2]
preflight_known = known_eval_ids[:PREFLIGHT_EVAL_IMAGES - PREFLIGHT_EVAL_IMAGES // 2]
preflight_eval_ids = unique([*preflight_unknown, *preflight_known])
write_ids(image_set_root / f"{PREFLIGHT_EVAL_SPLIT}.txt", preflight_eval_ids)
preflight_metrics_path = preflight_dir / "preflight_metrics.json"
run([sys.executable, str(bridge_path), "evaluate",
     "--checkpoint", str(task1_checkpoint),
     "--output", str(preflight_metrics_path),
     "--output-dir", str(preflight_dir / "evaluator_artifacts"),
     "--test-set", PREFLIGHT_EVAL_SPLIT,
     *COMMON_BRIDGE_ARGUMENTS],
    cwd=PROB_PATH, timeout=PROB_TIMEOUT_SECONDS)

preflight_metrics = read_json(preflight_metrics_path)
REQUIRED_METRICS = ("known_mAP", "U_Recall", "WI", "A_OSE",
                    "unknown_AP50", "previous_known_AP50")
missing_metrics = [key for key in REQUIRED_METRICS if key not in preflight_metrics]
if missing_metrics:
    raise RuntimeError(
        f"The evaluator produced no {missing_metrics}; every round's metrics.json "
        f"is validated against {list(REQUIRED_METRICS)}. Got: {sorted(preflight_metrics)}"
    )
if not isinstance(preflight_metrics["A_OSE"], int):
    raise RuntimeError(f"A_OSE is {preflight_metrics['A_OSE']!r}, not an integer.")
for _key in ("known_mAP", "U_Recall", "WI", "unknown_AP50"):
    if not isinstance(preflight_metrics[_key], (int, float)) \
            or not math.isfinite(float(preflight_metrics[_key])):
        raise RuntimeError(f"Metric {_key} is not a finite number: {preflight_metrics[_key]!r}")
if "official_metrics" not in preflight_metrics:
    raise RuntimeError("The evaluator did not record the official metric dictionary.")

PREFLIGHT_RECORD = {
    "model": prob_model,
    "predict": {
        "images": preflight_predict_ids,
        "proposals": proposal_count,
        "feature_dimension": int(proposals.embeddings.shape[1]),
        "metadata": preflight_metadata,
        "confidence_range": [float(proposals.confidence.min()),
                             float(proposals.confidence.max())],
        "objectness_range": [float(proposals.objectness.min()),
                             float(proposals.objectness.max())],
    },
    "acquisition": {
        "mean_uncertainty": round(float(preflight_scores.uncertainty.mean()), 6),
        "mean_novelty": round(float(preflight_scores.novelty.mean()), 6),
        "mean_rarity": round(float(preflight_scores.rarity.mean()), 6),
        "mean_coherence": round(float(preflight_scores.coherence.mean()), 6),
        "score_std": round(float(preflight_scores.scores.std()), 6),
        "selected": [str(value) for value in preflight_selection],
    },
    "evaluation": {
        "split": PREFLIGHT_EVAL_SPLIT,
        "images": preflight_eval_ids,
        "metrics": {key: preflight_metrics[key] for key in REQUIRED_METRICS},
    },
}
write_json(preflight_dir / "preflight_record.json", PREFLIGHT_RECORD)

table("PRE-FLIGHT RESULTS", [
    ("PROB parameters", f"{prob_model['parameters']:,}"),
    ("model tensors", prob_model["model_tensors"]),
    ("checkpoint tensors", prob_model["checkpoint_tensors"]),
    ("state-dict loaded fraction", prob_model["loaded_fraction"]),
    ("missing / unexpected keys", f"{prob_model['missing_keys']} / "
                                  f"{prob_model['unexpected_keys']}"),
    ("checkpoint carries args", prob_model["checkpoint_has_args"]),
    ("postprocessors", prob_model["postprocessors"]),
    ("prediction images", len(preflight_predict_ids)),
    ("exported proposals", proposal_count),
    ("feature dimension", proposals.embeddings.shape[1]),
    ("confidence range", f"[{proposals.confidence.min():.3e}, "
                         f"{proposals.confidence.max():.3e}]"),
    ("objectness range", f"[{proposals.objectness.min():.3e}, "
                         f"{proposals.objectness.max():.3e}]"),
    ("posterior row sums", f"[{posterior_sums.min():.6f}, {posterior_sums.max():.6f}]"),
    ("mean uncertainty", PREFLIGHT_RECORD["acquisition"]["mean_uncertainty"]),
    ("mean novelty", PREFLIGHT_RECORD["acquisition"]["mean_novelty"]),
    ("mean rarity", PREFLIGHT_RECORD["acquisition"]["mean_rarity"]),
    ("mean coherence", PREFLIGHT_RECORD["acquisition"]["mean_coherence"]),
    ("score spread", PREFLIGHT_RECORD["acquisition"]["score_std"]),
    ("selected image", PREFLIGHT_RECORD["acquisition"]["selected"]),
    ("evaluation split", f"{PREFLIGHT_EVAL_SPLIT} ({len(preflight_eval_ids)} images)"),
    *[(f"evaluator {key}", preflight_metrics[key]) for key in REQUIRED_METRICS],
])
banner("PRE-FLIGHT COMPLETE")
print("Build, import, CUDA execution, bridge, model construction, real inference,")
print("real acquisition scoring and real official evaluation all passed.")
print(f"The {len(VARIANTS) * ROUNDS} training runs may now start.")
mark("preflight")
cleanup_runtime()

if STOP_AFTER_SMOKE_TESTS:
    raise SystemExit("STOP_AFTER_SMOKE_TESTS is True; stopping before the rounds.")

In [ ]:
# ============================================================================
# P1. ROUND MACHINERY: RUN, VALIDATE, RECORD, PERSIST, RESTORE
#
# Defined once and used by the single-round gate and by the full campaign. Every
# round produces, beyond what the pinned DAOWOD code writes:
#
#   round_identity.json  the identity a resume is allowed to trust
#   round_record.json    the scientific record: selections, per-proposal and
#                        per-image acquisition components, class/distribution
#                        summaries, the exact training command, checkpoint hash,
#                        metrics, elapsed time, seed, commits, runtime and GPU
#
# Persistence is staged: a round is copied to Drive under a `.__copying__` name,
# validated there, atomically renamed, validated again, and its two largest files
# are compared by SHA-256 across the FUSE boundary.
# ============================================================================
require_stages("preflight")

COMMON_ROUND_FILES = (
    "candidate_proposals.npz",
    "selected_ids.txt",
    "labelled_ids.txt",
    "remaining_pool_ids.txt",
    "checkpoint.pth",
    "metrics.json",
    "round_manifest.json",
)
SCORED_ROUND_FILES = (
    "reference_proposals.npz",
    "proposal_scores.csv",
    "image_scores.csv",
)
NOTEBOOK_ROUND_FILES = ("round_identity.json", "round_record.json")
EXTRA_CHECKPOINT_PATTERN = re.compile(r"^checkpoint\d+\.pth$")


def round_directory(root, variant, index):
    return Path(root, variant, f"round_{index:02d}")


def round_identity(variant, index, record):
    return {
        "protocol": "contribution_a_multiround_v1",
        "variant": variant,
        "strategy": VARIANTS[variant]["strategy"],
        "coherence_power": float(VARIANTS[variant]["coherence_power"]),
        "round": int(index),
        "seed": int(SEED),
        "budget": int(BUDGET_PER_ROUND),
        "daowod_commit": repo_commits["DAOWOD"],
        "prob_commit": repo_commits["PROB"],
        "candidate_ids_sha256": sha256_of_ids(record["candidate_ids"]),
        "reference_ids_sha256": sha256_of_ids(record["reference_ids"]),
        "labelled_before_sha256": sha256_of_ids(record["labelled_ids"]),
        "evaluation_ids_sha256": evaluation_ids_sha256,
        "experiment_config_sha256": experiment_config_sha256,
    }


def validate_round(variant, index, directory, record):
    """Check a round's artifacts against the repository's own semantics."""
    directory = Path(directory)
    strategy = VARIANTS[variant]["strategy"]
    expected = list(COMMON_ROUND_FILES)
    if strategy != "random":
        expected.extend(SCORED_ROUND_FILES)
    missing = [name for name in expected if not (directory / name).exists()]
    if missing:
        raise RuntimeError(f"{variant} round {index}: missing artifacts {missing} in {directory}")

    manifest = read_json(directory / "round_manifest.json")
    metrics = read_json(directory / "metrics.json")
    selected = read_ids(directory / "selected_ids.txt")
    labelled = read_ids(directory / "labelled_ids.txt")
    remaining = read_ids(directory / "remaining_pool_ids.txt")

    checks = {
        "manifest completed": manifest.get("completed") is True,
        "manifest strategy": manifest.get("strategy") == strategy,
        "manifest round": manifest.get("round_index") == index,
        "manifest seed": manifest.get("seed") == SEED,
        "manifest budget": manifest.get("budget") == BUDGET_PER_ROUND,
        "manifest candidates": manifest.get("candidate_count_before") == len(record["candidate_ids"]),
        "manifest references": manifest.get("reference_count") == len(record["reference_ids"]),
        "manifest labelled": manifest.get("labelled_count_before") == len(record["labelled_ids"]),
        "selected size": len(selected) == len(set(selected)) == BUDGET_PER_ROUND,
        "selected from pool": set(selected) <= set(record["candidate_ids"]),
        "selected removed": set(selected).isdisjoint(remaining),
        "pool shrank": len(remaining) == len(record["candidate_ids"]) - BUDGET_PER_ROUND,
        "labelled grew": labelled == [*record["labelled_ids"], *selected],
        "metrics present": all(key in metrics for key in REQUIRED_METRICS),
    }
    coherence_power = (
        manifest.get("acquisition_parameters", {}).get("weights", {}).get("coherence_power")
    )
    if strategy == "full":
        checks["coherence power"] = (
            float(coherence_power) == float(VARIANTS[variant]["coherence_power"])
        )
    if strategy == "random":
        checks["no scoring artifacts"] = not any(
            (directory / name).exists() for name in SCORED_ROUND_FILES
        )
    else:
        checks["reference hash recorded"] = bool(manifest.get("reference_proposals_sha256"))

    with np.load(directory / "candidate_proposals.npz", allow_pickle=True) as proposals:
        proposal_ids = {str(value) for value in proposals["image_ids"].tolist()}
    checks["proposals cover pool"] = proposal_ids == set(record["candidate_ids"])

    if strategy == "rarity_no_coherence":
        # The ungated variant must apply rarity without the coherence gate.
        with (directory / "proposal_scores.csv").open(newline="", encoding="utf-8") as handle:
            ungated = all(
                math.isclose(float(row["rarity_bonus"]), float(row["rarity"]),
                             rel_tol=1e-9, abs_tol=1e-12)
                for row in csv.DictReader(handle)
            )
        checks["rarity bonus ungated"] = ungated

    failed = sorted(name for name, ok in checks.items() if not ok)
    if failed:
        raise RuntimeError(f"{variant} round {index}: failed checks {failed} in {directory}")

    return {
        "manifest": manifest,
        "metrics": metrics,
        "selected": selected,
        "labelled": labelled,
        "remaining": remaining,
        "directory": directory,
    }


def verify_checkpoint_save(variant, index, directory, input_checkpoint):
    """Prove the round produced a loadable, advanced PROB checkpoint."""
    path = Path(directory) / "checkpoint.pth"
    if not path.is_file():
        raise FileNotFoundError(f"{variant} round {index}: no checkpoint at {path}")
    if path.stat().st_size < 1024 * 1024:
        raise RuntimeError(
            f"{variant} round {index}: checkpoint is only {path.stat().st_size} bytes."
        )
    produced = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(produced, dict) or "model" not in produced:
        raise RuntimeError(f"{path} is not a PROB training checkpoint.")
    if not isinstance(produced.get("epoch"), int):
        raise RuntimeError(f"{path} has no integer 'epoch'; the next round cannot "
                           "derive its epoch range from it.")
    previous = torch.load(input_checkpoint, map_location="cpu", weights_only=False)
    expected_epoch = int(previous["epoch"]) + TRAIN_EPOCHS
    if produced["epoch"] != expected_epoch:
        raise RuntimeError(
            f"{path} is at epoch {produced['epoch']}, expected {expected_epoch} "
            f"({previous['epoch']} + {TRAIN_EPOCHS} fine-tuning epochs)."
        )
    if len(produced["model"]) != len(previous["model"]):
        raise RuntimeError(
            f"{path} holds {len(produced['model'])} tensors but the input checkpoint "
            f"holds {len(previous['model'])}."
        )
    changed = sum(
        1 for key, value in produced["model"].items()
        if key in previous["model"]
        and not torch.equal(value.float(), previous["model"][key].float())
    )
    if changed == 0:
        raise RuntimeError(
            f"{path} is numerically identical to {input_checkpoint}: training "
            "updated no weights."
        )
    summary = {
        "epoch": produced["epoch"],
        "previous_epoch": int(previous["epoch"]),
        "tensors": len(produced["model"]),
        "tensors_changed": changed,
        "bytes": path.stat().st_size,
        "has_optimizer": "optimizer" in produced,
    }
    del produced, previous
    cleanup_runtime()
    print(f"{variant} round {index} checkpoint verified: {summary}")
    return summary


def read_component_rows(path):
    """Per-proposal acquisition components as plain dicts, in file order."""
    if not Path(path).is_file():
        return []
    with Path(path).open(newline="", encoding="utf-8") as handle:
        return [dict(row) for row in csv.DictReader(handle)]


def component_statistics(rows, field):
    values = [float(row[field]) for row in rows if row.get(field) not in (None, "")]
    if not values:
        return None
    array = np.asarray(values, dtype=np.float64)
    return {
        "count": int(array.size),
        "mean": round(float(array.mean()), 8),
        "std": round(float(array.std()), 8),
        "min": round(float(array.min()), 8),
        "max": round(float(array.max()), 8),
    }


def distribution_summary(directory, selected):
    """Pseudo-label and ground-truth class distributions for one round."""
    with np.load(Path(directory, "candidate_proposals.npz"), allow_pickle=True) as data:
        labels = np.asarray(data["predicted_labels"], dtype=np.int64) \
            if "predicted_labels" in data else np.zeros(0, dtype=np.int64)
        confidence = np.asarray(data["confidence"], dtype=np.float64)
        image_ids = [str(value) for value in data["image_ids"].tolist()]
    proposal_label_counts = Counter(
        index_to_class.get(int(label), f"index_{int(label)}") for label in labels.tolist()
    )
    selected_ground_truth = Counter()
    for image_id in selected:
        for name in annotation_classes(image_id):
            if name in task_class_set:
                selected_ground_truth[name] += 1
    pool_ground_truth = Counter()
    for image_id in dict.fromkeys(image_ids):
        for name in annotation_classes(image_id):
            if name in task_class_set:
                pool_ground_truth[name] += 1
    return {
        "proposals": int(labels.size),
        "images_scored": len(dict.fromkeys(image_ids)),
        "unknown_confidence": {
            "mean": round(float(confidence.mean()), 8),
            "max": round(float(confidence.max()), 8),
            "min": round(float(confidence.min()), 8),
        },
        "proposal_pseudo_label_counts": dict(sorted(proposal_label_counts.items())),
        "unknown_index_proposals": int((labels == UNKNOWN_CLASS_INDEX).sum()),
        "pool_task_class_counts": dict(sorted(pool_ground_truth.items())),
        "selected_task_class_counts": dict(sorted(selected_ground_truth.items())),
        "selected_task_classes_covered": len(selected_ground_truth),
    }


def build_round_record(variant, index, directory, record, data, *, checkpoint_summary,
                       elapsed_seconds, resumed):
    """The complete, machine-readable scientific record of one round."""
    directory = Path(directory)
    checkpoint_path = directory / "checkpoint.pth"
    proposal_rows = read_component_rows(directory / "proposal_scores.csv")
    image_rows = read_component_rows(directory / "image_scores.csv")
    strategy = VARIANTS[variant]["strategy"]
    weights = acquisition_configs[variant].weights
    return {
        "experiment": EXPERIMENT_NAME,
        "protocol": "contribution_a_multiround_v1",
        "variant": variant,
        "strategy": strategy,
        "coherence_power": float(VARIANTS[variant]["coherence_power"]),
        "round": int(index),
        "seed": int(SEED),
        "budget": int(BUDGET_PER_ROUND),
        "resumed": bool(resumed),
        "status": "completed",
        "elapsed_seconds": elapsed_seconds,
        "input_checkpoint": str(record["input_checkpoint"]),
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_sha256": sha256_of_file(checkpoint_path),
        "checkpoint_summary": checkpoint_summary,
        "selected_image_ids": list(data["selected"]),
        "labelled_image_ids_after": list(data["labelled"]),
        "remaining_pool_ids": list(data["remaining"]),
        "candidate_ids_before": list(record["candidate_ids"]),
        "reference_ids": list(record["reference_ids"]),
        "counts": {
            "candidates_before": len(record["candidate_ids"]),
            "candidates_after": len(data["remaining"]),
            "references": len(record["reference_ids"]),
            "labelled_before": len(record["labelled_ids"]),
            "labelled_after": len(data["labelled"]),
        },
        "acquisition": {
            "strategy": strategy,
            "uncertainty_weight": weights.uncertainty,
            "novelty_weight": weights.novelty,
            "rarity_weight": weights.rarity,
            "coherence_power": weights.coherence_power,
            "rarity_power": weights.rarity_power,
            "uncertainty_mode": acquisition_configs[variant].uncertainty_mode,
            "pseudo_label_source": acquisition_configs[variant].pseudo_label_source,
            "cluster_count": acquisition_configs[variant].cluster_count,
            "neighbour_count": acquisition_configs[variant].neighbour_count,
            "top_k": acquisition_configs[variant].top_k,
            "components_available": bool(proposal_rows),
            "components_note": (
                "the random baseline selects without scoring, so the pinned "
                "run_active_round writes no per-proposal components"
                if strategy == "random" else
                "per-proposal components are in proposal_scores.csv"
            ),
            "proposal_component_statistics": {
                field: component_statistics(proposal_rows, field)
                for field in ("uncertainty", "novelty", "rarity", "coherence",
                              "rarity_bonus", "score")
            } if proposal_rows else {},
            "image_score_statistics": component_statistics(image_rows, "score"),
            "proposal_rows": len(proposal_rows),
            "image_rows": len(image_rows),
        },
        "distribution": distribution_summary(directory, data["selected"]),
        "metrics": data["metrics"],
        "round_manifest": data["manifest"],
        "training": {
            "command_template": adapter.train_command,
            "resolved": {
                "labelled_ids": str(directory / "labelled_ids.txt"),
                "previous_checkpoint": str(record["input_checkpoint"]),
                "output_checkpoint": str(checkpoint_path),
                "output_dir": str(directory),
            },
            "epochs": TRAIN_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "objectness_loss_coefficient": OBJECTNESS_LOSS_COEFFICIENT,
            "freeze_prob_model": FREEZE_PROB_MODEL,
            "eval_every": PROB_EVAL_EVERY,
            "test_set": PILOT_EVAL_SPLIT,
        },
        "predict_command_template": adapter.predict_command,
        "evaluate_command_template": adapter.evaluate_command,
        "commits": {"DAOWOD": repo_commits["DAOWOD"], "PROB": repo_commits["PROB"]},
        "runtime": RUNTIME_INFO,
        "experiment_config_sha256": experiment_config_sha256,
        "evaluation_split": PILOT_EVAL_SPLIT,
        "evaluation_ids_sha256": evaluation_ids_sha256,
    }


def prune_extra_checkpoints(directory):
    """Drop PROB's per-epoch duplicates of checkpoint.pth (weights are identical)."""
    if KEEP_EXTRA_EPOCH_CHECKPOINTS:
        return []
    removed = []
    for path in sorted(Path(directory).glob("checkpoint*.pth")):
        if EXTRA_CHECKPOINT_PATTERN.match(path.name):
            removed.append(path.name)
            path.unlink()
    if removed:
        print(f"removed redundant per-epoch checkpoints: {removed}")
    return removed


def persist_round_to_drive(variant, index, record):
    local = round_directory(LOCAL_RESULT_ROOT, variant, index)
    target = round_directory(DRIVE_RESULT_DIR, variant, index)
    staging = target.with_name(f"{target.name}.__copying__")
    if target.exists():
        raise RuntimeError(f"A completed Drive round already exists: {target}")
    if staging.exists():
        shutil.rmtree(staging)
    require_free_space(DRIVE_RESULT_PARENT, MINIMUM_DRIVE_FREE_GB,
                       f"Persisting {variant} round {index}")
    for name in NOTEBOOK_ROUND_FILES:
        if not (local / name).is_file():
            raise RuntimeError(f"{local / name} is missing; refusing to persist.")
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local, staging)
    validate_round(variant, index, staging, record)
    staging.replace(target)
    validate_round(variant, index, target, record)
    # Drive is a FUSE mount: confirm the bytes that matter actually landed.
    for name in ("checkpoint.pth", "candidate_proposals.npz"):
        local_digest = sha256_of_file(local / name)
        drive_digest = sha256_of_file(target / name)
        if local_digest != drive_digest:
            raise RuntimeError(
                f"{name} differs between {local} ({local_digest}) and Drive at "
                f"{target} ({drive_digest}); the Drive copy is not trustworthy."
            )
        print(f"  Drive {name} sha256 verified: {drive_digest}")
    print(f"persisted {target} ({directory_size_gb(target)} GB)")


def restore_round_from_drive(variant, index, record):
    """Return a validated Drive round, or None when there is nothing to resume."""
    source = round_directory(DRIVE_RESULT_DIR, variant, index)
    if not source.is_dir():
        return None
    for name in NOTEBOOK_ROUND_FILES:
        if not (source / name).is_file():
            raise RuntimeError(
                f"{source} has no {name}; it was not written by a completed run of "
                "this notebook and will not be resumed. Delete it to recompute the "
                "round."
            )
    expected = round_identity(variant, index, record)
    stored = read_json(source / "round_identity.json")
    if stored != expected:
        differences = {
            key: (stored.get(key), expected[key])
            for key in expected if stored.get(key) != expected[key]
        }
        raise RuntimeError(
            f"{source / 'round_identity.json'} does not match this configuration: "
            f"{differences}"
        )
    if not RESUME_COMPLETED_ROUNDS:
        raise RuntimeError(
            f"A completed Drive round exists but RESUME_COMPLETED_ROUNDS is False: "
            f"{source}"
        )
    destination = round_directory(LOCAL_RESULT_ROOT, variant, index)
    if destination.exists():
        shutil.rmtree(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination)
    return validate_round(variant, index, destination, record)


def execute_round(variant, index, state):
    """Run or resume one active-learning round and return its validated result."""
    stage = f"{variant} round {index}"
    record = {
        "candidate_ids": list(state["candidate_ids"]),
        "reference_ids": unique([*base_reference_ids, *state["labelled_ids"]]),
        "labelled_ids": list(state["labelled_ids"]),
        "input_checkpoint": str(state["checkpoint"]),
    }
    if len(record["candidate_ids"]) < BUDGET_PER_ROUND:
        raise RuntimeError(f"{stage}: {len(record['candidate_ids'])} candidates left, "
                           f"below the budget {BUDGET_PER_ROUND}.")
    if set(record["candidate_ids"]) & set(record["reference_ids"]):
        raise RuntimeError(f"{stage}: candidate and reference sets overlap.")
    if not Path(record["input_checkpoint"]).is_file():
        raise FileNotFoundError(f"{stage}: input checkpoint is missing: "
                                f"{record['input_checkpoint']}")

    resumed = restore_round_from_drive(variant, index, record)
    if resumed is not None:
        checkpoint_summary = verify_checkpoint_save(
            variant, index, resumed["directory"], state["checkpoint"])
        stored_record = read_json(Path(resumed["directory"], "round_record.json"))
        if stored_record.get("checkpoint_sha256") != sha256_of_file(
                Path(resumed["directory"], "checkpoint.pth")):
            raise RuntimeError(
                f"{stage}: the restored round_record.json records a different "
                "checkpoint digest than the restored checkpoint."
            )
        print(f"{stage} restored from Drive "
              f"(originally {elapsed_text(stored_record.get('elapsed_seconds') or 0)})")
        mark(stage, "RESTORED")
        return resumed, record, True, checkpoint_summary, stored_record

    directory = round_directory(LOCAL_RESULT_ROOT, variant, index)
    if directory.exists():
        shutil.rmtree(directory)
    require_free_space(CONTENT_ROOT, MINIMUM_LOCAL_FREE_GB, f"Running {stage}")

    banner(f"RUNNING {stage}")
    print(f"strategy         : {VARIANTS[variant]['strategy']} "
          f"(coherence power {VARIANTS[variant]['coherence_power']})")
    print(f"input checkpoint : {record['input_checkpoint']}")
    print(f"candidates       : {len(record['candidate_ids'])}")
    print(f"references       : {len(record['reference_ids'])}")
    print(f"labelled before  : {len(record['labelled_ids'])}")
    started = time.time()
    detector_call(
        stage,
        run_active_round,
        adapter=adapter,
        checkpoint=state["checkpoint"],
        candidate_ids=record["candidate_ids"],
        reference_ids=record["reference_ids"],
        labelled_ids=record["labelled_ids"],
        output_dir=directory,
        strategy=VARIANTS[variant]["strategy"],
        budget=BUDGET_PER_ROUND,
        acquisition_config=acquisition_configs[variant],
        seed=SEED,
        round_index=index,
    )
    elapsed = round(time.time() - started, 1)
    prune_extra_checkpoints(directory)
    checkpoint_summary = verify_checkpoint_save(variant, index, directory,
                                                state["checkpoint"])
    data = validate_round(variant, index, directory, record)
    round_record = build_round_record(
        variant, index, directory, record, data,
        checkpoint_summary=checkpoint_summary, elapsed_seconds=elapsed, resumed=False)
    write_json(directory / "round_identity.json", round_identity(variant, index, record))
    write_json(directory / "round_record.json", round_record)
    persist_round_to_drive(variant, index, record)
    print(f"{stage} finished in {elapsed_text(elapsed)} ({elapsed} s)")
    mark(stage)
    return data, record, False, checkpoint_summary, round_record


def commit_round(variant, index, state, data, record, resumed, round_record):
    """Record a finished round and move the variant's state to the next round."""
    round_history.append({
        "variant": variant,
        "round": index,
        "resumed": resumed,
        "elapsed_seconds": round_record.get("elapsed_seconds"),
        "candidate_count_before": len(record["candidate_ids"]),
        "reference_count": len(record["reference_ids"]),
        "labelled_count_before": len(record["labelled_ids"]),
        "selected": list(data["selected"]),
        "checkpoint_sha256": round_record.get("checkpoint_sha256"),
    })
    round_records[(variant, index)] = round_record
    completed_rounds.add((variant, index))
    state["checkpoint"] = str(Path(data["directory"], "checkpoint.pth"))
    state["candidate_ids"] = list(data["remaining"])
    state["labelled_ids"] = list(data["labelled"])
    execution_status = {
        "experiment": EXPERIMENT_NAME,
        "status": STATUS,
        "completed_rounds": sorted(f"{name} round {number}"
                                   for name, number in completed_rounds),
        "elapsed_since_start_seconds": round(time.time() - NOTEBOOK_STARTED, 1),
    }
    write_json(Path(LOCAL_RESULT_ROOT, "execution_status.json"), execution_status)
    write_json(Path(DRIVE_RESULT_DIR, "execution_status.json"), execution_status)


variant_states = {
    variant: {
        "checkpoint": str(task1_checkpoint),
        "candidate_ids": list(candidate_ids),
        "labelled_ids": [],
    }
    for variant in VARIANTS
}
round_history = []
round_records = {}
completed_rounds = set()

print(f"round machinery ready for {len(VARIANTS)} variants x {ROUNDS} rounds")
print(f"local rounds : {LOCAL_RESULT_ROOT}/<variant>/round_NN")
print(f"drive rounds : {DRIVE_RESULT_DIR}/<variant>/round_NN")

In [ ]:
# ============================================================================
# P2. GATE: ONE COMPLETE ROUND BEFORE COMMITTING TO THE CAMPAIGN
#
# The random baseline's round 1 runs first, alone. It exercises the full loop —
# proposal export, selection, PROB fine-tuning, official evaluation, artifact
# validation, checkpoint verification, Drive persistence with a byte-for-byte
# digest comparison — so a problem that only appears in a real round surfaces
# after one round rather than after twelve.
#
# The campaign cell below restores this round from Drive instead of recomputing it.
# ============================================================================
require_stages("preflight")

gate_variant = next(iter(VARIANTS))
if VARIANTS[gate_variant]["strategy"] != "random":
    raise RuntimeError(
        f"The gate round must be the random baseline, but {gate_variant!r} uses "
        f"strategy {VARIANTS[gate_variant]['strategy']!r}."
    )
if (gate_variant, 1) in completed_rounds:
    print(f"{gate_variant} round 1 already completed in this session; skipping the gate.")
else:
    gate_input_checkpoint = variant_states[gate_variant]["checkpoint"]
    gate_started = time.time()
    (gate_data, gate_record, gate_resumed,
     gate_checkpoint_summary, gate_round_record) = execute_round(
        gate_variant, 1, variant_states[gate_variant])
    gate_checkpoint = Path(gate_data["directory"], "checkpoint.pth")
    gate_drive_round = round_directory(DRIVE_RESULT_DIR, gate_variant, 1)
    if not gate_drive_round.is_dir():
        raise RuntimeError(f"Round 1 was not persisted to Drive at {gate_drive_round}.")
    gate_drive_checkpoint = gate_drive_round / "checkpoint.pth"
    local_gate_digest = sha256_of_file(gate_checkpoint)
    if local_gate_digest != sha256_of_file(gate_drive_checkpoint):
        raise RuntimeError(
            f"{gate_drive_checkpoint} does not match the local checkpoint byte for byte."
        )
    for _name in NOTEBOOK_ROUND_FILES:
        if not (gate_drive_round / _name).is_file():
            raise RuntimeError(f"{gate_drive_round / _name} was not persisted.")
    commit_round(gate_variant, 1, variant_states[gate_variant],
                 gate_data, gate_record, gate_resumed, gate_round_record)
    table(f"GATE ROUND ({gate_variant} round 1)", [
        ("strategy", VARIANTS[gate_variant]["strategy"]),
        ("resumed", gate_resumed),
        ("wall clock", elapsed_text(time.time() - gate_started)),
        ("input checkpoint", gate_input_checkpoint),
        ("selected images", gate_data["selected"]),
        ("remaining candidates", len(gate_data["remaining"])),
        ("labelled after", len(gate_data["labelled"])),
        *[(metric, gate_data["metrics"][metric]) for metric in REQUIRED_METRICS],
        ("checkpoint", gate_checkpoint),
        ("checkpoint bytes", f"{gate_checkpoint.stat().st_size:,}"),
        ("checkpoint epoch", gate_checkpoint_summary["epoch"]),
        ("tensors changed by training", gate_checkpoint_summary["tensors_changed"]),
        ("checkpoint sha256", local_gate_digest),
        ("drive checkpoint", gate_drive_checkpoint),
        ("drive copy identical", True),
        ("selected task classes",
         gate_round_record["distribution"]["selected_task_class_counts"]),
        ("free /content GB", free_gb(CONTENT_ROOT)),
        ("free Drive GB", free_gb(DRIVE_RESULT_PARENT)),
    ])
cleanup_runtime()

In [ ]:
# ============================================================================
# P3. THE FULL CAMPAIGN: FOUR VARIANTS x THREE ROUNDS
#
# Twelve PROB fine-tuning runs and twelve official evaluations. A round already
# on Drive — including the gate round above — is restored and validated instead
# of recomputed, so an interrupted session resumes exactly where it stopped.
# ============================================================================
require_stages("preflight")

campaign_started = time.time()
for variant in VARIANTS:
    state = variant_states[variant]
    for index in range(1, ROUNDS + 1):
        if (variant, index) in completed_rounds:
            print(f"[skip] {variant} round {index} already completed in this session")
            continue
        banner(f"CAMPAIGN {len(completed_rounds) + 1}/{len(VARIANTS) * ROUNDS}: "
               f"{variant} round {index}")
        data, record, resumed, checkpoint_summary, round_record = execute_round(
            variant, index, state)
        commit_round(variant, index, state, data, record, resumed, round_record)
        print(f"elapsed since start: {elapsed_text(time.time() - NOTEBOOK_STARTED)}")
        cleanup_runtime()

incomplete = sorted(stage for stage in ROUND_STAGES
                    if STATUS[stage] not in {"OK", "RESTORED"})
if incomplete:
    raise RuntimeError(f"Rounds did not complete: {incomplete}")
if len(round_history) != len(VARIANTS) * ROUNDS:
    raise RuntimeError(
        f"Expected {len(VARIANTS) * ROUNDS} rounds, recorded {len(round_history)}."
    )

# Each variant must have consumed exactly its cumulative budget.
for variant in VARIANTS:
    state = variant_states[variant]
    if len(state["labelled_ids"]) != ROUNDS * BUDGET_PER_ROUND:
        raise RuntimeError(
            f"{variant} labelled {len(state['labelled_ids'])} images, expected "
            f"{ROUNDS * BUDGET_PER_ROUND}."
        )
    if len(set(state["labelled_ids"])) != len(state["labelled_ids"]):
        raise RuntimeError(f"{variant} selected the same image twice.")
    if not set(state["labelled_ids"]) <= set(candidate_ids):
        raise RuntimeError(f"{variant} selected images from outside the candidate pool.")

executed = sum(1 for row in round_history if not row["resumed"])
restored = sum(1 for row in round_history if row["resumed"])
executed_seconds = sum(row["elapsed_seconds"] or 0
                       for row in round_history if not row["resumed"])
table("CAMPAIGN", [
    ("rounds executed now", executed),
    ("rounds restored from Drive", restored),
    ("total rounds", len(round_history)),
    ("training runs", len(round_history)),
    ("official evaluations", len(round_history)),
    ("campaign wall clock", elapsed_text(time.time() - campaign_started)),
    ("detector time this session", elapsed_text(executed_seconds)),
    ("mean seconds per executed round",
     round(executed_seconds / executed, 1) if executed else "n/a"),
    ("labelled per variant", ROUNDS * BUDGET_PER_ROUND),
    ("free /content GB", free_gb(CONTENT_ROOT)),
    ("free Drive GB", free_gb(DRIVE_RESULT_PARENT)),
])

In [ ]:
# ============================================================================
# Q. EVALUATION AND METRICS COLLECTION
#
# Everything is read back from the round artifacts on disk rather than from
# in-memory state, so the tables describe exactly what was persisted. Machine
# readable outputs are written as CSV and JSON.
# ============================================================================
require_stages(*ROUND_STAGES)

import pandas as pd

METRIC_DIRECTIONS = {
    "known_mAP": "higher is better",
    "U_Recall": "higher is better",
    "WI": "lower is better",
    "A_OSE": "lower is better",
    "unknown_AP50": "higher is better",
    "previous_known_AP50": "higher is better",
}
ACQUISITION_COMPONENTS = ("uncertainty", "novelty", "rarity", "coherence",
                          "rarity_bonus", "score")


def optional_float(value):
    return float("nan") if value is None else float(value)


metric_rows = []
proposal_component_rows = []
image_component_rows = []
distribution_rows = []
selected_by_variant = {variant: {} for variant in VARIANTS}

for entry in round_history:
    variant, index = entry["variant"], entry["round"]
    directory = round_directory(LOCAL_RESULT_ROOT, variant, index)
    record = read_json(directory / "round_record.json")
    manifest = read_json(directory / "round_manifest.json")
    metrics = read_json(directory / "metrics.json")
    selected = read_ids(directory / "selected_ids.txt")
    selected_by_variant[variant][f"round_{index:02d}"] = selected

    metric_rows.append({
        "variant": variant,
        "strategy": VARIANTS[variant]["strategy"],
        "coherence_power": float(VARIANTS[variant]["coherence_power"]),
        "round": index,
        "cumulative_budget": index * BUDGET_PER_ROUND,
        "candidate_count_before": manifest["candidate_count_before"],
        "candidate_count_after": manifest["candidate_count_after"],
        "reference_count": manifest["reference_count"],
        "labelled_images": manifest["labelled_count_after"],
        "known_mAP": optional_float(metrics["known_mAP"]),
        "U_Recall": optional_float(metrics["U_Recall"]),
        "WI": optional_float(metrics["WI"]),
        "A_OSE": optional_float(metrics["A_OSE"]),
        "unknown_AP50": optional_float(metrics["unknown_AP50"]),
        "previous_known_AP50": optional_float(metrics["previous_known_AP50"]),
        "current_known_AP50": optional_float(metrics.get("current_known_AP50")),
        "checkpoint_epoch": record["checkpoint_summary"]["epoch"],
        "tensors_changed": record["checkpoint_summary"]["tensors_changed"],
        "checkpoint_sha256": record["checkpoint_sha256"],
        "elapsed_seconds": optional_float(record.get("elapsed_seconds")),
        "resumed": entry["resumed"],
    })

    for row in read_component_rows(directory / "proposal_scores.csv"):
        proposal_component_rows.append({
            "variant": variant,
            "strategy": VARIANTS[variant]["strategy"],
            "coherence_power": float(VARIANTS[variant]["coherence_power"]),
            "round": index,
            "image_id": row["image_id"],
            **{field: float(row[field]) for field in ACQUISITION_COMPONENTS},
        })
    for row in read_component_rows(directory / "image_scores.csv"):
        image_component_rows.append({
            "variant": variant,
            "strategy": VARIANTS[variant]["strategy"],
            "round": index,
            "image_id": row["image_id"],
            "image_score": float(row["score"]),
            "selected": row["image_id"] in set(selected),
        })

    distribution = record["distribution"]
    distribution_rows.append({
        "variant": variant,
        "strategy": VARIANTS[variant]["strategy"],
        "round": index,
        "proposals": distribution["proposals"],
        "images_scored": distribution["images_scored"],
        "unknown_index_proposals": distribution["unknown_index_proposals"],
        "unknown_confidence_mean": distribution["unknown_confidence"]["mean"],
        "unknown_confidence_max": distribution["unknown_confidence"]["max"],
        "selected_task_classes_covered": distribution["selected_task_classes_covered"],
        "selected_task_class_counts": json.dumps(
            distribution["selected_task_class_counts"], sort_keys=True),
        "pool_task_class_counts": json.dumps(
            distribution["pool_task_class_counts"], sort_keys=True),
    })

metrics_frame = pd.DataFrame(metric_rows).sort_values(
    ["variant", "round"]).reset_index(drop=True)
if len(metrics_frame) != len(VARIANTS) * ROUNDS:
    raise RuntimeError(f"Expected {len(VARIANTS) * ROUNDS} metric rows, "
                       f"found {len(metrics_frame)}.")
proposal_components_frame = pd.DataFrame(proposal_component_rows)
image_components_frame = pd.DataFrame(image_component_rows)
distribution_frame = pd.DataFrame(distribution_rows)

scored_variants = [name for name, config in VARIANTS.items()
                   if config["strategy"] != "random"]
if not proposal_components_frame.empty:
    covered = set(proposal_components_frame["variant"].unique())
    missing_scored = sorted(set(scored_variants) - covered)
    if missing_scored:
        raise RuntimeError(
            f"No per-proposal acquisition components were recorded for {missing_scored}."
        )
elif scored_variants:
    raise RuntimeError(
        "No per-proposal acquisition components were recorded at all, although "
        f"{scored_variants} are scored variants."
    )

# --- selection overlap between variants -------------------------------------
overlap_rows = []
for index in range(1, ROUNDS + 1):
    sets = {variant: set(selected_by_variant[variant][f"round_{index:02d}"])
            for variant in VARIANTS}
    for left in VARIANTS:
        for right in VARIANTS:
            overlap_rows.append({"round": index, "left": left, "right": right,
                                 "overlap": len(sets[left] & sets[right])})
    banner(f"SELECTION OVERLAP, ROUND {index}")
    print(pd.DataFrame({left: {right: len(sets[left] & sets[right])
                               for right in VARIANTS} for left in VARIANTS}
                       ).loc[list(VARIANTS), list(VARIANTS)].to_string())
overlap_frame = pd.DataFrame(overlap_rows)

cumulative_overlap_rows = []
for left in VARIANTS:
    for right in VARIANTS:
        left_ids = set(variant_states[left]["labelled_ids"])
        right_ids = set(variant_states[right]["labelled_ids"])
        cumulative_overlap_rows.append({
            "left": left, "right": right,
            "overlap": len(left_ids & right_ids),
            "jaccard": round(len(left_ids & right_ids) / len(left_ids | right_ids), 4),
        })
cumulative_overlap_frame = pd.DataFrame(cumulative_overlap_rows)

learning_curve = metrics_frame[
    ["variant", "strategy", "coherence_power", "round", "cumulative_budget",
     *METRIC_DIRECTIONS]
].sort_values(["variant", "cumulative_budget"]).reset_index(drop=True)
final_round = metrics_frame[metrics_frame["round"] == ROUNDS].reset_index(drop=True)

component_summary = (
    proposal_components_frame
    .groupby(["variant", "round"])[list(ACQUISITION_COMPONENTS)]
    .agg(["mean", "std"])
    .round(6)
    if not proposal_components_frame.empty else pd.DataFrame()
)

banner("METRIC DIRECTIONS")
for metric, direction in METRIC_DIRECTIONS.items():
    print(f"  {metric}: {direction}")
banner("ALL ROUNDS")
print(metrics_frame.drop(columns=["checkpoint_sha256"]).to_string(index=False))
banner("FINAL ROUND")
print(final_round.drop(columns=["checkpoint_sha256"]).to_string(index=False))
banner("LEARNING CURVE")
print(learning_curve.to_string(index=False))
banner("ACQUISITION COMPONENTS (scored variants only)")
if component_summary.empty:
    print("no scored variants")
else:
    print(component_summary.to_string())
banner("PER-ROUND DISTRIBUTION SUMMARY")
print(distribution_frame.drop(
    columns=["pool_task_class_counts"]).to_string(index=False))
banner("CUMULATIVE SELECTION JACCARD")
print(cumulative_overlap_frame.pivot(index="left", columns="right", values="jaccard")
      .loc[list(VARIANTS), list(VARIANTS)].to_string())

CSV_ARTIFACTS = {
    "multiround_metrics.csv": metrics_frame,
    "multiround_learning_curve.csv": learning_curve,
    "multiround_final_round.csv": final_round,
    "multiround_overlap.csv": overlap_frame,
    "multiround_cumulative_overlap.csv": cumulative_overlap_frame,
    "multiround_proposal_components.csv": proposal_components_frame,
    "multiround_image_components.csv": image_components_frame,
    "multiround_distribution.csv": distribution_frame,
}
for name, frame in CSV_ARTIFACTS.items():
    frame.to_csv(Path(LOCAL_RESULT_ROOT, name), index=False)

table("COLLECTED OUTPUTS", [
    ("metric rows", len(metrics_frame)),
    ("per-proposal component rows", len(proposal_components_frame)),
    ("per-image score rows", len(image_components_frame)),
    ("distribution rows", len(distribution_frame)),
    ("overlap rows", len(overlap_frame)),
    ("csv artifacts", len(CSV_ARTIFACTS)),
    ("random baseline components",
     "not written by design: the pinned run_active_round selects at random "
     "without scoring"),
])
mark("metrics")

In [ ]:
# ============================================================================
# R. RESULT TABLES AND PLOTS
#
# One summary table comparing every variant across every round, plus three
# figures: the learning curves, the acquisition-component means, and the
# cumulative selection-overlap heatmap.
#
# The figures deliberately commit to a single light surface so a saved PNG reads
# the same wherever it is opened. Series colours are the first three slots of the
# categorical palette plus violet; that four-colour set was validated on the
# all-pairs list against this surface (worst pair CVD dE 9.2, normal-vision 16.3),
# and identity never rests on colour alone: every figure carries a legend, the
# learning-curve panel is directly labelled, and the full numbers are printed as
# text tables below.
# ============================================================================
require_stages("metrics")

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

SURFACE = "#fcfcfb"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#8a8880"
GRID = "#e6e5e1"
# Categorical slots 1, 2, 3 and 7 of the reference palette, in fixed order.
VARIANT_COLOURS = dict(zip(VARIANTS, ["#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"]))
VARIANT_MARKERS = dict(zip(VARIANTS, ["o", "s", "^", "D"]))
# Sequential blue ramp, steps 100 -> 700, for continuous magnitude.
SEQUENTIAL_BLUE = LinearSegmentedColormap.from_list(
    "daowod_blue",
    ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"],
)

matplotlib.rcParams.update({
    "figure.facecolor": SURFACE,
    "figure.dpi": 130,
    "savefig.facecolor": SURFACE,
    "savefig.bbox": "tight",
    "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID,
    "axes.labelcolor": INK_SECONDARY,
    "axes.titlecolor": INK_PRIMARY,
    "axes.titlesize": 11,
    "axes.titleweight": "semibold",
    "axes.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "text.color": INK_PRIMARY,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "lines.linewidth": 2.0,
    "lines.markersize": 7,
})

figures_dir = Path(LOCAL_RESULT_ROOT, "figures")
figures_dir.mkdir(parents=True, exist_ok=True)
saved_figures = []


def save_figure(figure, name):
    path = figures_dir / name
    figure.savefig(path)
    saved_figures.append(path)
    print(f"saved {path}")


# --- the summary table ------------------------------------------------------
summary_table = (
    metrics_frame
    .pivot_table(index=["variant", "strategy", "coherence_power"],
                 columns="round", values=list(METRIC_DIRECTIONS))
    .reindex(index=[(name, VARIANTS[name]["strategy"],
                     float(VARIANTS[name]["coherence_power"])) for name in VARIANTS])
    .round(4)
)
summary_table.columns = [f"{metric}_r{index}" for metric, index in summary_table.columns]
banner("SUMMARY: EVERY VARIANT ACROSS EVERY ROUND")
print(summary_table.to_string())
summary_table.to_csv(Path(LOCAL_RESULT_ROOT, "multiround_summary_table.csv"))

delta_rows = []
for variant in VARIANTS:
    rows = metrics_frame[metrics_frame["variant"] == variant].sort_values("round")
    first, last = rows.iloc[0], rows.iloc[-1]
    delta_rows.append({
        "variant": variant,
        "strategy": VARIANTS[variant]["strategy"],
        "coherence_power": float(VARIANTS[variant]["coherence_power"]),
        **{f"delta_{metric}": round(float(last[metric]) - float(first[metric]), 4)
           for metric in METRIC_DIRECTIONS},
    })
delta_frame = pd.DataFrame(delta_rows)
banner(f"CHANGE FROM ROUND 1 TO ROUND {ROUNDS} (direction matters: "
       "WI and A_OSE are better when lower)")
print(delta_frame.to_string(index=False))
delta_frame.to_csv(Path(LOCAL_RESULT_ROOT, "multiround_deltas.csv"), index=False)

# --- figure 1: learning curves ---------------------------------------------
metric_names = list(METRIC_DIRECTIONS)
figure, axes = plt.subplots(2, 3, figsize=(13, 7.2))
for position, metric in enumerate(metric_names):
    axis = axes.flat[position]
    for variant in VARIANTS:
        rows = metrics_frame[metrics_frame["variant"] == variant].sort_values("round")
        axis.plot(rows["cumulative_budget"], rows[metric],
                  color=VARIANT_COLOURS[variant], marker=VARIANT_MARKERS[variant],
                  markeredgecolor=SURFACE, markeredgewidth=2.0, label=variant)
    axis.set_title(f"{metric}  ({METRIC_DIRECTIONS[metric].replace(' is better', '')})")
    axis.set_xlabel("labelled images")
    axis.set_xticks(sorted(metrics_frame["cumulative_budget"].unique()))
    if position == 0:
        # Direct labels on the first panel, ordered by final value so they cannot
        # collide; the figure legend covers every panel.
        finals = sorted(
            ((variant,
              float(metrics_frame[(metrics_frame["variant"] == variant)
                                  & (metrics_frame["round"] == ROUNDS)][metric].iloc[0]))
             for variant in VARIANTS),
            key=lambda item: item[1],
        )
        span = (max(value for _, value in finals) - min(value for _, value in finals)) or 1.0
        for order, (variant, value) in enumerate(finals):
            axis.annotate(
                variant,
                xy=(ROUNDS * BUDGET_PER_ROUND, value),
                xytext=(6, (order - (len(finals) - 1) / 2) * 0.14 * span),
                textcoords="offset points", fontsize=8, color=INK_SECONDARY,
                va="center", annotation_clip=False,
            )
figure.suptitle(
    f"Contribution A pilot: PROB metrics per acquisition variant  "
    f"(seed {SEED}, {ROUNDS} rounds x {BUDGET_PER_ROUND} images)",
    color=INK_PRIMARY, fontsize=12, fontweight="semibold",
)
handles, labels = axes.flat[0].get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", ncol=len(VARIANTS),
              bbox_to_anchor=(0.5, -0.02))
figure.tight_layout(rect=(0, 0.03, 1, 0.96))
save_figure(figure, "learning_curves.png")
plt.show()

# --- figure 2: acquisition components ---------------------------------------
if proposal_components_frame.empty:
    print("no scored variants, so there are no acquisition components to plot")
else:
    components = ["uncertainty", "novelty", "rarity", "coherence", "score"]
    final_components = proposal_components_frame[
        proposal_components_frame["round"] == ROUNDS]
    figure, axes = plt.subplots(1, len(components), figsize=(14, 3.6))
    for axis, component in zip(axes, components):
        names, means, deviations, colours = [], [], [], []
        for variant in VARIANTS:
            rows = final_components[final_components["variant"] == variant]
            if rows.empty:
                continue
            mean = float(rows[component].mean())
            deviation = float(rows[component].std())
            names.append(variant)
            means.append(mean if math.isfinite(mean) else 0.0)
            deviations.append(deviation if math.isfinite(deviation) else 0.0)
            colours.append(VARIANT_COLOURS[variant])
        positions = range(len(names))
        axis.bar(positions, means, width=0.62, color=colours,
                 edgecolor=SURFACE, linewidth=2.0)
        axis.errorbar(positions, means, yerr=deviations, fmt="none",
                      ecolor=INK_MUTED, elinewidth=1.4, capsize=4)
        # Labels sit above the whisker, never across it.
        for position, value, deviation in zip(positions, means, deviations):
            axis.annotate(f"{value:.3f}", xy=(position, value + deviation),
                          xytext=(0, 6), textcoords="offset points",
                          ha="center", fontsize=8, color=INK_SECONDARY)
        axis.set_title(component)
        axis.set_xticks(list(positions))
        axis.set_xticklabels(names, rotation=20, ha="right", fontsize=8)
        axis.set_ylim(0, max([*(m + d for m, d in zip(means, deviations)), 1e-6]) * 1.28)
    axes[0].set_ylabel(f"mean over round-{ROUNDS} proposals")
    figure.suptitle(
        f"Acquisition components in round {ROUNDS} (mean +/- 1 sd over all "
        "candidate proposals; the random baseline is unscored by design)",
        color=INK_PRIMARY, fontsize=11, fontweight="semibold",
    )
    figure.tight_layout(rect=(0, 0, 1, 0.9))
    save_figure(figure, "acquisition_components.png")
    plt.show()

# --- figure 3: cumulative selection overlap ---------------------------------
overlap_matrix = (
    cumulative_overlap_frame
    .pivot(index="left", columns="right", values="jaccard")
    .loc[list(VARIANTS), list(VARIANTS)]
)
figure, axis = plt.subplots(figsize=(5.6, 5.0))
image = axis.imshow(overlap_matrix.to_numpy(), cmap=SEQUENTIAL_BLUE, vmin=0.0, vmax=1.0)
axis.set_xticks(range(len(VARIANTS)))
axis.set_yticks(range(len(VARIANTS)))
axis.set_xticklabels(list(VARIANTS), rotation=25, ha="right", fontsize=8)
axis.set_yticklabels(list(VARIANTS), fontsize=8)
axis.grid(False)
for row, left in enumerate(VARIANTS):
    for column, right in enumerate(VARIANTS):
        value = float(overlap_matrix.iloc[row, column])
        axis.annotate(f"{value:.2f}", xy=(column, row), ha="center", va="center",
                      fontsize=9, color=SURFACE if value > 0.55 else INK_PRIMARY)
axis.set_title(f"Cumulative selection overlap after {ROUNDS} rounds\n"
               f"(Jaccard over {ROUNDS * BUDGET_PER_ROUND} labelled images)")
colourbar = figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
colourbar.set_label("Jaccard index", color=INK_SECONDARY, fontsize=9)
colourbar.outline.set_edgecolor(GRID)
figure.tight_layout()
save_figure(figure, "selection_overlap.png")
plt.show()

table("FIGURES", [
    ("figures directory", figures_dir),
    ("figures saved", [path.name for path in saved_figures]),
    ("summary table", Path(LOCAL_RESULT_ROOT, "multiround_summary_table.csv")),
    ("delta table", Path(LOCAL_RESULT_ROOT, "multiround_deltas.csv")),
    ("palette", "categorical slots 1/2/3/7 validated all-pairs on a light surface"),
])
mark("report")

In [ ]:
# ============================================================================
# S. ARTIFACTS AND MANIFEST TO DRIVE
#
# The twelve round directories were already persisted and digest-verified as they
# completed. This cell writes the campaign-level summary, the experiment manifest
# (commits, runtime, GPU, build, pre-flight, PROB source validation) and the figures,
# then verifies that every file actually landed on the FUSE mount.
# ============================================================================
require_stages("report")

# The pinned PROB checkout must still be exactly what was cloned, now that every
# round has run. Untracked build products (models/ops/build, the compiled .so)
# are covered by PROB's own .gitignore; a tracked difference means something
# wrote into the checkout and invalidates the source validation recorded below.
final_prob_worktree = run(["git", "-C", PROB_PATH, "status", "--porcelain"],
                          timeout=300).stdout
final_prob_tracked = sorted(
    line[3:] for line in final_prob_worktree.splitlines()
    if line.strip() and not line.startswith("??")
)
final_prob_head = run(["git", "-C", PROB_PATH, "rev-parse", "HEAD"],
                      timeout=120).stdout.strip()
if final_prob_tracked or final_prob_head != PROB_COMMIT:
    raise RuntimeError(
        f"The PROB checkout changed during the campaign: HEAD={final_prob_head}, "
        f"modified tracked files={final_prob_tracked}. It is required to stay at "
        f"{PROB_COMMIT} with a clean working tree."
    )
PROB_SOURCE_VALIDATION["clean_checkout_at_end_of_run"] = True
PROB_SOURCE_VALIDATION["commit_at_end_of_run"] = final_prob_head
repo_details["PROB"]["clean_at_end_of_run"] = True

EXPERIMENT_MANIFEST = {
    "experiment": EXPERIMENT_NAME,
    "protocol": "contribution_a_multiround_v1",
    "benchmark_result": False,
    "note": ("Pilot protocol under a controlled long-tail candidate pool. Not a "
             "published benchmark number."),
    "seed": SEED,
    "generated_by": "notebooks/contribution_a_multiround_prob.ipynb",
    "repositories": repo_details,
    "prob_source_validation": PROB_SOURCE_VALIDATION,
    "runtime": RUNTIME_INFO,
    "baseline_versions": BASELINE_VERSIONS,
    "extension_build": EXTENSION_BUILD,
    "extension_validation": EXTENSION_VALIDATION,
    "attention_backend": attention_backend,
    "dino_backbone": {
        "url": DINO_BACKBONE_URL,
        "path": str(backbone_path),
        "bytes": backbone_path.stat().st_size,
        "sha256": DINO_BACKBONE_SHA256,
        "tensors": backbone_tensor_count,
    },
    "inputs": {
        "dataset_source": str(dataset_source),
        "dataset_mode": dataset_mode,
        "dataset_selected_by": dataset_selection,
        "dataset_discovery": DATASET_DISCOVERY,
        "data_root": DATA_ROOT,
        "annotations": annotation_count,
        "task1_checkpoint_drive": str(task1_source),
        "task1_checkpoint_local": str(task1_checkpoint),
        "task1_checkpoint_sha256": task1_checkpoint_sha256,
        "task1_epoch": task1_epoch,
        "task1_tensors": task1_tensor_count,
    },
    "protocol_record": PROTOCOL_RECORD,
    "experiment_config": experiment_config,
    "experiment_config_sha256": experiment_config_sha256,
    "detector_commands": DETECTOR_COMMANDS,
    "preflight": PREFLIGHT_RECORD,
    "metric_directions": METRIC_DIRECTIONS,
    "rounds": [
        {
            key: value for key, value in round_records[(variant, index)].items()
            if key not in ("candidate_ids_before", "reference_ids",
                           "remaining_pool_ids", "labelled_image_ids_after",
                           "round_manifest", "runtime")
        }
        for variant in VARIANTS for index in range(1, ROUNDS + 1)
    ],
    "selected_ids": selected_by_variant,
    "cumulative_labelled_ids": {
        variant: list(variant_states[variant]["labelled_ids"]) for variant in VARIANTS
    },
    "round_history": round_history,
    "status": STATUS,
    "elapsed_seconds_total": round(time.time() - NOTEBOOK_STARTED, 1),
}

EXPERIMENT_SUMMARY = {
    "experiment": EXPERIMENT_NAME,
    "experiment_config_sha256": experiment_config_sha256,
    "metric_directions": METRIC_DIRECTIONS,
    "metrics": metrics_frame.to_dict(orient="records"),
    "learning_curve": learning_curve.to_dict(orient="records"),
    "final_round": final_round.to_dict(orient="records"),
    "summary_table": summary_table.reset_index().to_dict(orient="records"),
    "deltas": delta_frame.to_dict(orient="records"),
    "selection_overlap": overlap_rows,
    "cumulative_overlap": cumulative_overlap_rows,
    "distribution": distribution_rows,
    "selected_ids": selected_by_variant,
    "round_history": round_history,
    "status": STATUS,
    "benchmark_result": False,
}

JSON_ARTIFACTS = {
    "experiment_manifest.json": EXPERIMENT_MANIFEST,
    "experiment_summary.json": EXPERIMENT_SUMMARY,
    "multiround_selected_ids.json": selected_by_variant,
    "execution_status.json": {
        "experiment": EXPERIMENT_NAME,
        "status": STATUS,
        "elapsed_seconds_total": round(time.time() - NOTEBOOK_STARTED, 1),
    },
}
for name, payload in JSON_ARTIFACTS.items():
    write_json(Path(LOCAL_RESULT_ROOT, name), payload)

FLAT_ARTIFACTS = [
    *JSON_ARTIFACTS,
    *CSV_ARTIFACTS,
    "multiround_summary_table.csv",
    "multiround_deltas.csv",
    "experiment_config.json",
]
missing_locally = [name for name in FLAT_ARTIFACTS
                   if not Path(LOCAL_RESULT_ROOT, name).is_file()]
if missing_locally:
    raise RuntimeError(f"Local artifacts were not written: {missing_locally}")

# --- copy to Drive ----------------------------------------------------------
require_free_space(DRIVE_RESULT_PARENT, MINIMUM_DRIVE_FREE_GB, "Persisting the summary")
for name in FLAT_ARTIFACTS:
    source = Path(LOCAL_RESULT_ROOT, name)
    destination = Path(DRIVE_RESULT_DIR, name)
    staging = destination.with_name(f".{destination.name}.partial")
    shutil.copy2(source, staging)
    staging.replace(destination)

for directory_name, local_directory in (("protocol", protocol_dir),
                                        ("figures", figures_dir),
                                        ("preflight", preflight_dir)):
    target = Path(DRIVE_RESULT_DIR, directory_name)
    staging = target.with_name(f"{directory_name}.__copying__")
    if staging.exists():
        shutil.rmtree(staging)
    shutil.copytree(local_directory, staging)
    if target.exists():
        shutil.rmtree(target)
    staging.replace(target)

# --- verify what landed -----------------------------------------------------
missing_on_drive = [name for name in FLAT_ARTIFACTS
                    if not Path(DRIVE_RESULT_DIR, name).is_file()]
if missing_on_drive:
    raise RuntimeError(f"Drive persistence incomplete: {missing_on_drive}")
for relative in ("protocol/protocol.json", "preflight/preflight_record.json"):
    if not Path(DRIVE_RESULT_DIR, relative).is_file():
        raise RuntimeError(f"Missing on Drive: {DRIVE_RESULT_DIR}/{relative}")
for figure_path in saved_figures:
    drive_figure = Path(DRIVE_RESULT_DIR, "figures", figure_path.name)
    if not drive_figure.is_file():
        raise RuntimeError(f"Missing figure on Drive: {drive_figure}")
    if drive_figure.stat().st_size != figure_path.stat().st_size:
        raise RuntimeError(f"{drive_figure} is a different size from {figure_path}.")

for variant in VARIANTS:
    for index in range(1, ROUNDS + 1):
        directory = round_directory(DRIVE_RESULT_DIR, variant, index)
        for name in (*COMMON_ROUND_FILES, *NOTEBOOK_ROUND_FILES):
            if not (directory / name).is_file():
                raise RuntimeError(f"Round artifact missing from Drive: {directory / name}")

# Two representative digests re-checked across the FUSE boundary.
for name in ("experiment_manifest.json", "multiround_metrics.csv"):
    if sha256_of_file(Path(LOCAL_RESULT_ROOT, name)) != \
            sha256_of_file(Path(DRIVE_RESULT_DIR, name)):
        raise RuntimeError(f"{name} differs between /content and Drive.")

table("DRIVE PERSISTENCE", [
    ("drive result dir", DRIVE_RESULT_DIR),
    ("flat artifacts", len(FLAT_ARTIFACTS)),
    ("directories", ["protocol", "figures", "preflight",
                     *[f"{variant}/round_NN" for variant in VARIANTS]]),
    ("rounds on Drive", len(VARIANTS) * ROUNDS),
    ("figures on Drive", len(saved_figures)),
    ("PROB checkout clean", PROB_SOURCE_VALIDATION["clean_checkout_at_end_of_run"]),
    ("PROB commit unchanged", final_prob_head == PROB_COMMIT),
    ("digests re-verified", 2),
    ("drive size GB", directory_size_gb(DRIVE_RESULT_DIR)),
    ("free Drive GB", free_gb(DRIVE_RESULT_PARENT)),
])
mark("drive persistence")

In [ ]:
# ============================================================================
# T. FINAL SUCCESS REPORT
#
# Fails loudly if any stage did not complete. Nothing here is cosmetic: the same
# STATUS dictionary was written to Drive after every round, so this report and the
# persisted state cannot disagree.
# ============================================================================
banner("STAGE STATUS")
width = max(len(stage) for stage in STAGES)
for stage in STAGES:
    print(f"{stage:<{width}} : {STATUS[stage]}")

pending = [stage for stage in STAGES if STATUS[stage] not in {"OK", "RESTORED"}]
if pending:
    raise RuntimeError(f"Stages did not complete: {pending}")

banner("FINAL ROUND, ALL VARIANTS")
print(final_round[["variant", "strategy", "coherence_power", "labelled_images",
                   *METRIC_DIRECTIONS]].to_string(index=False))

table("RUN SUMMARY", [
    ("experiment", EXPERIMENT_NAME),
    ("seed", SEED),
    ("variants", list(VARIANTS)),
    ("rounds per variant", ROUNDS),
    ("budget per round", BUDGET_PER_ROUND),
    ("PROB training runs", len(round_history)),
    ("official evaluations", len(round_history)),
    ("rounds executed this session", sum(1 for r in round_history if not r["resumed"])),
    ("rounds restored from Drive", sum(1 for r in round_history if r["resumed"])),
    ("DAOWOD commit", repo_commits["DAOWOD"]),
    ("PROB branch / commit", f"{PROB_BRANCH} / {repo_commits['PROB']}"),
    ("PROB checkout", "clean and unmodified; decoder features exported"),
    ("attention backend", attention_backend),
    ("GPU", RUNTIME_INFO["gpu_name"]),
    ("torch / CUDA", f"{RUNTIME_INFO['torch']} / {RUNTIME_INFO['torch_cuda']}"),
    ("interpreter", sys.executable),
    ("total wall clock", elapsed_text(time.time() - NOTEBOOK_STARTED)),
    ("local results", LOCAL_RESULT_ROOT),
    ("drive results", DRIVE_RESULT_DIR),
])

banner("PINNED PROB CHECKOUT")
print(f"  {PROB_SOURCE_VALIDATION['repository']} @ {PROB_SOURCE_VALIDATION['commit']}")
print(f"    branch                 : {PROB_SOURCE_VALIDATION['branch']}")
print(f"    clean working tree     : "
      f"{PROB_SOURCE_VALIDATION['clean_checkout_after_validation']}")
print(f"    modified by notebook   : "
      f"{PROB_SOURCE_VALIDATION['modified_by_notebook']}")
print(f"    decoder-feature export : "
      f"{PROB_SOURCE_VALIDATION['decoder_feature_marker']} in "
      f"{PROB_SOURCE_VALIDATION['model_file']}")
print(f"    bridge check exit code : "
      f"{PROB_SOURCE_VALIDATION['bridge_check_returncode']}")

banner("DONE")
print(f"Contribution A multi-round pilot complete: {EXPERIMENT_NAME}")
print(f"local results : {LOCAL_RESULT_ROOT}")
print(f"drive results : {DRIVE_RESULT_DIR}")
print("  experiment_manifest.json      commits, runtime, GPU, build, pre-flight,")
print("                                PROB source validation")
print("  experiment_summary.json       metrics, learning curve, overlaps, selections")
print("  multiround_metrics.csv        one row per variant and round")
print("  multiround_summary_table.csv  every variant across every round")
print("  multiround_proposal_components.csv   per-proposal acquisition components")
print("  multiround_image_components.csv      per-image acquisition scores")
print("  multiround_distribution.csv         class and distribution summaries")
print("  figures/                      learning curves, components, overlap")
print("  <variant>/round_NN/           proposals, selections, checkpoint, metrics,")
print("                               round_manifest.json, round_record.json")
print("\nThis is a pilot protocol under a controlled long-tail pool, "
      "not a benchmark result.")